In [ ]:
# ============================================================
# STEP 1 — PROJECT CONFIGURATION
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Hugging Face
# ------------------------------------------------------------

HF_USERNAME = "ndeda"

# OLD repository — this is the repository from the old
# Objaverse pipeline.
OLD_HF_DATASET = (
    f"{HF_USERNAME}/"
    "neural-object-reconstruction-raw"
)

# NEW repository — fresh repository for the ABO project.
HF_DATASET = (
    f"{HF_USERNAME}/"
    "neural-object-reconstruction-abo"
)

# ------------------------------------------------------------
# Dataset targets
# ------------------------------------------------------------

TARGET_OBJECTS = 250

# We will initially identify more candidates than we need.
CANDIDATE_TARGET = 1000

# After metadata filtering, we aim for approximately 400
# serious candidates before Blender validation.
FINAL_CANDIDATE_TARGET = 400

# Download only small batches so Colab RAM/storage is not
# unnecessarily consumed.
DOWNLOAD_BATCH_SIZE = 10

# Upload folders in batches instead of committing one object
# at a time to Hugging Face.
UPLOAD_BATCH_SIZE = 25

# ------------------------------------------------------------
# Local project
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/neural_object_reconstruction"
)

STATE_DIR = PROJECT_DIR / "state"
RAW_DIR = PROJECT_DIR / "raw"
CANDIDATES_DIR = PROJECT_DIR / "candidates"
APPROVED_DIR = PROJECT_DIR / "approved"
REJECTED_DIR = PROJECT_DIR / "rejected"
THUMBNAILS_DIR = PROJECT_DIR / "thumbnails"
MANIFEST_DIR = PROJECT_DIR / "manifests"

print("=" * 70)
print("PROJECT CONFIGURATION")
print("=" * 70)

print("Old HF repository:")
print(OLD_HF_DATASET)

print("\nNew HF repository:")
print(HF_DATASET)

print("\nTarget objects:")
print(TARGET_OBJECTS)

print("\nCandidate target:")
print(CANDIDATE_TARGET)

print("\nFinal candidate target:")
print(FINAL_CANDIDATE_TARGET)

print("\nProject directory:")
print(PROJECT_DIR)

PROJECT CONFIGURATION
Old HF repository:
ndeda/neural-object-reconstruction-raw

New HF repository:
ndeda/neural-object-reconstruction-abo

Target objects:
250

Candidate target:
1000

Final candidate target:
400

Project directory:
/content/neural_object_reconstruction


In [ ]:
# ============================================================
# STEP 2 — DELETE OLD LOCAL PROJECT DATA
# ============================================================

import shutil

print("=" * 70)
print("DELETING OLD LOCAL PROJECT DATA")
print("=" * 70)

paths_to_delete = [
    PROJECT_DIR,
    Path("/root/.objaverse"),
    Path("/content/objaverse"),
]

for path in paths_to_delete:

    if path.exists():

        print(f"\nDeleting:")
        print(path)

        shutil.rmtree(
            path,
            ignore_errors=True
        )

        print("✓ Deleted")

    else:

        print(f"\nNot found:")
        print(path)

# ------------------------------------------------------------
# Recreate clean project structure
# ------------------------------------------------------------

directories = [
    PROJECT_DIR,
    STATE_DIR,
    RAW_DIR,
    CANDIDATES_DIR,
    APPROVED_DIR,
    REJECTED_DIR,
    THUMBNAILS_DIR,
    MANIFEST_DIR,
]

for directory in directories:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("\n" + "=" * 70)
print("✓ LOCAL RESET COMPLETE")
print("=" * 70)

print("\nFresh project:")
print(PROJECT_DIR)

DELETING OLD LOCAL PROJECT DATA

Not found:
/content/neural_object_reconstruction

Not found:
/root/.objaverse

Not found:
/content/objaverse

✓ LOCAL RESET COMPLETE

Fresh project:
/content/neural_object_reconstruction


In [ ]:
# ============================================================
# STEP 3 — INSTALL DEPENDENCIES
# ============================================================

!pip -q install -U \
    huggingface_hub \
    pandas \
    numpy \
    requests \
    tqdm \
    pillow

print("✓ Dependencies installed.")

✓ Dependencies installed.


In [ ]:
# ============================================================
# STEP 4 — HUGGING FACE LOGIN
# ============================================================

from huggingface_hub import login, HfApi

print("=" * 70)
print("HUGGING FACE LOGIN")
print("=" * 70)

print("""
A Hugging Face token is required.

Use a token with permission to:
- create repositories
- delete the old dataset repository
- upload to the new dataset repository
""")

login()

api = HfApi()

account = api.whoami()

print("\n✓ Hugging Face authentication successful.")

print(
    "Account:",
    account.get(
        "name",
        account.get("fullname", "Unknown")
    )
)

HUGGING FACE LOGIN

A Hugging Face token is required.

Use a token with permission to:
- create repositories
- delete the old dataset repository
- upload to the new dataset repository




✓ Hugging Face authentication successful.
Account: ndeda


In [ ]:
# ============================================================
# STEP 5 — DELETE OLD HUGGING FACE DATASET
# ============================================================

print("=" * 70)
print("WARNING — PERMANENT HUGGING FACE DELETION")
print("=" * 70)

print()
print("Repository to DELETE:")
print(OLD_HF_DATASET)

print()
print("This will permanently remove the old dataset repository")
print("and its files from Hugging Face.")
print()
print("It will NOT delete your Hugging Face account.")
print()

confirmation = input(
    "Type exactly: DELETE OLD REPOSITORY\n\n"
    "> "
)

if confirmation != "DELETE OLD REPOSITORY":

    raise RuntimeError(
        "Deletion cancelled. "
        "Nothing was deleted from Hugging Face."
    )

print("\nDeleting old repository...")

try:

    api.delete_repo(
        repo_id=OLD_HF_DATASET,
        repo_type="dataset",
    )

    print(
        "\n✓ OLD HUGGING FACE REPOSITORY DELETED"
    )

except Exception as e:

    error = str(e)

    if (
        "404" in error
        or "not found" in error.lower()
    ):

        print(
            "\n✓ Repository does not exist."
        )

    else:

        print("\nDeletion failed:")
        print(error)

        raise

WARNING — PERMANENT HUGGING FACE DELETION

Repository to DELETE:
ndeda/neural-object-reconstruction-raw

This will permanently remove the old dataset repository
and its files from Hugging Face.

It will NOT delete your Hugging Face account.

Type exactly: DELETE OLD REPOSITORY

> DELETE OLD REPOSITORY

Deleting old repository...

✓ OLD HUGGING FACE REPOSITORY DELETED


In [ ]:
# ============================================================
# STEP 6 — CREATE NEW PUBLIC DATASET REPOSITORY
# ============================================================

print("=" * 70)
print("CREATING NEW PUBLIC DATASET REPOSITORY")
print("=" * 70)

print()
print("Repository:")
print(HF_DATASET)

api.create_repo(
    repo_id=HF_DATASET,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)

print()
print("✓ PUBLIC DATASET REPOSITORY READY")
print()
print(
    f"https://huggingface.co/datasets/{HF_DATASET}"
)

CREATING NEW PUBLIC DATASET REPOSITORY

Repository:
ndeda/neural-object-reconstruction-abo

✓ PUBLIC DATASET REPOSITORY READY

https://huggingface.co/datasets/ndeda/neural-object-reconstruction-abo


In [ ]:
# ============================================================
# STEP 7 — PERSISTENT PIPELINE STATE
# ============================================================

import json
from datetime import datetime, timezone

STATE_FILE = STATE_DIR / "pipeline_state.json"


def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def save_json(path, data):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    temporary.replace(path)


def load_json(
    path,
    default=None
):

    path = Path(path)

    if not path.exists():

        return default

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


state = {

    "project": (
        "Neural Object Reconstruction "
        "& Relighting Lab"
    ),

    "dataset_source": "Amazon Berkeley Objects",

    "target_objects": TARGET_OBJECTS,

    "stage": "initialized",

    "downloaded": [],

    "validated": [],

    "approved": [],

    "rejected": [],

    "failed": [],

    "updated_at": utc_now(),
}


save_json(
    STATE_FILE,
    state
)

print("=" * 70)
print("✓ CHECKPOINT SYSTEM INITIALIZED")
print("=" * 70)

print()
print("State file:")
print(STATE_FILE)

✓ CHECKPOINT SYSTEM INITIALIZED

State file:
/content/neural_object_reconstruction/state/pipeline_state.json


In [ ]:
# ============================================================
# STEP 8 — DOWNLOAD ABO METADATA ONLY
# ============================================================

import requests
from pathlib import Path
from tqdm.auto import tqdm

# Correct official ABO archive location
METADATA_URL = (
    "https://amazon-berkeley-objects.s3.amazonaws.com/"
    "archives/abo-listings.tar"
)

METADATA_ARCHIVE = (
    PROJECT_DIR / "abo-listings.tar"
)

print("=" * 70)
print("ABO METADATA DOWNLOAD")
print("=" * 70)

print("\nSource:")
print(METADATA_URL)

print("\nThis downloads ONLY the product metadata.")
print("We are NOT downloading the 154 GB 3D-model archive.")

# ------------------------------------------------------------
# Download only if we don't already have it
# ------------------------------------------------------------

if METADATA_ARCHIVE.exists():

    print("\n✓ Metadata archive already exists.")

else:

    print("\nConnecting to Amazon S3...")

    response = requests.get(
        METADATA_URL,
        stream=True,
        timeout=120
    )

    response.raise_for_status()

    total = int(
        response.headers.get(
            "content-length",
            0
        )
    )

    print(
        f"Expected size: "
        f"{total / (1024**2):.2f} MB"
    )

    with open(
        METADATA_ARCHIVE,
        "wb"
    ) as f:

        with tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Downloading ABO metadata"
        ) as progress:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if chunk:

                    f.write(chunk)

                    progress.update(
                        len(chunk)
                    )

    print("\n✓ Download complete.")

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

file_size = (
    METADATA_ARCHIVE.stat().st_size
    / (1024 ** 2)
)

print("\n" + "=" * 70)
print("DOWNLOAD COMPLETE")
print("=" * 70)

print("File:")
print(METADATA_ARCHIVE)

print(
    f"\nSize: {file_size:.2f} MB"
)

ABO METADATA DOWNLOAD

Source:
https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar

This downloads ONLY the product metadata.
We are NOT downloading the 154 GB 3D-model archive.

Connecting to Amazon S3...
Expected size: 83.43 MB



✓ Download complete.

DOWNLOAD COMPLETE
File:
/content/neural_object_reconstruction/abo-listings.tar

Size: 83.43 MB


In [ ]:
# ============================================================
# STEP 9 — EXTRACT ABO METADATA
# ============================================================

import tarfile

ABO_METADATA_DIR = (
    PROJECT_DIR / "abo_metadata"
)

ABO_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("EXTRACTING ABO METADATA")
print("=" * 70)

with tarfile.open(
    METADATA_ARCHIVE,
    "r"
) as tar:

    members = tar.getmembers()

    print(
        f"\nArchive contains {len(members)} files."
    )

    for member in members:
        print(" -", member.name)

    tar.extractall(
        ABO_METADATA_DIR
    )

print("\n✓ ABO metadata extracted.")
print(
    "Location:",
    ABO_METADATA_DIR
)

EXTRACTING ABO METADATA

Archive contains 20 files.
 - LICENSE-CC-BY-4.0.txt
 - listings
 - listings/README.md
 - listings/metadata
 - listings/metadata/listings_7.json.gz
 - listings/metadata/listings_4.json.gz
 - listings/metadata/listings_2.json.gz
 - listings/metadata/listings_c.json.gz
 - listings/metadata/listings_6.json.gz
 - listings/metadata/listings_0.json.gz
 - listings/metadata/listings_9.json.gz
 - listings/metadata/listings_e.json.gz
 - listings/metadata/listings_1.json.gz
 - listings/metadata/listings_5.json.gz
 - listings/metadata/listings_3.json.gz
 - listings/metadata/listings_d.json.gz
 - listings/metadata/listings_f.json.gz
 - listings/metadata/listings_8.json.gz
 - listings/metadata/listings_a.json.gz
 - listings/metadata/listings_b.json.gz

✓ ABO metadata extracted.
Location: /content/neural_object_reconstruction/abo_metadata


/tmp/ipykernel_1129/2011275800.py:34: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(


In [ ]:
# ============================================================
# STEP 10 — FIND EXTRACTED ABO FILES
# ============================================================

from pathlib import Path

print("=" * 70)
print("EXTRACTED ABO FILES")
print("=" * 70)

metadata_files = []

for path in ABO_METADATA_DIR.rglob("*"):

    if path.is_file():

        metadata_files.append(path)

for path in metadata_files:

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{size_mb:8.2f} MB  {path}"
    )

print(
    "\n✓ Found",
    len(metadata_files),
    "files."
)

EXTRACTED ABO FILES
    0.01 MB  /content/neural_object_reconstruction/abo_metadata/LICENSE-CC-BY-4.0.txt
    0.01 MB  /content/neural_object_reconstruction/abo_metadata/listings/README.md
    5.24 MB  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_6.json.gz
    5.19 MB  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_2.json.gz
    5.19 MB  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_0.json.gz
    5.29 MB  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_4.json.gz
    5.21 MB  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_a.json.gz
    5.14 MB  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_3.json.gz
    5.19 MB  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_c.json.gz
    5.21 MB  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_f.json.gz
   

In [ ]:
# ============================================================
# STEP 11 — INSPECT ABO LISTING RECORDS
# ============================================================

import gzip
import json
from pathlib import Path

print("=" * 70)
print("INSPECTING ABO METADATA STRUCTURE")
print("=" * 70)

# Find the compressed listing files
listing_files = sorted(
    (
        ABO_METADATA_DIR /
        "listings" /
        "metadata"
    ).glob("listings_*.json.gz")
)

if not listing_files:
    raise RuntimeError(
        "No listings_*.json.gz files were found."
    )

print(
    f"\nFound {len(listing_files)} listing files."
)

# ------------------------------------------------------------
# Read ONLY the first few records from the first file.
# We deliberately do not load the entire file into RAM.
# ------------------------------------------------------------

test_file = listing_files[0]

print("\nInspecting:")
print(test_file)

records = []

with gzip.open(
    test_file,
    "rt",
    encoding="utf-8"
) as f:

    for i, line in enumerate(f):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
            records.append(record)

        except json.JSONDecodeError:
            print(
                "Could not decode line:",
                i
            )

        if len(records) >= 3:
            break


print(
    f"\n✓ Successfully read "
    f"{len(records)} sample records."
)

# ------------------------------------------------------------
# Display the structure
# ------------------------------------------------------------

for index, record in enumerate(records):

    print("\n")
    print("=" * 70)
    print(f"RECORD {index + 1}")
    print("=" * 70)

    print(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        )[:12000]
    )

print("\n" + "=" * 70)
print("TOP-LEVEL KEYS")
print("=" * 70)

for index, record in enumerate(records):

    print(
        f"\nRecord {index + 1}:"
    )

    for key in record.keys():

        print(
            "  -",
            key
        )

INSPECTING ABO METADATA STRUCTURE

Found 16 listing files.

Inspecting:
/content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_0.json.gz

✓ Successfully read 3 sample records.


RECORD 1
{
  "brand": [
    {
      "language_tag": "nl_NL",
      "value": "find."
    }
  ],
  "bullet_point": [
    {
      "language_tag": "nl_NL",
      "value": "Schoen in Loafer-stijl"
    },
    {
      "language_tag": "nl_NL",
      "value": "Platform hak"
    },
    {
      "language_tag": "nl_NL",
      "value": "Cap teen"
    },
    {
      "language_tag": "nl_NL",
      "value": "Middenhak"
    }
  ],
  "color": [
    {
      "language_tag": "nl_NL",
      "value": "Veelkleurig Vrouw Blauw"
    }
  ],
  "item_id": "B06X9STHNG",
  "item_name": [
    {
      "language_tag": "nl_NL",
      "value": "Amazon-merk - vinden. Dames Leder Gesloten Teen Hakken,Veelkleurig Vrouw Blauw,5 UK"
    }
  ],
  "model_name": [
    {
      "language_tag": "nl_NL",
      "value": "37753"
    }
  

In [ ]:
# ============================================================
# STEP 12 — READ ABO LICENSE
# ============================================================

LICENSE_FILE = (
    ABO_METADATA_DIR /
    "LICENSE-CC-BY-4.0.txt"
)

print("=" * 70)
print("ABO LICENSE")
print("=" * 70)

if not LICENSE_FILE.exists():

    raise FileNotFoundError(
        f"License file not found: {LICENSE_FILE}"
    )

license_text = LICENSE_FILE.read_text(
    encoding="utf-8"
)

print(license_text)

ABO LICENSE
Creative Commons Attribution 4.0 International Public License

By exercising the Licensed Rights (defined below), You accept and agree
to be bound by the terms and conditions of this Creative Commons
Attribution 4.0 International Public License ("Public License"). To the
extent this Public License may be interpreted as a contract, You are
granted the Licensed Rights in consideration of Your acceptance of
these terms and conditions, and the Licensor grants You such rights in
consideration of benefits the Licensor receives from making the
Licensed Material available under these terms and conditions.


Section 1 -- Definitions.

  a. Adapted Material means material subject to Copyright and Similar
     Rights that is derived from or based upon the Licensed Material
     and in which the Licensed Material is translated, altered,
     arranged, transformed, or otherwise modified in a manner requiring
     permission under the Copyright and Similar Rights held by the
     Licenso

In [ ]:
# ============================================================
# STEP 13 — COUNT ABO LISTING RECORDS
# ============================================================

import gzip
import json
from tqdm.auto import tqdm

total_records = 0

records_per_file = {}

print("=" * 70)
print("COUNTING ABO LISTINGS")
print("=" * 70)

for listing_file in listing_files:

    count = 0

    print(
        f"\nScanning: {listing_file.name}"
    )

    with gzip.open(
        listing_file,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:

                json.loads(line)

                count += 1

            except json.JSONDecodeError:

                continue

    records_per_file[
        listing_file.name
    ] = count

    total_records += count

    print(
        f"  Records: {count:,}"
    )

print("\n" + "=" * 70)
print("TOTAL")
print("=" * 70)

print(
    f"Total ABO listing records: "
    f"{total_records:,}"
)

print("\nRecords per file:")

for filename, count in records_per_file.items():

    print(
        f"{filename:20s} "
        f"{count:,}"
    )

COUNTING ABO LISTINGS

Scanning: listings_0.json.gz
  Records: 9,232

Scanning: listings_1.json.gz
  Records: 9,232

Scanning: listings_2.json.gz
  Records: 9,232

Scanning: listings_3.json.gz
  Records: 9,232

Scanning: listings_4.json.gz
  Records: 9,232

Scanning: listings_5.json.gz
  Records: 9,232

Scanning: listings_6.json.gz
  Records: 9,232

Scanning: listings_7.json.gz
  Records: 9,232

Scanning: listings_8.json.gz
  Records: 9,232

Scanning: listings_9.json.gz
  Records: 9,232

Scanning: listings_a.json.gz
  Records: 9,232

Scanning: listings_b.json.gz
  Records: 9,232

Scanning: listings_c.json.gz
  Records: 9,232

Scanning: listings_d.json.gz
  Records: 9,232

Scanning: listings_e.json.gz
  Records: 9,232

Scanning: listings_f.json.gz
  Records: 9,222

TOTAL
Total ABO listing records: 147,702

Records per file:
listings_0.json.gz   9,232
listings_1.json.gz   9,232
listings_2.json.gz   9,232
listings_3.json.gz   9,232
listings_4.json.gz   9,232
listings_5.json.gz   9,232
lis

In [ ]:
# ============================================================
# STEP 14 — BUILD ABO CANDIDATE POOL
# ============================================================
#
# We do NOT download 3D assets yet.
#
# We scan the metadata and select promising product listings.
#
# Target:
#     ~1,000 candidates
#
# Final target later:
#     250 high-quality objects
#
# ============================================================

import gzip
import json
import re
import random
from pathlib import Path
from collections import Counter, defaultdict

print("=" * 70)
print("BUILDING ABO CANDIDATE POOL")
print("=" * 70)


# ============================================================
# CONFIGURATION
# ============================================================

TARGET_CANDIDATES = 1000

RANDOM_SEED = 42

random.seed(RANDOM_SEED)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

CANDIDATE_DIR = (
    Path("/content/neural_object_reconstruction")
    / "abo_candidates"
)

CANDIDATE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CANDIDATE_FILE = (
    CANDIDATE_DIR /
    "candidate_pool.jsonl"
)


# ============================================================
# PRODUCT CATEGORIES
# ============================================================
#
# We use broad semantic groups rather than relying only on
# Amazon's product_type values because product_type can be
# extremely broad.
#
# ============================================================

CATEGORY_KEYWORDS = {

    "tools": [
        "hammer",
        "screwdriver",
        "wrench",
        "pliers",
        "drill",
        "saw",
        "tool",
        "clamp",
        "chisel",
        "axe",
        "mallet",
        "trowel"
    ],

    "kitchen": [
        "pan",
        "pot",
        "knife",
        "spoon",
        "fork",
        "plate",
        "bowl",
        "cup",
        "mug",
        "kettle",
        "blender",
        "toaster",
        "colander",
        "bottle",
        "container"
    ],

    "electronics": [
        "camera",
        "speaker",
        "headphone",
        "controller",
        "keyboard",
        "mouse",
        "phone",
        "tablet",
        "remote",
        "monitor",
        "microphone",
        "charger",
        "computer"
    ],

    "toys": [
        "toy",
        "robot",
        "doll",
        "figure",
        "action figure",
        "puzzle",
        "game",
        "lego",
        "vehicle",
        "car",
        "train",
        "airplane"
    ],

    "sports": [
        "ball",
        "racket",
        "racquet",
        "bat",
        "helmet",
        "glove",
        "skate",
        "bicycle",
        "bike",
        "football",
        "soccer",
        "basketball",
        "golf"
    ],

    "musical_instruments": [
        "guitar",
        "violin",
        "piano",
        "drum",
        "flute",
        "trumpet",
        "saxophone",
        "ukulele",
        "instrument"
    ],

    "decorative": [
        "statue",
        "sculpture",
        "figurine",
        "ornament",
        "decoration",
        "decorative",
        "bust",
        "trophy"
    ],

    "fashion_accessories": [
        "watch",
        "handbag",
        "bag",
        "wallet",
        "shoe",
        "boot",
        "sandal",
        "jewelry",
        "necklace",
        "bracelet",
        "ring",
        "glasses"
    ],

    "containers": [
        "box",
        "jar",
        "bottle",
        "case",
        "basket",
        "bin",
        "storage",
        "container",
        "canister"
    ],

    "mechanical": [
        "gear",
        "bearing",
        "motor",
        "engine",
        "mechanical",
        "component",
        "bracket",
        "mount",
        "hinge",
        "valve"
    ]
}


# ============================================================
# WORDS WE DO NOT WANT
# ============================================================

EXCLUDE_KEYWORDS = [

    # Furniture
    "sofa",
    "couch",
    "chair",
    "table",
    "desk",
    "bed",
    "mattress",
    "cabinet",
    "wardrobe",
    "bookshelf",
    "shelf",

    # Very large / architectural
    "door",
    "window",
    "roof",
    "wall",
    "floor",
    "building",
    "house",

    # Clothing
    "shirt",
    "dress",
    "pants",
    "trousers",
    "jacket",
    "coat",

    # Consumables / food
    "food",
    "snack",
    "chocolate",
    "candy",
    "coffee",
    "tea",
    "juice",
    "water",
    "soap",
    "shampoo",

    # Packaging-heavy products
    "pack of",
    "case of",
    "set of 50",
    "set of 100",
    "refill",

    # Extremely generic products
    "replacement part",
    "spare part"
]


# ============================================================
# TEXT EXTRACTION
# ============================================================

def extract_values(record, field):

    values = record.get(field, [])

    if not isinstance(values, list):

        return []

    output = []

    for value in values:

        if isinstance(value, dict):

            text = value.get("value")

            if text is not None:

                output.append(
                    str(text)
                )

        elif isinstance(value, str):

            output.append(value)

    return output


def get_search_text(record):

    parts = []

    for field in [
        "item_name",
        "item_keywords",
        "bullet_point",
        "style",
        "brand",
        "material",
        "color"
    ]:

        parts.extend(
            extract_values(
                record,
                field
            )
        )

    for node in record.get(
        "node",
        []
    ):

        if isinstance(node, dict):

            node_name = node.get(
                "node_name"
            )

            if node_name:

                parts.append(
                    str(node_name)
                )

    return " ".join(
        parts
    ).lower()


# ============================================================
# CATEGORY DETECTION
# ============================================================

def detect_category(text):

    matches = []

    for category, keywords in (
        CATEGORY_KEYWORDS.items()
    ):

        for keyword in keywords:

            if keyword.lower() in text:

                matches.append(
                    category
                )

                break

    if not matches:

        return None

    # Prefer the first detected category.
    return matches[0]


# ============================================================
# EXCLUSION CHECK
# ============================================================

def is_excluded(text):

    for keyword in EXCLUDE_KEYWORDS:

        if keyword.lower() in text:

            return True

    return False


# ============================================================
# SCORE CANDIDATE
# ============================================================

def candidate_score(
    record,
    text,
    category
):

    score = 0

    # --------------------------------------------------------
    # Product name exists
    # --------------------------------------------------------

    if record.get("item_name"):

        score += 10

    # --------------------------------------------------------
    # Keywords available
    # --------------------------------------------------------

    keyword_count = len(
        extract_values(
            record,
            "item_keywords"
        )
    )

    score += min(
        keyword_count,
        10
    )

    # --------------------------------------------------------
    # Material information
    # --------------------------------------------------------

    if record.get("material"):

        score += 5

    # --------------------------------------------------------
    # Dimensions
    # --------------------------------------------------------

    if record.get(
        "item_dimensions"
    ):

        score += 5

    # --------------------------------------------------------
    # Color
    # --------------------------------------------------------

    if record.get("color"):

        score += 2

    # --------------------------------------------------------
    # Multiple images
    # --------------------------------------------------------

    image_count = len(
        record.get(
            "other_image_id",
            []
        )
    )

    score += min(
        image_count,
        5
    )

    # --------------------------------------------------------
    # Category
    # --------------------------------------------------------

    if category:

        score += 10

    return score


# ============================================================
# STREAM THROUGH DATASET
# ============================================================

category_candidates = defaultdict(list)

seen_item_ids = set()

total_scanned = 0
accepted = 0
excluded = 0
uncategorized = 0


print(
    "\nScanning ABO metadata..."
)


for listing_file in listing_files:

    print(
        f"\nProcessing "
        f"{listing_file.name}"
    )

    with gzip.open(
        listing_file,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:

                continue

            try:

                record = json.loads(
                    line
                )

            except json.JSONDecodeError:

                continue

            total_scanned += 1

            item_id = record.get(
                "item_id"
            )

            if not item_id:

                continue

            # ------------------------------------------------
            # Duplicate protection
            # ------------------------------------------------

            if item_id in seen_item_ids:

                continue

            seen_item_ids.add(
                item_id
            )

            # ------------------------------------------------
            # Searchable text
            # ------------------------------------------------

            text = get_search_text(
                record
            )

            # ------------------------------------------------
            # Exclude undesirable products
            # ------------------------------------------------

            if is_excluded(text):

                excluded += 1

                continue

            # ------------------------------------------------
            # Detect category
            # ------------------------------------------------

            category = detect_category(
                text
            )

            if category is None:

                uncategorized += 1

                continue

            # ------------------------------------------------
            # Score
            # ------------------------------------------------

            score = candidate_score(
                record,
                text,
                category
            )

            category_candidates[
                category
            ].append(
                (
                    score,
                    record
                )
            )

            accepted += 1


print("\n" + "=" * 70)
print("SCAN COMPLETE")
print("=" * 70)

print(
    f"Records scanned:      {total_scanned:,}"
)

print(
    f"Accepted candidates:  {accepted:,}"
)

print(
    f"Excluded:              {excluded:,}"
)

print(
    f"Uncategorized:         {uncategorized:,}"
)


# ============================================================
# SELECT TOP CANDIDATES PER CATEGORY
# ============================================================

print("\n" + "=" * 70)
print("CATEGORY COUNTS")
print("=" * 70)

for category in sorted(
    category_candidates
):

    count = len(
        category_candidates[
            category
        ]
    )

    print(
        f"{category:25s}"
        f"{count:8,}"
    )


# ============================================================
# BALANCED SELECTION
# ============================================================
#
# We don't want:
#
#     900 shoes
#     50 tools
#     20 toys
#
# Instead, distribute the candidates across categories.
# ============================================================

categories = sorted(
    category_candidates.keys()
)

if not categories:

    raise RuntimeError(
        "No suitable categories were found."
    )


per_category = max(
    1,
    TARGET_CANDIDATES // len(categories)
)


selected_candidates = []


for category in categories:

    candidates = category_candidates[
        category
    ]

    # Highest-quality records first.
    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    selected = candidates[
        :per_category
    ]

    for score, record in selected:

        selected_candidates.append({

            "item_id":
                record["item_id"],

            "category":
                category,

            "score":
                score,

            "record":
                record

        })


# ============================================================
# TOP-UP IF WE ARE BELOW TARGET
# ============================================================

if len(selected_candidates) < TARGET_CANDIDATES:

    remaining = []

    selected_ids = {
        x["item_id"]
        for x in selected_candidates
    }

    for category in categories:

        for score, record in (
            category_candidates[
                category
            ]
        ):

            if (
                record["item_id"]
                not in selected_ids
            ):

                remaining.append(
                    (
                        score,
                        category,
                        record
                    )
                )

    remaining.sort(
        key=lambda x: x[0],
        reverse=True
    )

    needed = (
        TARGET_CANDIDATES
        - len(selected_candidates)
    )

    for score, category, record in (
        remaining[:needed]
    ):

        selected_candidates.append({

            "item_id":
                record["item_id"],

            "category":
                category,

            "score":
                score,

            "record":
                record

        })


# ============================================================
# WRITE JSONL
# ============================================================

with open(
    CANDIDATE_FILE,
    "w",
    encoding="utf-8"
) as f:

    for candidate in selected_candidates:

        f.write(
            json.dumps(
                candidate,
                ensure_ascii=False
            )
            + "\n"
        )


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("CANDIDATE POOL CREATED")
print("=" * 70)

print(
    f"Candidates selected: "
    f"{len(selected_candidates):,}"
)

print(
    f"Saved to:\n"
    f"{CANDIDATE_FILE}"
)


print("\nCategory distribution:")

distribution = Counter(
    x["category"]
    for x in selected_candidates
)

for category, count in sorted(
    distribution.items()
):

    print(
        f"  {category:25s}"
        f"{count:5d}"
    )

BUILDING ABO CANDIDATE POOL

Scanning ABO metadata...

Processing listings_0.json.gz

Processing listings_1.json.gz

Processing listings_2.json.gz

Processing listings_3.json.gz

Processing listings_4.json.gz

Processing listings_5.json.gz

Processing listings_6.json.gz

Processing listings_7.json.gz

Processing listings_8.json.gz

Processing listings_9.json.gz

Processing listings_a.json.gz

Processing listings_b.json.gz

Processing listings_c.json.gz

Processing listings_d.json.gz

Processing listings_e.json.gz

Processing listings_f.json.gz

SCAN COMPLETE
Records scanned:      147,702
Accepted candidates:  67,049
Excluded:              67,492
Uncategorized:         11,074

CATEGORY COUNTS
containers                  1,007
decorative                    340
electronics                 5,667
fashion_accessories         8,293
kitchen                    42,437
mechanical                    161
musical_instruments           118
sports                      1,733
tools                      

In [ ]:
# ============================================================
# STEP 15 — INSPECT ABO DATASET STRUCTURE
# ============================================================
#
# We need to locate the ABO 3D asset metadata.
#
# IMPORTANT:
# This cell DOES NOT download the 154 GB 3D archive.
#
# It only examines what we already have locally.
# ============================================================

from pathlib import Path

ABO_ROOT = Path(
    "/content/neural_object_reconstruction/abo_metadata"
)

print("=" * 70)
print("ABO DATASET STRUCTURE")
print("=" * 70)

if not ABO_ROOT.exists():

    raise FileNotFoundError(
        f"ABO directory not found:\n{ABO_ROOT}"
    )


# ------------------------------------------------------------
# Show directories
# ------------------------------------------------------------

print("\nDIRECTORIES")
print("-" * 70)

directories = sorted(
    p for p in ABO_ROOT.rglob("*")
    if p.is_dir()
)

for directory in directories:

    relative = directory.relative_to(
        ABO_ROOT
    )

    print(
        "DIR:",
        relative
    )


# ------------------------------------------------------------
# Show files
# ------------------------------------------------------------

print("\nFILES")
print("-" * 70)

files = sorted(
    p for p in ABO_ROOT.rglob("*")
    if p.is_file()
)

for file in files:

    relative = file.relative_to(
        ABO_ROOT
    )

    size_mb = (
        file.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{size_mb:8.2f} MB   {relative}"
    )


# ------------------------------------------------------------
# Look specifically for 3D-related files
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("POSSIBLE 3D / MODEL FILES")
print("=" * 70)

three_d_extensions = {
    ".json",
    ".jsonl",
    ".gz",
    ".tar",
    ".zip",
    ".glb",
    ".gltf",
    ".obj",
    ".ply",
    ".bin"
}

three_d_keywords = [
    "3d",
    "model",
    "mesh",
    "asset",
    "metadata",
    "listing"
]

matches = []

for file in files:

    name = file.name.lower()

    if (
        file.suffix.lower()
        in three_d_extensions
        and any(
            keyword in name
            for keyword in three_d_keywords
        )
    ):

        matches.append(file)


if matches:

    for file in matches:

        size_mb = (
            file.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{size_mb:8.2f} MB   "
            f"{file.relative_to(ABO_ROOT)}"
        )

else:

    print(
        "No obvious 3D metadata files found "
        "in the downloaded ABO metadata."
    )

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ABO DATASET STRUCTURE

DIRECTORIES
----------------------------------------------------------------------
DIR: listings
DIR: listings/metadata

FILES
----------------------------------------------------------------------
    0.01 MB   LICENSE-CC-BY-4.0.txt
    0.01 MB   listings/README.md
    5.19 MB   listings/metadata/listings_0.json.gz
    5.10 MB   listings/metadata/listings_1.json.gz
    5.19 MB   listings/metadata/listings_2.json.gz
    5.14 MB   listings/metadata/listings_3.json.gz
    5.29 MB   listings/metadata/listings_4.json.gz
    5.32 MB   listings/metadata/listings_5.json.gz
    5.24 MB   listings/metadata/listings_6.json.gz
    5.18 MB   listings/metadata/listings_7.json.gz
    5.31 MB   listings/metadata/listings_8.json.gz
    5.24 MB   listings/metadata/listings_9.json.gz
    5.21 MB   listings/metadata/listings_a.json.gz
    5.16 MB   listings/metadata/listings_b.json.gz
    5.19 MB   listings/metadata/listings_c.json.gz
    5.16 MB   listings/metadata/listings_d.json

In [ ]:
# ============================================================
# STEP 16 — FIND ABO 3D ASSET MAPPING
# ============================================================
#
# IMPORTANT:
# We are NOT downloading any 3D models yet.
#
# We first need to determine exactly how ABO identifies
# its 3D assets.
# ============================================================

from pathlib import Path
import gzip
import json
import re

ABO_ROOT = Path(
    "/content/neural_object_reconstruction/abo_metadata"
)

LISTINGS_DIR = (
    ABO_ROOT
    / "listings"
    / "metadata"
)

print("=" * 70)
print("SEARCHING FOR ABO 3D ASSET MAPPING")
print("=" * 70)

# ------------------------------------------------------------
# 1. Search every local filename for 3D-related files
# ------------------------------------------------------------

print("\n1. 3D-RELATED FILES")
print("-" * 70)

all_files = list(
    ABO_ROOT.rglob("*")
)

three_d_file_matches = []

for path in all_files:

    if not path.is_file():
        continue

    name = path.name.lower()

    keywords = [
        "3d",
        "model",
        "mesh",
        "asset",
        "object",
        "glb",
        "gltf",
        "obj"
    ]

    if any(
        keyword in name
        for keyword in keywords
    ):
        three_d_file_matches.append(path)


if three_d_file_matches:

    for path in three_d_file_matches:

        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{size_mb:8.2f} MB   "
            f"{path.relative_to(ABO_ROOT)}"
        )

else:

    print(
        "No separate 3D-related files found."
    )


# ------------------------------------------------------------
# 2. Search JSON keys for possible 3D fields
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. SEARCHING JSON KEYS")
print("=" * 70)

possible_keys = set()

sample_records = 0

listing_files = sorted(
    LISTINGS_DIR.glob("*.json.gz")
)

for listing_file in listing_files:

    print(
        f"\nScanning: {listing_file.name}"
    )

    try:

        with gzip.open(
            listing_file,
            "rt",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)

                except json.JSONDecodeError:
                    continue

                sample_records += 1

                # Top-level keys
                for key in record.keys():

                    key_lower = key.lower()

                    if any(
                        term in key_lower
                        for term in [
                            "3d",
                            "model",
                            "mesh",
                            "asset",
                            "object"
                        ]
                    ):

                        possible_keys.add(key)

                # Nested values converted to text
                # so we can look for possible identifiers.
                record_text = json.dumps(
                    record
                ).lower()

                if any(
                    term in record_text
                    for term in [
                        "glb",
                        "gltf",
                        ".obj",
                        "3d model",
                        "3dmodel"
                    ]
                ):

                    print(
                        "  Possible 3D reference found "
                        "in a sample record."
                    )

                # We only need a small number of samples
                # from each file for structural inspection.
                if sample_records >= 20:
                    break

    except Exception as e:

        print(
            "  ERROR:",
            e
        )

    sample_records = 0


print("\nPossible 3D-related JSON keys:")

if possible_keys:

    for key in sorted(
        possible_keys
    ):
        print(
            "  -",
            key
        )

else:

    print(
        "  None found."
    )


# ------------------------------------------------------------
# 3. Search README for 3D information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. SEARCHING ABO README")
print("=" * 70)

readme = (
    ABO_ROOT
    / "listings"
    / "README.md"
)

if readme.exists():

    text = readme.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    lines = text.splitlines()

    found = False

    for i, line in enumerate(lines):

        lower = line.lower()

        if any(
            term in lower
            for term in [
                "3d",
                "model",
                "mesh",
                "asset"
            ]
        ):

            found = True

            start = max(
                0,
                i - 3
            )

            end = min(
                len(lines),
                i + 4
            )

            print()

            for context_line in lines[
                start:end
            ]:

                print(
                    context_line
                )

    if not found:

        print(
            "No 3D/model references found "
            "in README."
        )

else:

    print(
        "README not found."
    )


print("\n" + "=" * 70)
print("3D MAPPING SEARCH COMPLETE")
print("=" * 70)

SEARCHING FOR ABO 3D ASSET MAPPING

1. 3D-RELATED FILES
----------------------------------------------------------------------
No separate 3D-related files found.

2. SEARCHING JSON KEYS

Scanning: listings_0.json.gz

Scanning: listings_1.json.gz

Scanning: listings_2.json.gz
  Possible 3D reference found in a sample record.

Scanning: listings_3.json.gz
  Possible 3D reference found in a sample record.

Scanning: listings_4.json.gz
  Possible 3D reference found in a sample record.
  Possible 3D reference found in a sample record.
  Possible 3D reference found in a sample record.

Scanning: listings_5.json.gz
  Possible 3D reference found in a sample record.

Scanning: listings_6.json.gz
  Possible 3D reference found in a sample record.
  Possible 3D reference found in a sample record.

Scanning: listings_7.json.gz

Scanning: listings_8.json.gz
  Possible 3D reference found in a sample record.

Scanning: listings_9.json.gz
  Possible 3D reference found in a sample record.
  Possible 3D

In [ ]:
# ============================================================
# STEP 17 — DOWNLOAD ABO 3D MODEL METADATA
# ============================================================
#
# This downloads ONLY the small 3D metadata file.
#
# We are NOT downloading the 154 GB 3D model archive.
#
# Correct ABO filename:
#
# 3dmodels/metadata/3dmodels.csv.gz
#
# ============================================================

from pathlib import Path
import requests
import os

# ------------------------------------------------------------
# PROJECT DIRECTORIES
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/neural_object_reconstruction"
)

ABO_ROOT = (
    PROJECT_ROOT
    / "abo_metadata"
)

MODEL_METADATA_DIR = (
    ABO_ROOT
    / "3dmodels"
    / "metadata"
)

MODEL_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    MODEL_METADATA_DIR
    / "3dmodels.csv.gz"
)

# ------------------------------------------------------------
# CORRECT OFFICIAL ABO URL
# ------------------------------------------------------------

URL = (
    "https://amazon-berkeley-objects"
    ".s3.us-east-1.amazonaws.com/"
    "3dmodels/metadata/3dmodels.csv.gz"
)

print("=" * 70)
print("ABO 3D MODEL METADATA DOWNLOAD")
print("=" * 70)

print()
print("Source:")
print(URL)

print()
print("Destination:")
print(OUTPUT_FILE)

print()
print("Downloading METADATA ONLY.")
print("No 3D models are being downloaded yet.")

# ------------------------------------------------------------
# EXISTING FILE CHECK
# ------------------------------------------------------------

if OUTPUT_FILE.exists():

    size_mb = (
        OUTPUT_FILE.stat().st_size
        / (1024 ** 2)
    )

    print()
    print(
        f"✓ Metadata already exists "
        f"({size_mb:.2f} MB)"
    )

else:

    print()
    print("Downloading...")

    temp_file = (
        OUTPUT_FILE.with_suffix(
            ".tmp"
        )
    )

    try:

        with requests.get(
            URL,
            stream=True,
            timeout=120
        ) as response:

            print(
                "HTTP status:",
                response.status_code
            )

            response.raise_for_status()

            total = int(
                response.headers.get(
                    "content-length",
                    0
                )
            )

            downloaded = 0

            with open(
                temp_file,
                "wb"
            ) as f:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if not chunk:
                        continue

                    f.write(chunk)

                    downloaded += len(chunk)

                    if total:

                        percent = (
                            downloaded
                            / total
                            * 100
                        )

                        print(
                            f"\r"
                            f"Downloaded: "
                            f"{downloaded / 1024**2:.2f} MB"
                            f" / "
                            f"{total / 1024**2:.2f} MB"
                            f" "
                            f"({percent:.1f}%)",
                            end=""
                        )

                    else:

                        print(
                            f"\r"
                            f"Downloaded: "
                            f"{downloaded / 1024**2:.2f} MB",
                            end=""
                        )

        print()

        # Only rename after the download
        # has completed successfully.
        temp_file.replace(
            OUTPUT_FILE
        )

        print(
            "✓ Download completed."
        )

    except Exception as e:

        if temp_file.exists():
            temp_file.unlink()

        print()
        print(
            "✗ DOWNLOAD FAILED"
        )

        raise e


# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

if not OUTPUT_FILE.exists():

    raise RuntimeError(
        "3D model metadata file does not exist."
    )

size_mb = (
    OUTPUT_FILE.stat().st_size
    / (1024 ** 2)
)

print()
print("=" * 70)
print("SUCCESS")
print("=" * 70)

print(
    f"File: {OUTPUT_FILE}"
)

print(
    f"Size: {size_mb:.2f} MB"
)

print()
print(
    "✓ ABO 3D metadata is ready."
)

print()
print(
    "NEXT:"
)

print(
    "We will inspect the 3D metadata and "
    "build a quality filter before downloading models."
)

ABO 3D MODEL METADATA DOWNLOAD

Source:
https://amazon-berkeley-objects.s3.us-east-1.amazonaws.com/3dmodels/metadata/3dmodels.csv.gz

Destination:
/content/neural_object_reconstruction/abo_metadata/3dmodels/metadata/3dmodels.csv.gz

No 3D models are being downloaded yet.

Downloading...
HTTP status: 200
Downloaded: 0.32 MB / 0.32 MB (100.0%)
✓ Download completed.

SUCCESS
File: /content/neural_object_reconstruction/abo_metadata/3dmodels/metadata/3dmodels.csv.gz
Size: 0.32 MB

✓ ABO 3D metadata is ready.

NEXT:
We will inspect the 3D metadata and build a quality filter before downloading models.


In [ ]:
# ============================================================
# STEP 18 — INSPECT ABO 3D MODEL METADATA
# ============================================================
#
# Purpose:
#   Read the 3D metadata WITHOUT downloading any 3D models.
#
# We want to discover:
#   - available columns
#   - number of 3D models
#   - vertex counts
#   - face counts
#   - texture information
#   - material information
#   - model paths
#   - geometry dimensions
#   - duplicate / suspicious records
#
# IMPORTANT:
#   This cell does NOT download GLB files.
#
# ============================================================

from pathlib import Path
import pandas as pd
import gzip
import csv
import json
import os

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/neural_object_reconstruction"
)

MODEL_METADATA_FILE = (
    PROJECT_ROOT
    / "abo_metadata"
    / "3dmodels"
    / "metadata"
    / "3dmodels.csv.gz"
)

if not MODEL_METADATA_FILE.exists():

    raise FileNotFoundError(
        f"3D metadata not found:\n"
        f"{MODEL_METADATA_FILE}"
    )

print("=" * 70)
print("ABO 3D MODEL METADATA INSPECTION")
print("=" * 70)

print()
print("Metadata file:")
print(MODEL_METADATA_FILE)

print()

file_size_mb = (
    MODEL_METADATA_FILE.stat().st_size
    / (1024 ** 2)
)

print(
    f"File size: {file_size_mb:.2f} MB"
)

# ------------------------------------------------------------
# READ CSV
# ------------------------------------------------------------

print()
print("Reading metadata...")

df = pd.read_csv(
    MODEL_METADATA_FILE,
    compression="gzip"
)

print(
    f"✓ Metadata loaded."
)

# ------------------------------------------------------------
# BASIC INFORMATION
# ------------------------------------------------------------

print()
print("=" * 70)
print("BASIC INFORMATION")
print("=" * 70)

print()

print(
    "Number of 3D models:",
    len(df)
)

print()

print(
    "Number of columns:",
    len(df.columns)
)

print()

print("Columns:")
for column in df.columns:
    print(
        f"  - {column}"
    )

# ------------------------------------------------------------
# DATA TYPES
# ------------------------------------------------------------

print()
print("=" * 70)
print("DATA TYPES")
print("=" * 70)

print()

print(
    df.dtypes.to_string()
)

# ------------------------------------------------------------
# FIRST RECORDS
# ------------------------------------------------------------

print()
print("=" * 70)
print("FIRST 10 RECORDS")
print("=" * 70)

print()

display(
    df.head(10)
)

# ------------------------------------------------------------
# MISSING VALUES
# ------------------------------------------------------------

print()
print("=" * 70)
print("MISSING VALUES")
print("=" * 70)

print()

missing = (
    df.isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

for column, count in missing.items():

    percentage = (
        count
        / len(df)
        * 100
    )

    print(
        f"{column:35} "
        f"{count:8} "
        f"({percentage:6.2f}%)"
    )

# ------------------------------------------------------------
# UNIQUE VALUES
# ------------------------------------------------------------

print()
print("=" * 70)
print("UNIQUE VALUE COUNTS")
print("=" * 70)

print()

for column in df.columns:

    try:

        unique_count = (
            df[column]
            .nunique(
                dropna=True
            )
        )

        print(
            f"{column:35} "
            f"{unique_count}"
        )

    except Exception:

        pass

# ------------------------------------------------------------
# NUMERIC SUMMARY
# ------------------------------------------------------------

print()
print("=" * 70)
print("NUMERIC SUMMARY")
print("=" * 70)

print()

numeric_columns = (
    df.select_dtypes(
        include="number"
    )
    .columns
)

if len(numeric_columns) > 0:

    display(
        df[numeric_columns]
        .describe()
        .T
    )

else:

    print(
        "No numeric columns detected."
    )

# ------------------------------------------------------------
# SEARCH FOR IMPORTANT QUALITY COLUMNS
# ------------------------------------------------------------

print()
print("=" * 70)
print("POSSIBLE QUALITY COLUMNS")
print("=" * 70)

print()

quality_keywords = [
    "vertex",
    "face",
    "mesh",
    "texture",
    "material",
    "geometry",
    "dimension",
    "extent",
    "width",
    "height",
    "depth",
    "path",
    "model"
]

quality_columns = []

for column in df.columns:

    column_lower = (
        str(column)
        .lower()
    )

    if any(
        keyword in column_lower
        for keyword in quality_keywords
    ):

        quality_columns.append(
            column
        )

if quality_columns:

    for column in quality_columns:

        print(
            f"  ✓ {column}"
        )

else:

    print(
        "No obvious quality columns found."
    )

# ------------------------------------------------------------
# SAVE INSPECTION COPY
# ------------------------------------------------------------

INSPECTION_FILE = (
    PROJECT_ROOT
    / "abo_candidates"
    / "3d_metadata_inspection.csv"
)

INSPECTION_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    INSPECTION_FILE,
    index=False
)

print()
print("=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

print()

print(
    "Models:",
    len(df)
)

print(
    "Columns:",
    len(df.columns)
)

print()

print(
    "Inspection copy saved to:"
)

print(
    INSPECTION_FILE
)

print()
print(
    "IMPORTANT:"
)

print(
    "No 3D model files were downloaded."
)

print(
    "The next step will use this metadata "
    "to identify high-quality models."
)

ABO 3D MODEL METADATA INSPECTION

Metadata file:
/content/neural_object_reconstruction/abo_metadata/3dmodels/metadata/3dmodels.csv.gz

File size: 0.32 MB

Reading metadata...
✓ Metadata loaded.

BASIC INFORMATION

Number of 3D models: 7953

Number of columns: 15

Columns:
  - 3dmodel_id
  - path
  - meshes
  - materials
  - textures
  - images
  - image_height_max
  - image_height_min
  - image_width_max
  - image_width_min
  - vertices
  - faces
  - extent_x
  - extent_y
  - extent_z

DATA TYPES

3dmodel_id              str
path                    str
meshes                int64
materials             int64
textures              int64
images                int64
image_height_max    float64
image_height_min    float64
image_width_max     float64
image_width_min     float64
vertices              int64
faces                 int64
extent_x            float64
extent_y            float64
extent_z            float64

FIRST 10 RECORDS



,3dmodel_id,path,meshes,materials,textures,images,image_height_max,image_height_min,image_width_max,image_width_min,vertices,faces,extent_x,extent_y,extent_z
0,B01N2PLWIL,L/B01N2PLWIL.glb,1,1,3,3,4096.0,4096.0,4096.0,4096.0,10990,14380,0.571500,0.116840,0.071120
1,B075QFCHM9,9/B075QFCHM9.glb,1,1,3,3,2048.0,2048.0,2048.0,2048.0,11973,19568,1.840072,1.066910,2.367592
2,B07H469871,1/B07H469871.glb,1,1,3,3,4096.0,4096.0,4096.0,4096.0,1602,1950,1.111352,1.388081,0.397949
3,B07H8V49M2,2/B07H8V49M2.glb,1,1,3,3,4096.0,4096.0,4096.0,4096.0,3760,5710,1.499870,2.119884,0.589788
4,B07DBHPK4G,G/B07DBHPK4G.glb,1,1,3,3,4096.0,4096.0,4096.0,4096.0,13704,22736,0.379215,1.622815,0.379215
5,B0842LM2DN,N/B0842LM2DN.glb,1,1,3,3,4096.0,4096.0,4096.0,4096.0,4078,7584,0.227790,0.234859,0.227790
6,B07HK6B4D7,7/B07HK6B4D7.glb,1,1,4,4,2048.0,2048.0,2048.0,2048.0,12221,19268,0.188729,0.665094,0.421692
7,B07B4FZN9H,H/B07B4FZN9H.glb,1,1,3,3,2048.0,2048.0,2048.0,2048.0,13595,22644,3.383829,0.996365,2.048074
8,B07B4Z9BS4,4/B07B4Z9BS4.glb,1,1,4,4,2048.0,2048.0,2048.0,2048.0,9259,16178,0.279354,0.269393,0.279354
9,B07B7N6JH3,3/B07B7N6JH3.glb,1,1,3,3,4096.0,2048.0,4096.0,2048.0,72222,132736,0.506930,0.550388,0.400410



MISSING VALUES

image_height_max                           4 (  0.05%)
image_width_max                            4 (  0.05%)
image_height_min                           4 (  0.05%)
image_width_min                            4 (  0.05%)
meshes                                     0 (  0.00%)
path                                       0 (  0.00%)
3dmodel_id                                 0 (  0.00%)
images                                     0 (  0.00%)
textures                                   0 (  0.00%)
materials                                  0 (  0.00%)
vertices                                   0 (  0.00%)
faces                                      0 (  0.00%)
extent_x                                   0 (  0.00%)
extent_y                                   0 (  0.00%)
extent_z                                   0 (  0.00%)

UNIQUE VALUE COUNTS

3dmodel_id                          7953
path                                7953
meshes                              3
materials       

,count,mean,std,min,25%,50%,75%,max
meshes,7953.0,1.004652,0.285645,1.000000,1.000000,1.000000,1.000000,1.900000e+01
materials,7953.0,1.006664,0.082898,1.000000,1.000000,1.000000,1.000000,3.000000e+00
textures,7953.0,3.044889,0.222867,0.000000,3.000000,3.000000,3.000000,6.000000e+00
images,7953.0,3.044889,0.222867,0.000000,3.000000,3.000000,3.000000,6.000000e+00
image_height_max,7949.0,3141.305825,1023.457926,1024.000000,2048.000000,4096.000000,4096.000000,8.192000e+03
image_height_min,7949.0,3122.755693,1025.060109,1.000000,2048.000000,4096.000000,4096.000000,8.192000e+03
image_width_max,7949.0,3141.305825,1023.457926,1024.000000,2048.000000,4096.000000,4096.000000,8.192000e+03
image_width_min,7949.0,3122.755693,1025.060109,1.000000,2048.000000,4096.000000,4096.000000,8.192000e+03
vertices,7953.0,24548.267446,89941.822431,56.000000,4396.000000,10990.000000,22898.000000,5.870562e+06
faces,7953.0,42768.405130,174330.788284,20.000000,6964.000000,18240.000000,38776.000000,1.154022e+07



POSSIBLE QUALITY COLUMNS

  ✓ 3dmodel_id
  ✓ path
  ✓ meshes
  ✓ materials
  ✓ textures
  ✓ image_height_max
  ✓ image_height_min
  ✓ image_width_max
  ✓ image_width_min
  ✓ faces
  ✓ extent_x
  ✓ extent_y
  ✓ extent_z

INSPECTION COMPLETE

Models: 7953
Columns: 15

Inspection copy saved to:
/content/neural_object_reconstruction/abo_candidates/3d_metadata_inspection.csv

IMPORTANT:
No 3D model files were downloaded.
The next step will use this metadata to identify high-quality models.


In [ ]:
# ============================================================
# STEP 2 — BUILD ACTUAL 3D PRODUCT POOL
# ============================================================

import os
import json
import gzip
import pandas as pd

BASE = "/content/neural_object_reconstruction"

META = os.path.join(
    BASE,
    "abo_metadata/3dmodels/metadata/3dmodels.csv.gz"
)

LISTINGS = os.path.join(
    BASE,
    "abo_metadata/listings/metadata"
)

OUT = os.path.join(
    BASE,
    "abo_candidates/3d_product_pool.jsonl"
)

print("=" * 60)
print("BUILDING ACTUAL ABO 3D PRODUCT POOL")
print("=" * 60)

models = pd.read_csv(
    META,
    compression="gzip"
)

model_ids = set(
    models["3dmodel_id"].astype(str)
)

print(f"3D models available: {len(model_ids):,}")

results = []

files = sorted(
    f for f in os.listdir(LISTINGS)
    if f.endswith(".json.gz")
)

for filename in files:

    path = os.path.join(LISTINGS, filename)

    print(f"Scanning {filename}...")

    with gzip.open(
        path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:
                r = json.loads(line)
            except:
                continue

            model_id = r.get("3dmodel_id")

            if model_id is None:
                continue

            model_id = str(model_id)

            if model_id not in model_ids:
                continue

            results.append({
                "item_id": str(
                    r.get("item_id", "")
                ),
                "3dmodel_id": model_id,
                "record": r
            })

# Remove duplicate products
unique = {}

for x in results:
    unique[x["3dmodel_id"]] = x

results = list(unique.values())

print()
print("=" * 60)
print("RESULT")
print("=" * 60)

print(
    f"Products with actual 3D models: {len(results):,}"
)

with open(
    OUT,
    "w",
    encoding="utf-8"
) as f:

    for x in results:

        f.write(
            json.dumps(
                x,
                ensure_ascii=False
            ) + "\n"
        )

print()
print("Saved:")
print(OUT)

BUILDING ACTUAL ABO 3D PRODUCT POOL
3D models available: 7,953
Scanning listings_0.json.gz...
Scanning listings_1.json.gz...
Scanning listings_2.json.gz...
Scanning listings_3.json.gz...
Scanning listings_4.json.gz...
Scanning listings_5.json.gz...
Scanning listings_6.json.gz...
Scanning listings_7.json.gz...
Scanning listings_8.json.gz...
Scanning listings_9.json.gz...
Scanning listings_a.json.gz...
Scanning listings_b.json.gz...
Scanning listings_c.json.gz...
Scanning listings_d.json.gz...
Scanning listings_e.json.gz...
Scanning listings_f.json.gz...

RESULT
Products with actual 3D models: 7,953

Saved:
/content/neural_object_reconstruction/abo_candidates/3d_product_pool.jsonl


In [ ]:
import os
import json
import gzip
import pandas as pd
import numpy as np

BASE = "/content/neural_object_reconstruction"

POOL_FILE = f"{BASE}/abo_candidates/3d_product_pool.jsonl"
META_FILE = f"{BASE}/abo_metadata/3dmodels/metadata/3dmodels.csv.gz"
OUT_FILE = f"{BASE}/abo_candidates/3d_quality_pool.jsonl"

TARGET = 250

print("=" * 60)
print("BUILDING 3D QUALITY-RANKED POOL")
print("=" * 60)

# ------------------------------------------------------------
# Load product pool
# ------------------------------------------------------------

products = []

with open(POOL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            products.append(json.loads(line))

print(f"Products loaded: {len(products):,}")

# ------------------------------------------------------------
# Load 3D metadata
# ------------------------------------------------------------

meta = pd.read_csv(META_FILE, compression="gzip")

print(f"3D metadata loaded: {len(meta):,}")

# ------------------------------------------------------------
# Normalize IDs
# ------------------------------------------------------------

meta["3dmodel_id"] = meta["3dmodel_id"].astype(str)

product_map = {}

for item in products:
    item_id = str(item.get("item_id", ""))

    model_id = item.get("3dmodel_id")

    if model_id:
        product_map[item_id] = str(model_id)

# ------------------------------------------------------------
# If the pool uses a different field, detect it
# ------------------------------------------------------------

if not product_map:
    possible_fields = [
        "3dmodel_id",
        "model_id",
        "item_id"
    ]

    sample = products[0]

    print("\nAvailable fields:")
    print(list(sample.keys()))

    raise RuntimeError(
        "Could not identify product → 3D model mapping."
    )

# ------------------------------------------------------------
# Join metadata
# ------------------------------------------------------------

meta_index = meta.set_index("3dmodel_id")

records = []

for item in products:

    item_id = str(item.get("item_id", ""))

    model_id = item.get("3dmodel_id")

    if not model_id:
        continue

    model_id = str(model_id)

    if model_id not in meta_index.index:
        continue

    m = meta_index.loc[model_id]

    record = dict(item)

    record.update({
        "3dmodel_id": model_id,
        "model_path": m["path"],
        "meshes": int(m["meshes"]),
        "materials": int(m["materials"]),
        "textures": int(m["textures"]),
        "images": int(m["images"]),
        "vertices": int(m["vertices"]),
        "faces": int(m["faces"]),
        "image_height_max": m["image_height_max"],
        "image_width_max": m["image_width_max"],
        "extent_x": float(m["extent_x"]),
        "extent_y": float(m["extent_y"]),
        "extent_z": float(m["extent_z"]),
    })

    records.append(record)

print(f"Successfully joined: {len(records):,}")

if not records:
    raise RuntimeError("No products could be joined with 3D metadata.")

df = pd.DataFrame(records)

# ------------------------------------------------------------
# Quality scoring
# ------------------------------------------------------------

def quality_score(row):

    score = 0.0

    # Textured models
    if row["textures"] >= 3:
        score += 25

    elif row["textures"] >= 1:
        score += 10

    # Materials
    if row["materials"] >= 1:
        score += 15

    # Multiple reference images
    if row["images"] >= 3:
        score += 15

    elif row["images"] >= 1:
        score += 5

    # Reasonable image resolution
    resolution = min(
        row["image_height_max"],
        row["image_width_max"]
    )

    if resolution >= 4096:
        score += 20
    elif resolution >= 2048:
        score += 15
    elif resolution >= 1024:
        score += 8

    # Geometry
    vertices = row["vertices"]
    faces = row["faces"]

    if 1_000 <= vertices <= 250_000:
        score += 15
    elif vertices > 250_000:
        score += 5

    if 2_000 <= faces <= 500_000:
        score += 10

    # Single mesh is preferred
    if row["meshes"] == 1:
        score += 5

    return score


df["quality_score"] = df.apply(quality_score, axis=1)

# ------------------------------------------------------------
# Remove obviously unusable models
# ------------------------------------------------------------

before = len(df)

df = df[
    (df["vertices"] >= 100) &
    (df["faces"] >= 100) &
    (df["textures"] >= 1) &
    (df["materials"] >= 1) &
    (df["images"] >= 1)
].copy()

print(f"After basic quality filter: {len(df):,}")
print(f"Removed: {before - len(df):,}")

# ------------------------------------------------------------
# Sort by quality
# ------------------------------------------------------------

df = df.sort_values(
    "quality_score",
    ascending=False
)

# ------------------------------------------------------------
# Select 250
#
# Prefer category diversity if category exists.
# ------------------------------------------------------------

if "category" in df.columns:

    selected = (
        df.groupby("category", group_keys=False)
          .apply(
              lambda x: x.head(
                  max(1, TARGET // max(1, df["category"].nunique()))
              )
          )
          .reset_index(drop=True)
    )

    # Fill remaining slots from global ranking
    if len(selected) < TARGET:

        selected_ids = set(selected["3dmodel_id"])

        remaining = df[
            ~df["3dmodel_id"].isin(selected_ids)
        ]

        selected = pd.concat(
            [
                selected,
                remaining.head(TARGET - len(selected))
            ],
            ignore_index=True
        )

else:
    selected = df.head(TARGET).copy()

selected = selected.drop_duplicates(
    subset=["3dmodel_id"]
).head(TARGET)

# ------------------------------------------------------------
# Save JSONL
# ------------------------------------------------------------

with open(OUT_FILE, "w", encoding="utf-8") as f:

    for record in selected.to_dict("records"):
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print()
print("=" * 60)
print("3D QUALITY POOL CREATED")
print("=" * 60)

print(f"Available 3D products: {len(products):,}")
print(f"Joined with metadata:  {len(df):,}")
print(f"Selected:              {len(selected):,}")
print()
print(f"Saved:")
print(OUT_FILE)

print()
print("Quality score range:")
print(
    f"{selected['quality_score'].min():.1f}"
    f" → "
    f"{selected['quality_score'].max():.1f}"
)

if "category" in selected.columns:

    print()
    print("Category distribution:")

    for category, count in (
        selected["category"]
        .value_counts()
        .sort_index()
        .items()
    ):
        print(f"  {category:<25} {count}")

BUILDING 3D QUALITY-RANKED POOL
Products loaded: 7,953
3D metadata loaded: 7,953
Successfully joined: 7,953
After basic quality filter: 7,908
Removed: 45

3D QUALITY POOL CREATED
Available 3D products: 7,953
Joined with metadata:  7,908
Selected:              250

Saved:
/content/neural_object_reconstruction/abo_candidates/3d_quality_pool.jsonl

Quality score range:
105.0 → 105.0


In [ ]:
from pathlib import Path
import os
import json
import gzip

# ============================================================
# ABO NEURAL OBJECT RECONSTRUCTION
# ROBUST DATASET PATH DISCOVERY
# ============================================================

PROJECT_ROOT = Path("/content/neural_object_reconstruction")

print("=" * 60)
print("ABO DATASET PATH DISCOVERY")
print("=" * 60)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project directory does not exist:\n{PROJECT_ROOT}"
    )

print(f"Project root found:")
print(f"  {PROJECT_ROOT}")

# ------------------------------------------------------------
# 1. FIND LISTING FILES
# ------------------------------------------------------------

print("\nSearching for ABO listing files...")

LISTING_FILES = sorted(
    PROJECT_ROOT.rglob("listings_*.json.gz")
)

if not LISTING_FILES:
    raise FileNotFoundError(
        f"""
No ABO listing files were found anywhere under:

{PROJECT_ROOT}

Expected files such as:
  listings_0.json.gz
  listings_1.json.gz
  ...
  listings_f.json.gz

Check whether the ABO listings were deleted or whether
the dataset is mounted somewhere else.
"""
    )

print(f"✓ Found {len(LISTING_FILES)} listing files")

for path in LISTING_FILES:
    print(f"  {path}")

# ------------------------------------------------------------
# 2. FIND 3D METADATA
# ------------------------------------------------------------

print("\nSearching for 3D metadata...")

THREED_METADATA_CANDIDATES = list(
    PROJECT_ROOT.rglob("3dmodels.csv.gz")
)

if not THREED_METADATA_CANDIDATES:
    raise FileNotFoundError(
        f"""
Could not find 3dmodels.csv.gz under:

{PROJECT_ROOT}
"""
    )

THREED_METADATA = THREED_METADATA_CANDIDATES[0]

print(f"✓ 3D metadata found:")
print(f"  {THREED_METADATA}")

# ------------------------------------------------------------
# 3. FIND 3D PRODUCT POOL
# ------------------------------------------------------------

print("\nSearching for 3D product pool...")

PRODUCT_POOL_CANDIDATES = list(
    PROJECT_ROOT.rglob("3d_product_pool.jsonl")
)

if not PRODUCT_POOL_CANDIDATES:
    raise FileNotFoundError(
        "Could not find 3d_product_pool.jsonl"
    )

PRODUCT_POOL = PRODUCT_POOL_CANDIDATES[0]

print(f"✓ Product pool found:")
print(f"  {PRODUCT_POOL}")

# ------------------------------------------------------------
# 4. FIND QUALITY POOL
# ------------------------------------------------------------

print("\nSearching for quality-ranked pool...")

QUALITY_POOL_CANDIDATES = list(
    PROJECT_ROOT.rglob("3d_quality_pool.jsonl")
)

if not QUALITY_POOL_CANDIDATES:
    raise FileNotFoundError(
        "Could not find 3d_quality_pool.jsonl"
    )

QUALITY_POOL = QUALITY_POOL_CANDIDATES[0]

print(f"✓ Quality pool found:")
print(f"  {QUALITY_POOL}")

# ------------------------------------------------------------
# 5. FIND ORIGINAL CANDIDATE POOL
# ------------------------------------------------------------

print("\nSearching for original candidate pool...")

CANDIDATE_POOL_CANDIDATES = list(
    PROJECT_ROOT.rglob("candidate_pool.jsonl")
)

if CANDIDATE_POOL_CANDIDATES:
    CANDIDATE_POOL = CANDIDATE_POOL_CANDIDATES[0]

    print(f"✓ Candidate pool found:")
    print(f"  {CANDIDATE_POOL}")
else:
    CANDIDATE_POOL = None
    print("  Candidate pool not found")
    print("  This is not required for the 3D pipeline.")

# ------------------------------------------------------------
# 6. DIRECTORY SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DISCOVERY COMPLETE")
print("=" * 60)

print(f"Listing files:       {len(LISTING_FILES)}")
print(f"3D metadata:         {THREED_METADATA}")
print(f"3D product pool:     {PRODUCT_POOL}")
print(f"3D quality pool:     {QUALITY_POOL}")

if CANDIDATE_POOL:
    print(f"Candidate pool:      {CANDIDATE_POOL}")

# ------------------------------------------------------------
# 7. VERIFY FILES
# ------------------------------------------------------------

print("\nVerifying files...")

for path in LISTING_FILES:
    if not path.exists():
        raise FileNotFoundError(path)

if not THREED_METADATA.exists():
    raise FileNotFoundError(THREED_METADATA)

if not PRODUCT_POOL.exists():
    raise FileNotFoundError(PRODUCT_POOL)

if not QUALITY_POOL.exists():
    raise FileNotFoundError(QUALITY_POOL)

print("✓ All required files exist")

# ------------------------------------------------------------
# 8. QUICK FILE SIZE CHECK
# ------------------------------------------------------------

print("\nFile sizes:")

total_listing_size = 0

for path in LISTING_FILES:
    size_mb = path.stat().st_size / (1024 * 1024)
    total_listing_size += size_mb

print(
    f"  ABO listings:      {total_listing_size:.2f} MB"
)

print(
    f"  3D metadata:       "
    f"{THREED_METADATA.stat().st_size / (1024 * 1024):.2f} MB"
)

print(
    f"  Product pool:      "
    f"{PRODUCT_POOL.stat().st_size / (1024 * 1024):.2f} MB"
)

print(
    f"  Quality pool:      "
    f"{QUALITY_POOL.stat().st_size / (1024 * 1024):.2f} MB"
)

print("\n✓ READY FOR NEXT STAGE")

ABO DATASET PATH DISCOVERY
Project root found:
  /content/neural_object_reconstruction

Searching for ABO listing files...
✓ Found 16 listing files
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_0.json.gz
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_1.json.gz
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_2.json.gz
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_3.json.gz
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_4.json.gz
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_5.json.gz
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_6.json.gz
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_7.json.gz
  /content/neural_object_reconstruction/abo_metadata/listings/metadata/listings_8.json.gz
  /content/neural_object_reconstruction/ab

In [ ]:
# ============================================================
# ABO 3D DATASET — BUILD DOWNLOAD BATCH
# ============================================================

import os
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/neural_object_reconstruction")

QUALITY_POOL = (
    PROJECT_ROOT
    / "abo_candidates"
    / "3d_quality_pool.jsonl"
)

BATCH_ROOT = (
    PROJECT_ROOT
    / "abo_batches"
)

BATCH_SIZE = 25
BATCH_ID = 1

BATCH_DIR = BATCH_ROOT / f"batch_{BATCH_ID:03d}"

BATCH_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("ABO 3D BATCH BUILDER")
print("=" * 60)

print(f"Quality pool: {QUALITY_POOL}")
print(f"Batch size:   {BATCH_SIZE}")
print(f"Batch ID:     {BATCH_ID:03d}")
print(f"Output:       {BATCH_DIR}")

# ------------------------------------------------------------
# LOAD QUALITY POOL
# ------------------------------------------------------------

if not QUALITY_POOL.exists():
    raise FileNotFoundError(
        f"Quality pool not found:\n{QUALITY_POOL}"
    )

records = []

with open(QUALITY_POOL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        records.append(json.loads(line))

print(f"\nRecords in quality pool: {len(records)}")

if len(records) < BATCH_ID * BATCH_SIZE:
    raise RuntimeError(
        "Not enough records remaining for this batch."
    )

# ------------------------------------------------------------
# SELECT BATCH
# ------------------------------------------------------------

start = (BATCH_ID - 1) * BATCH_SIZE
end = start + BATCH_SIZE

batch_records = records[start:end]

print(f"Selected records: {len(batch_records)}")

# ------------------------------------------------------------
# CREATE MANIFEST
# ------------------------------------------------------------

manifest = []

for index, record in enumerate(batch_records):

    item_id = (
        record.get("item_id")
        or record.get("3dmodel_id")
        or record.get("asin")
    )

    model_id = (
        record.get("3dmodel_id")
        or item_id
    )

    model_path = record.get("path")

    if not model_path:
        print(
            f"WARNING: no model path for {item_id}"
        )
        continue

    model_url = (
        "https://amazon-berkeley-objects.s3.amazonaws.com/"
        "3dmodels/original/"
        + model_path
    )

    category = record.get("category", "unknown")

    entry = {
        "batch_id": BATCH_ID,
        "batch_index": index,
        "item_id": item_id,
        "3dmodel_id": model_id,
        "category": category,
        "path": model_path,
        "model_url": model_url,
        "quality_score": record.get("score"),
    }

    manifest.append(entry)

# ------------------------------------------------------------
# SAVE MANIFEST
# ------------------------------------------------------------

manifest_path = BATCH_DIR / "manifest.jsonl"

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    for entry in manifest:
        f.write(
            json.dumps(
                entry,
                ensure_ascii=False
            ) + "\n"
        )

# Also save a readable CSV

csv_path = BATCH_DIR / "manifest.csv"

pd.DataFrame(manifest).to_csv(
    csv_path,
    index=False
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BATCH CREATED")
print("=" * 60)

print(f"Batch:             {BATCH_ID:03d}")
print(f"Models:            {len(manifest)}")
print(f"Manifest:          {manifest_path}")
print(f"CSV:               {csv_path}")

print("\nCategory distribution:")

category_counts = {}

for item in manifest:
    category = item["category"]
    category_counts[category] = (
        category_counts.get(category, 0) + 1
    )

for category, count in sorted(
    category_counts.items()
):
    print(f"  {category:<25} {count}")

print("\nFirst models:")

for item in manifest[:5]:
    print(
        f"  {item['item_id']}  "
        f"{item['category']}  "
        f"{item['path']}"
    )

print("\n✓ Batch ready for download")

ABO 3D BATCH BUILDER
Quality pool: /content/neural_object_reconstruction/abo_candidates/3d_quality_pool.jsonl
Batch size:   25
Batch ID:     001
Output:       /content/neural_object_reconstruction/abo_batches/batch_001

Records in quality pool: 250
Selected records: 25

BATCH CREATED
Batch:             001
Models:            0
Manifest:          /content/neural_object_reconstruction/abo_batches/batch_001/manifest.jsonl
CSV:               /content/neural_object_reconstruction/abo_batches/batch_001/manifest.csv

Category distribution:

First models:

✓ Batch ready for download


In [ ]:
# ============================================================
# ABO 3D BATCH BUILDER — FIXED
# Reconstructs missing GLB paths from 3D metadata
# ============================================================

import os
import json
import gzip
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/neural_object_reconstruction")

QUALITY_POOL = (
    PROJECT_ROOT
    / "abo_candidates"
    / "3d_quality_pool.jsonl"
)

METADATA_FILE = (
    PROJECT_ROOT
    / "abo_metadata"
    / "3dmodels"
    / "metadata"
    / "3dmodels.csv.gz"
)

BATCH_ROOT = (
    PROJECT_ROOT
    / "abo_batches"
)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

BATCH_SIZE = 25
BATCH_ID = 1

BATCH_DIR = (
    BATCH_ROOT
    / f"batch_{BATCH_ID:03d}"
)

BATCH_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# HEADER
# ------------------------------------------------------------

print("=" * 60)
print("ABO 3D BATCH BUILDER — FIXED")
print("=" * 60)

print(f"Quality pool:")
print(f"  {QUALITY_POOL}")

print(f"\n3D metadata:")
print(f"  {METADATA_FILE}")

print(f"\nBatch:")
print(f"  {BATCH_ID:03d}")

print(f"Batch size:")
print(f"  {BATCH_SIZE}")

# ------------------------------------------------------------
# VERIFY FILES
# ------------------------------------------------------------

if not QUALITY_POOL.exists():
    raise FileNotFoundError(
        f"Quality pool not found:\n{QUALITY_POOL}"
    )

if not METADATA_FILE.exists():
    raise FileNotFoundError(
        f"3D metadata not found:\n{METADATA_FILE}"
    )

# ------------------------------------------------------------
# LOAD QUALITY POOL
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("LOADING QUALITY POOL")
print("-" * 60)

quality_records = []

with open(
    QUALITY_POOL,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        quality_records.append(
            json.loads(line)
        )

print(
    f"Quality pool records: "
    f"{len(quality_records)}"
)

if not quality_records:
    raise RuntimeError(
        "Quality pool is empty."
    )

# ------------------------------------------------------------
# INSPECT FIRST RECORD
# ------------------------------------------------------------

print("\nQuality pool fields:")

if quality_records:

    for key in quality_records[0].keys():
        print(f"  - {key}")

# ------------------------------------------------------------
# LOAD 3D METADATA
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("LOADING 3D METADATA")
print("-" * 60)

metadata = pd.read_csv(
    METADATA_FILE,
    compression="gzip"
)

print(
    f"3D metadata records: "
    f"{len(metadata):,}"
)

print(
    f"3D metadata columns: "
    f"{len(metadata.columns)}"
)

required_columns = {
    "3dmodel_id",
    "path"
}

missing = (
    required_columns
    - set(metadata.columns)
)

if missing:

    raise RuntimeError(
        "Missing required metadata columns: "
        + ", ".join(sorted(missing))
    )

# ------------------------------------------------------------
# BUILD FAST LOOKUP
# ------------------------------------------------------------

print("\nBuilding 3D model lookup...")

metadata_lookup = {}

for _, row in metadata.iterrows():

    model_id = str(
        row["3dmodel_id"]
    ).strip()

    model_path = str(
        row["path"]
    ).strip()

    metadata_lookup[model_id] = {
        "path": model_path,
        "meshes": int(row["meshes"]),
        "materials": int(row["materials"]),
        "textures": int(row["textures"]),
        "images": int(row["images"]),
        "vertices": int(row["vertices"]),
        "faces": int(row["faces"]),
        "extent_x": float(row["extent_x"]),
        "extent_y": float(row["extent_y"]),
        "extent_z": float(row["extent_z"]),
    }

print(
    f"Lookup entries: "
    f"{len(metadata_lookup):,}"
)

# ------------------------------------------------------------
# SELECT BATCH
# ------------------------------------------------------------

start = (
    BATCH_ID - 1
) * BATCH_SIZE

end = start + BATCH_SIZE

batch_records = quality_records[
    start:end
]

print("\n" + "-" * 60)
print("SELECTING BATCH")
print("-" * 60)

print(
    f"Pool range: "
    f"{start} → {end - 1}"
)

print(
    f"Records selected: "
    f"{len(batch_records)}"
)

# ------------------------------------------------------------
# BUILD ENRICHED MANIFEST
# ------------------------------------------------------------

manifest = []

missing_models = []

for index, record in enumerate(
    batch_records
):

    # --------------------------------------------------------
    # Find model ID
    # --------------------------------------------------------

    item_id = (
        record.get("item_id")
        or record.get("asin")
        or record.get("3dmodel_id")
    )

    model_id = (
        record.get("3dmodel_id")
        or item_id
    )

    if model_id is None:

        missing_models.append({
            "index": index,
            "reason": "No model ID",
            "record": record
        })

        continue

    model_id = str(
        model_id
    ).strip()

    # --------------------------------------------------------
    # Look up actual 3D metadata
    # --------------------------------------------------------

    model_info = metadata_lookup.get(
        model_id
    )

    if model_info is None:

        missing_models.append({
            "index": index,
            "model_id": model_id,
            "reason": "3D model ID not found"
        })

        continue

    model_path = model_info["path"]

    if (
        not model_path
        or model_path.lower() == "nan"
    ):

        missing_models.append({
            "index": index,
            "model_id": model_id,
            "reason": "3D model path missing"
        })

        continue

    # --------------------------------------------------------
    # Category
    # --------------------------------------------------------

    category = (
        record.get("category")
        or "unknown"
    )

    # --------------------------------------------------------
    # Quality score
    # --------------------------------------------------------

    quality_score = record.get(
        "score"
    )

    # --------------------------------------------------------
    # Construct official ABO URL
    # --------------------------------------------------------

    model_url = (
        "https://amazon-berkeley-objects.s3.amazonaws.com/"
        "3dmodels/original/"
        + model_path
    )

    # --------------------------------------------------------
    # Build record
    # --------------------------------------------------------

    enriched = {

        # Dataset identity
        "item_id": item_id,
        "3dmodel_id": model_id,

        # Dataset category
        "category": category,

        # Original ABO path
        "path": model_path,

        # Download URL
        "model_url": model_url,

        # Quality ranking
        "quality_score": quality_score,

        # 3D quality metadata
        "meshes": model_info["meshes"],
        "materials": model_info["materials"],
        "textures": model_info["textures"],
        "images": model_info["images"],

        "vertices": model_info["vertices"],
        "faces": model_info["faces"],

        "extent_x": model_info["extent_x"],
        "extent_y": model_info["extent_y"],
        "extent_z": model_info["extent_z"],

        # Batch information
        "batch_id": BATCH_ID,
        "batch_index": index,
    }

    manifest.append(
        enriched
    )

# ------------------------------------------------------------
# REPORT MATCHING
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("3D MODEL PATH RECONSTRUCTION")
print("-" * 60)

print(
    f"Selected:        {len(batch_records)}"
)

print(
    f"Matched:         {len(manifest)}"
)

print(
    f"Missing:         {len(missing_models)}"
)

if missing_models:

    print("\nMissing models:")

    for item in missing_models:

        print(
            f"  {item.get('model_id', 'UNKNOWN')} "
            f"→ {item['reason']}"
        )

# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

if len(manifest) == 0:

    raise RuntimeError(
        "No 3D models were successfully matched. "
        "Do NOT proceed to downloading."
    )

# ------------------------------------------------------------
# SAVE MANIFEST
# ------------------------------------------------------------

manifest_path = (
    BATCH_DIR
    / "manifest.jsonl"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    for record in manifest:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

# ------------------------------------------------------------
# SAVE CSV
# ------------------------------------------------------------

csv_path = (
    BATCH_DIR
    / "manifest.csv"
)

pd.DataFrame(
    manifest
).to_csv(
    csv_path,
    index=False
)

# ------------------------------------------------------------
# SAVE MISSING REPORT
# ------------------------------------------------------------

missing_path = (
    BATCH_DIR
    / "missing_models.json"
)

with open(
    missing_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        missing_models,
        f,
        indent=2
    )

# ------------------------------------------------------------
# CATEGORY DISTRIBUTION
# ------------------------------------------------------------

category_counts = {}

for record in manifest:

    category = record["category"]

    category_counts[category] = (
        category_counts.get(
            category,
            0
        ) + 1
    )

# ------------------------------------------------------------
# FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BATCH CREATED SUCCESSFULLY")
print("=" * 60)

print(
    f"Batch:       {BATCH_ID:03d}"
)

print(
    f"Models:      {len(manifest)}"
)

print(
    f"Manifest:    {manifest_path}"
)

print(
    f"CSV:         {csv_path}"
)

print(
    f"Missing:     {missing_path}"
)

print("\nCategory distribution:")

for category, count in sorted(
    category_counts.items()
):

    print(
        f"  {category:<25} {count}"
    )

print("\nFirst 5 models:")

for record in manifest[:5]:

    print(
        f"  {record['item_id']} | "
        f"{record['category']} | "
        f"{record['path']}"
    )

print("\n" + "=" * 60)
print("READY FOR DOWNLOAD")
print("=" * 60)

ABO 3D BATCH BUILDER — FIXED
Quality pool:
  /content/neural_object_reconstruction/abo_candidates/3d_quality_pool.jsonl

3D metadata:
  /content/neural_object_reconstruction/abo_metadata/3dmodels/metadata/3dmodels.csv.gz

Batch:
  001
Batch size:
  25

------------------------------------------------------------
LOADING QUALITY POOL
------------------------------------------------------------
Quality pool records: 250

Quality pool fields:
  - item_id
  - 3dmodel_id
  - record
  - model_path
  - meshes
  - materials
  - textures
  - images
  - vertices
  - faces
  - image_height_max
  - image_width_max
  - extent_x
  - extent_y
  - extent_z
  - quality_score

------------------------------------------------------------
LOADING 3D METADATA
------------------------------------------------------------
3D metadata records: 7,953
3D metadata columns: 15

Building 3D model lookup...
Lookup entries: 7,953

------------------------------------------------------------
SELECTING BATCH
----------

In [ ]:
import json

QUALITY_POOL = "/content/neural_object_reconstruction/abo_candidates/3d_quality_pool.jsonl"

with open(QUALITY_POOL, "r", encoding="utf-8") as f:
    first = json.loads(f.readline())

print("Top-level fields:")
print(list(first.keys()))

print("\nRecord type:")
print(type(first.get("record")))

print("\nRecord contents:")
print(json.dumps(first.get("record"), indent=2, ensure_ascii=False)[:5000])

Top-level fields:
['item_id', '3dmodel_id', 'record', 'model_path', 'meshes', 'materials', 'textures', 'images', 'vertices', 'faces', 'image_height_max', 'image_width_max', 'extent_x', 'extent_y', 'extent_z', 'quality_score']

Record type:
<class 'dict'>

Record contents:
{
  "item_dimensions": {
    "height": {
      "normalized_value": {
        "unit": "inches",
        "value": 30.25
      },
      "unit": "inches",
      "value": 30.25
    },
    "length": {
      "normalized_value": {
        "unit": "inches",
        "value": 35.25
      },
      "unit": "inches",
      "value": 35.25
    },
    "width": {
      "normalized_value": {
        "unit": "inches",
        "value": 87
      },
      "unit": "inches",
      "value": 87
    }
  },
  "brand": [
    {
      "language_tag": "en_US",
      "value": "Rivet"
    }
  ],
  "bullet_point": [
    {
      "language_tag": "en_US",
      "value": "Bring stunning, Mid-century elegance to your living room with this hand-tufted sofa fe

In [ ]:
# ============================================================
# ABO 3D BATCH BUILDER — CATEGORY-AWARE + RESUMABLE
# ============================================================

import os
import json
import csv
import re
import shutil
from collections import Counter, defaultdict

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

ROOT = "/content/neural_object_reconstruction"

QUALITY_POOL = os.path.join(
    ROOT,
    "abo_candidates",
    "3d_quality_pool.jsonl"
)

BATCH_ROOT = os.path.join(
    ROOT,
    "abo_batches"
)

BATCH_SIZE = 25
TOTAL_MODELS = 250

os.makedirs(BATCH_ROOT, exist_ok=True)

print("=" * 60)
print("ABO 3D BATCH BUILDER — CATEGORY-AWARE")
print("=" * 60)

print(f"Quality pool:")
print(f"  {QUALITY_POOL}")

print(f"Batch size: {BATCH_SIZE}")
print(f"Total target: {TOTAL_MODELS}")


# ============================================================
# CATEGORY NORMALIZATION
# ============================================================

CATEGORY_MAP = {
    # Kitchen
    "kitchen": "kitchen",
    "cookware": "kitchen",
    "bakeware": "kitchen",
    "drinkware": "kitchen",
    "tableware": "kitchen",
    "dinnerware": "kitchen",
    "cutlery": "kitchen",
    "utensil": "kitchen",
    "utensils": "kitchen",
    "mug": "kitchen",
    "cups": "kitchen",
    "cup": "kitchen",
    "plate": "kitchen",
    "bowl": "kitchen",
    "pot": "kitchen",
    "pan": "kitchen",
    "kettle": "kitchen",
    "bottle": "kitchen",
    "container": "containers",
    "containers": "containers",
    "storage": "containers",

    # Electronics
    "electronics": "electronics",
    "electronic": "electronics",
    "headphones": "electronics",
    "headphone": "electronics",
    "earbuds": "electronics",
    "speaker": "electronics",
    "speakers": "electronics",
    "camera": "electronics",
    "cameras": "electronics",
    "computer": "electronics",
    "computers": "electronics",
    "laptop": "electronics",
    "tablet": "electronics",
    "monitor": "electronics",
    "television": "electronics",
    "tv": "electronics",
    "phone": "electronics",
    "smartphone": "electronics",
    "keyboard": "electronics",
    "mouse": "electronics",
    "router": "electronics",
    "charger": "electronics",
    "projector": "electronics",

    # Fashion
    "fashion": "fashion_accessories",
    "accessories": "fashion_accessories",
    "jewelry": "fashion_accessories",
    "jewellery": "fashion_accessories",
    "watch": "fashion_accessories",
    "watches": "fashion_accessories",
    "bag": "fashion_accessories",
    "bags": "fashion_accessories",
    "handbag": "fashion_accessories",
    "wallet": "fashion_accessories",
    "belt": "fashion_accessories",
    "hat": "fashion_accessories",
    "hats": "fashion_accessories",
    "sunglasses": "fashion_accessories",
    "scarf": "fashion_accessories",

    # Sports
    "sports": "sports",
    "sport": "sports",
    "fitness": "sports",
    "exercise": "sports",
    "gym": "sports",
    "football": "sports",
    "soccer": "sports",
    "basketball": "sports",
    "baseball": "sports",
    "tennis": "sports",
    "golf": "sports",
    "yoga": "sports",
    "dumbbell": "sports",
    "dumbbells": "sports",
    "bicycle": "sports",
    "bike": "sports",
    "skateboard": "sports",

    # Tools
    "tools": "tools",
    "tool": "tools",
    "hardware": "tools",
    "drill": "tools",
    "screwdriver": "tools",
    "wrench": "tools",
    "hammer": "tools",
    "saw": "tools",
    "pliers": "tools",
    "toolbox": "tools",

    # Mechanical
    "mechanical": "mechanical",
    "automotive": "mechanical",
    "automotive parts": "mechanical",
    "car": "mechanical",
    "vehicle": "mechanical",
    "motor": "mechanical",
    "engine": "mechanical",
    "machine": "mechanical",
    "bearing": "mechanical",
    "gear": "mechanical",

    # Musical
    "musical instruments": "musical_instruments",
    "musical instrument": "musical_instruments",
    "instrument": "musical_instruments",
    "guitar": "musical_instruments",
    "piano": "musical_instruments",
    "keyboard instrument": "musical_instruments",
    "violin": "musical_instruments",
    "drum": "musical_instruments",
    "drums": "musical_instruments",
    "flute": "musical_instruments",
    "trumpet": "musical_instruments",
    "saxophone": "musical_instruments",

    # Toys
    "toys": "toys",
    "toy": "toys",
    "games": "toys",
    "game": "toys",
    "doll": "toys",
    "dolls": "toys",
    "puzzle": "toys",
    "puzzles": "toys",
    "action figure": "toys",
    "building blocks": "toys",

    # Decorative
    "decorative": "decorative",
    "decoration": "decorative",
    "decor": "decorative",
    "ornament": "decorative",
    "ornaments": "decorative",
    "vase": "decorative",
    "wall art": "decorative",
    "mirror": "decorative",
    "candle": "decorative",
    "candles": "decorative",

    # Furniture
    "furniture": "furniture",
    "sofa": "furniture",
    "sofas": "furniture",
    "couch": "furniture",
    "sectional": "furniture",
    "chair": "furniture",
    "chairs": "furniture",
    "table": "furniture",
    "tables": "furniture",
    "desk": "furniture",
    "desks": "furniture",
    "bed": "furniture",
    "beds": "furniture",
    "cabinet": "furniture",
    "shelf": "furniture",
    "shelves": "furniture",
}


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):
    if value is None:
        return ""

    if isinstance(value, str):
        return value.strip().lower()

    return str(value).strip().lower()


def extract_values(value):
    """
    Handles ABO fields such as:

        [
            {"language_tag": "en_US", "value": "SOFA"}
        ]

    and plain strings.
    """

    values = []

    if value is None:
        return values

    if isinstance(value, str):
        return [value]

    if isinstance(value, list):

        for item in value:

            if isinstance(item, str):
                values.append(item)

            elif isinstance(item, dict):

                if "value" in item:
                    values.append(str(item["value"]))

                if "node_name" in item:
                    values.append(str(item["node_name"]))

    elif isinstance(value, dict):

        if "value" in value:
            values.append(str(value["value"]))

        if "node_name" in value:
            values.append(str(value["node_name"]))

    return values


def normalize_category(text):
    """
    Convert an arbitrary ABO product/category description
    into one of our project categories.
    """

    text = clean_text(text)

    if not text:
        return None

    # Direct match
    if text in CATEGORY_MAP:
        return CATEGORY_MAP[text]

    # Exact phrase matching
    for keyword, category in CATEGORY_MAP.items():

        keyword = keyword.lower()

        if keyword in text:
            return category

    return None


# ============================================================
# CATEGORY EXTRACTION
# ============================================================

def determine_category(record):
    """
    Determine category using multiple ABO fields.

    Priority:

    1. product_type
    2. node
    3. item_keywords
    4. item_name
    5. brand-independent text fallback
    """

    if not isinstance(record, dict):
        return "unknown"

    # --------------------------------------------------------
    # 1. PRODUCT TYPE
    # --------------------------------------------------------

    product_types = extract_values(
        record.get("product_type")
    )

    for value in product_types:

        category = normalize_category(value)

        if category:
            return category

    # --------------------------------------------------------
    # 2. NODE
    # --------------------------------------------------------

    nodes = extract_values(
        record.get("node")
    )

    for node in nodes:

        category = normalize_category(node)

        if category:
            return category

    # --------------------------------------------------------
    # 3. KEYWORDS
    # --------------------------------------------------------

    keywords = extract_values(
        record.get("item_keywords")
    )

    # Check each keyword individually first
    for keyword in keywords:

        category = normalize_category(keyword)

        if category:
            return category

    # --------------------------------------------------------
    # 4. ITEM NAME
    # --------------------------------------------------------

    names = extract_values(
        record.get("item_name")
    )

    for name in names:

        category = normalize_category(name)

        if category:
            return category

    # --------------------------------------------------------
    # 5. COMBINED TEXT FALLBACK
    # --------------------------------------------------------

    combined = " ".join(
        product_types +
        nodes +
        keywords +
        names
    )

    category = normalize_category(combined)

    if category:
        return category

    return "unknown"


# ============================================================
# LOAD QUALITY POOL
# ============================================================

print()
print("-" * 60)
print("LOADING QUALITY POOL")
print("-" * 60)

if not os.path.exists(QUALITY_POOL):
    raise FileNotFoundError(
        f"Quality pool not found:\n{QUALITY_POOL}"
    )

records = []

with open(
    QUALITY_POOL,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        records.append(
            json.loads(line)
        )

print(f"Quality pool records: {len(records):,}")


# ============================================================
# DERIVE CATEGORIES
# ============================================================

print()
print("-" * 60)
print("DERIVING PRODUCT CATEGORIES")
print("-" * 60)

category_counts = Counter()

for record in records:

    nested_record = record.get("record", {})

    category = determine_category(
        nested_record
    )

    record["category"] = category

    category_counts[category] += 1


print()
print("Category distribution:")

for category, count in sorted(
    category_counts.items(),
    key=lambda x: (-x[1], x[0])
):

    print(
        f"  {category:<25} {count:>5}"
    )


# ============================================================
# REMOVE DUPLICATE PRODUCTS
# ============================================================

print()
print("-" * 60)
print("DEDUPLICATING")
print("-" * 60)

unique = {}
duplicates = 0

for record in records:

    item_id = record.get("item_id")

    if not item_id:
        continue

    if item_id in unique:
        duplicates += 1
        continue

    unique[item_id] = record

records = list(unique.values())

print(f"Unique products: {len(records):,}")
print(f"Duplicates removed: {duplicates:,}")


# ============================================================
# SAVE CATEGORY-ENRICHED QUALITY POOL
# ============================================================

ENRICHED_POOL = os.path.join(
    ROOT,
    "abo_candidates",
    "3d_quality_pool_categorized.jsonl"
)

print()
print("Saving categorized quality pool...")

with open(
    ENRICHED_POOL,
    "w",
    encoding="utf-8"
) as f:

    for record in records:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print(f"Saved:")
print(f"  {ENRICHED_POOL}")


# ============================================================
# SELECT BATCH 001
# ============================================================

BATCH_ID = 1

batch_start = (BATCH_ID - 1) * BATCH_SIZE
batch_end = batch_start + BATCH_SIZE

selected = records[
    batch_start:batch_end
]

print()
print("-" * 60)
print("SELECTING BATCH")
print("-" * 60)

print(f"Batch: {BATCH_ID:03d}")
print(f"Range: {batch_start} → {batch_end - 1}")
print(f"Selected: {len(selected)}")


if len(selected) < BATCH_SIZE:

    raise RuntimeError(
        f"Only {len(selected)} records available "
        f"for batch {BATCH_ID:03d}."
    )


# ============================================================
# CREATE BATCH DIRECTORY
# ============================================================

batch_name = f"batch_{BATCH_ID:03d}"

batch_dir = os.path.join(
    BATCH_ROOT,
    batch_name
)

os.makedirs(
    batch_dir,
    exist_ok=True
)

manifest_jsonl = os.path.join(
    batch_dir,
    "manifest.jsonl"
)

manifest_csv = os.path.join(
    batch_dir,
    "manifest.csv"
)

missing_json = os.path.join(
    batch_dir,
    "missing_models.json"
)


# ============================================================
# BUILD MANIFEST
# ============================================================

manifest_records = []

for record in selected:

    item_id = record.get(
        "item_id"
    )

    model_id = record.get(
        "3dmodel_id"
    )

    model_path = record.get(
        "model_path"
    )

    category = record.get(
        "category",
        "unknown"
    )

    # --------------------------------------------------------
    # Fallback: model path from quality record
    # --------------------------------------------------------

    if not model_path:

        nested = record.get(
            "record",
            {}
        )

        model_path = nested.get(
            "model_path"
        )

    # --------------------------------------------------------
    # Create clean manifest record
    # --------------------------------------------------------

    manifest_record = {
        "item_id": item_id,
        "3dmodel_id": model_id,
        "category": category,
        "model_path": model_path,
        "meshes": record.get("meshes"),
        "materials": record.get("materials"),
        "textures": record.get("textures"),
        "images": record.get("images"),
        "vertices": record.get("vertices"),
        "faces": record.get("faces"),
        "image_height_max": record.get(
            "image_height_max"
        ),
        "image_width_max": record.get(
            "image_width_max"
        ),
        "extent_x": record.get("extent_x"),
        "extent_y": record.get("extent_y"),
        "extent_z": record.get("extent_z"),
        "quality_score": record.get(
            "quality_score"
        )
    }

    manifest_records.append(
        manifest_record
    )


# ============================================================
# CHECK MODEL PATHS
# ============================================================

print()
print("-" * 60)
print("CHECKING 3D MODEL PATHS")
print("-" * 60)

missing = []

for record in manifest_records:

    if not record["model_path"]:

        missing.append({
            "item_id": record["item_id"],
            "3dmodel_id": record["3dmodel_id"]
        })


print(
    f"Selected: {len(manifest_records)}"
)

print(
    f"Missing model paths: {len(missing)}"
)


# ============================================================
# WRITE MANIFEST JSONL
# ============================================================

with open(
    manifest_jsonl,
    "w",
    encoding="utf-8"
) as f:

    for record in manifest_records:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )


# ============================================================
# WRITE CSV
# ============================================================

csv_fields = [
    "item_id",
    "3dmodel_id",
    "category",
    "model_path",
    "meshes",
    "materials",
    "textures",
    "images",
    "vertices",
    "faces",
    "image_height_max",
    "image_width_max",
    "extent_x",
    "extent_y",
    "extent_z",
    "quality_score"
]

with open(
    manifest_csv,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=csv_fields
    )

    writer.writeheader()

    for record in manifest_records:

        writer.writerow(record)


# ============================================================
# WRITE MISSING MODELS
# ============================================================

with open(
    missing_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        missing,
        f,
        indent=2
    )


# ============================================================
# FINAL REPORT
# ============================================================

final_categories = Counter(
    record["category"]
    for record in manifest_records
)

print()
print("=" * 60)
print("BATCH CREATED SUCCESSFULLY")
print("=" * 60)

print(
    f"Batch:       {batch_name}"
)

print(
    f"Models:      {len(manifest_records)}"
)

print(
    f"Missing:     {len(missing)}"
)

print()
print("Category distribution:")

for category, count in sorted(
    final_categories.items()
):

    print(
        f"  {category:<25} {count:>3}"
    )

print()
print("First 5 models:")

for record in manifest_records[:5]:

    print(
        f"  {record['item_id']} | "
        f"{record['category']} | "
        f"{record['model_path']}"
    )

print()
print("Files:")

print(
    f"  Manifest: {manifest_jsonl}"
)

print(
    f"  CSV:      {manifest_csv}"
)

print(
    f"  Missing:  {missing_json}"
)

print()
print("=" * 60)
print("READY FOR 3D DOWNLOAD")
print("=" * 60)

ABO 3D BATCH BUILDER — CATEGORY-AWARE
Quality pool:
  /content/neural_object_reconstruction/abo_candidates/3d_quality_pool.jsonl
Batch size: 25
Total target: 250

------------------------------------------------------------
LOADING QUALITY POOL
------------------------------------------------------------
Quality pool records: 250

------------------------------------------------------------
DERIVING PRODUCT CATEGORIES
------------------------------------------------------------

Category distribution:
  furniture                   166
  decorative                   26
  tools                        15
  kitchen                      11
  containers                    9
  mechanical                    7
  fashion_accessories           6
  unknown                       5
  electronics                   4
  sports                        1

------------------------------------------------------------
DEDUPLICATING
------------------------------------------------------------
Unique products:

In [ ]:
# ============================================================
# ABO 3D BATCH DOWNLOADER — CORRECT S3 PATH
# ============================================================

import os
import json
import time
import requests
from pathlib import Path


# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/neural_object_reconstruction")

BATCH_ID = "001"

BATCH_DIR = PROJECT_ROOT / "abo_batches" / f"batch_{BATCH_ID}"
MANIFEST_PATH = BATCH_DIR / "manifest.jsonl"

MODELS_DIR = BATCH_DIR / "models"

# IMPORTANT:
# ABO metadata gives paths such as:
#     7/B075X4QMW7.glb
#
# Actual S3 objects are under:
#     3dmodels/original/7/B075X4QMW7.glb
#
BASE_URL = (
    "https://amazon-berkeley-objects.s3.amazonaws.com"
    "/3dmodels/original"
)


# ------------------------------------------------------------
# VALIDATE MANIFEST
# ------------------------------------------------------------

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest not found:\n{MANIFEST_PATH}"
    )


MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# LOAD MANIFEST
# ------------------------------------------------------------

records = []

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        records.append(json.loads(line))


print("=" * 60)
print("ABO 3D BATCH DOWNLOADER")
print("=" * 60)

print(f"Batch:   {BATCH_ID}")
print(f"Models:  {len(records)}")
print(f"Output:  {MODELS_DIR}")


# ------------------------------------------------------------
# DOWNLOAD FUNCTION
# ------------------------------------------------------------

def download_model(model_path, output_path, timeout=120):

    """
    Download one ABO GLB model.

    IMPORTANT:
    404 is NOT retried because it indicates an invalid
    object URL/path rather than a temporary network failure.
    """

    url = f"{BASE_URL}/{model_path}"

    # Already downloaded
    if output_path.exists():

        size_mb = output_path.stat().st_size / (
            1024 * 1024
        )

        if output_path.stat().st_size > 0:

            return {
                "status": "exists",
                "url": url,
                "size_mb": size_mb
            }


    temp_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )


    try:

        with requests.get(
            url,
            stream=True,
            timeout=timeout
        ) as response:

            # ------------------------------------------------
            # DO NOT RETRY 404
            # ------------------------------------------------

            if response.status_code == 404:

                return {
                    "status": "404",
                    "url": url,
                    "error": "Model not found at S3 URL"
                }


            response.raise_for_status()


            # ------------------------------------------------
            # DOWNLOAD
            # ------------------------------------------------

            total = response.headers.get(
                "content-length"
            )

            if total is not None:
                total = int(total)

            downloaded = 0

            with open(
                temp_path,
                "wb"
            ) as out:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if not chunk:
                        continue

                    out.write(chunk)

                    downloaded += len(chunk)


            # ------------------------------------------------
            # VALIDATE
            # ------------------------------------------------

            if downloaded == 0:

                if temp_path.exists():
                    temp_path.unlink()

                return {
                    "status": "empty",
                    "url": url,
                    "error": "Downloaded zero bytes"
                }


            # Atomic rename
            temp_path.replace(output_path)


            return {
                "status": "downloaded",
                "url": url,
                "size_mb": downloaded / (
                    1024 * 1024
                )
            }


    except requests.exceptions.Timeout as e:

        if temp_path.exists():
            temp_path.unlink()

        return {
            "status": "timeout",
            "url": url,
            "error": str(e)
        }


    except requests.exceptions.ConnectionError as e:

        if temp_path.exists():
            temp_path.unlink()

        return {
            "status": "connection_error",
            "url": url,
            "error": str(e)
        }


    except requests.exceptions.HTTPError as e:

        if temp_path.exists():
            temp_path.unlink()

        return {
            "status": "http_error",
            "url": url,
            "error": str(e)
        }


    except Exception as e:

        if temp_path.exists():
            temp_path.unlink()

        return {
            "status": "error",
            "url": url,
            "error": str(e)
        }


# ------------------------------------------------------------
# DOWNLOAD BATCH
# ------------------------------------------------------------

results = []

downloaded_count = 0
existing_count = 0
failed_count = 0
not_found_count = 0


print()
print("-" * 60)
print("DOWNLOADING MODELS")
print("-" * 60)


for index, record in enumerate(
    records,
    start=1
):

    item_id = record.get(
        "item_id",
        "UNKNOWN"
    )

    category = record.get(
        "category",
        "unknown"
    )

    model_path = record.get(
        "model_path"
    )

    model_id = record.get(
        "3dmodel_id",
        item_id
    )


    print(
        f"[{index:02d}/{len(records):02d}] "
        f"{item_id} | {category}"
    )


    # --------------------------------------------------------
    # Validate model path
    # --------------------------------------------------------

    if not model_path:

        print("       ✗ MISSING MODEL PATH")

        result = {
            "item_id": item_id,
            "3dmodel_id": model_id,
            "status": "missing_path"
        }

        results.append(result)

        failed_count += 1

        continue


    # --------------------------------------------------------
    # Output filename
    # --------------------------------------------------------

    output_path = MODELS_DIR / f"{item_id}.glb"


    # --------------------------------------------------------
    # Download
    # --------------------------------------------------------

    result = download_model(
        model_path=model_path,
        output_path=output_path
    )


    result["item_id"] = item_id
    result["3dmodel_id"] = model_id
    result["model_path"] = model_path
    result["category"] = category
    result["local_path"] = str(
        output_path
    )


    results.append(result)


    # --------------------------------------------------------
    # Status
    # --------------------------------------------------------

    if result["status"] == "downloaded":

        downloaded_count += 1

        print(
            f"       ✓ DOWNLOADED "
            f"({result['size_mb']:.2f} MB)"
        )


    elif result["status"] == "exists":

        existing_count += 1

        print(
            f"       ✓ ALREADY EXISTS "
            f"({result['size_mb']:.2f} MB)"
        )


    elif result["status"] == "404":

        not_found_count += 1

        print(
            "       ✗ 404 NOT FOUND"
        )

        print(
            f"       {result['url']}"
        )


    else:

        failed_count += 1

        print(
            f"       ✗ {result['status']}"
        )

        if "error" in result:
            print(
                f"       {result['error']}"
            )


# ------------------------------------------------------------
# SAVE DOWNLOAD MANIFEST
# ------------------------------------------------------------

DOWNLOAD_MANIFEST = (
    BATCH_DIR /
    "download_manifest.jsonl"
)

with open(
    DOWNLOAD_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    for result in results:

        f.write(
            json.dumps(
                result,
                ensure_ascii=False
            )
            + "\n"
        )


# ------------------------------------------------------------
# SAVE FAILED MODELS
# ------------------------------------------------------------

failed_results = [
    r for r in results
    if r["status"] not in (
        "downloaded",
        "exists"
    )
]


FAILED_PATH = (
    BATCH_DIR /
    "failed_downloads.json"
)

with open(
    FAILED_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        failed_results,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print()
print("=" * 60)
print("DOWNLOAD COMPLETE")
print("=" * 60)

print(
    f"Total models:       {len(records)}"
)

print(
    f"Downloaded:         {downloaded_count}"
)

print(
    f"Already existed:    {existing_count}"
)

print(
    f"404 not found:      {not_found_count}"
)

print(
    f"Other failures:     {failed_count - not_found_count}"
)

print()
print(
    f"Models directory:"
)
print(
    f"  {MODELS_DIR}"
)

print()
print(
    f"Download manifest:"
)
print(
    f"  {DOWNLOAD_MANIFEST}"
)

print()
print(
    f"Failed downloads:"
)
print(
    f"  {FAILED_PATH}"
)

print("=" * 60)

ABO 3D BATCH DOWNLOADER
Batch:   001
Models:  25
Output:  /content/neural_object_reconstruction/abo_batches/batch_001/models

------------------------------------------------------------
DOWNLOADING MODELS
------------------------------------------------------------
[01/25] B075X4QMW7 | furniture
       ✓ DOWNLOADED (40.49 MB)
[02/25] B084T7GSXM | decorative
       ✓ DOWNLOADED (32.80 MB)
[03/25] B07S74D9T7 | electronics
       ✓ DOWNLOADED (18.72 MB)
[04/25] B07F2X8K62 | furniture
       ✓ DOWNLOADED (53.24 MB)
[05/25] B07Q754Q5X | decorative
       ✓ DOWNLOADED (63.86 MB)
[06/25] B082VLWBGD | furniture
       ✓ DOWNLOADED (34.60 MB)
[07/25] B07ML7JSGM | containers
       ✓ DOWNLOADED (13.10 MB)
[08/25] B07HSJDMXQ | decorative
       ✓ DOWNLOADED (36.05 MB)
[09/25] B087CQFYDY | decorative
       ✓ DOWNLOADED (14.85 MB)
[10/25] B00IIFW2L4 | furniture
       ✓ DOWNLOADED (20.29 MB)
[11/25] B07JK89F5M | furniture
       ✓ DOWNLOADED (9.36 MB)
[12/25] B075X4N4W9 | furniture
       ✓ DOWNL

In [ ]:
# ============================================================
# ABO 3D BATCH 001 — GLB VALIDATION
# ============================================================

import os
import json
from pathlib import Path

PROJECT_ROOT = Path("/content/neural_object_reconstruction")
BATCH_ID = "001"

BATCH_DIR = PROJECT_ROOT / "abo_batches" / f"batch_{BATCH_ID}"
MODELS_DIR = BATCH_DIR / "models"
MANIFEST_PATH = BATCH_DIR / "manifest.jsonl"

print("=" * 60)
print("ABO 3D GLB VALIDATION")
print("=" * 60)

# ------------------------------------------------------------
# CHECK DIRECTORY
# ------------------------------------------------------------

if not MODELS_DIR.exists():
    raise FileNotFoundError(
        f"Models directory not found:\n{MODELS_DIR}"
    )

# ------------------------------------------------------------
# LOAD MANIFEST
# ------------------------------------------------------------

records = []

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if line:
            records.append(json.loads(line))

print(f"Manifest records: {len(records)}")
print(f"GLB directory:    {MODELS_DIR}")

# ------------------------------------------------------------
# VALIDATE FILES
# ------------------------------------------------------------

valid = []
invalid = []
missing = []

print()
print("-" * 60)
print("CHECKING MODELS")
print("-" * 60)

for index, record in enumerate(records, 1):

    item_id = record["item_id"]

    glb_path = MODELS_DIR / f"{item_id}.glb"

    print(
        f"[{index:02d}/{len(records):02d}] "
        f"{item_id}",
        end=" "
    )

    # --------------------------------------------------------
    # Check existence
    # --------------------------------------------------------

    if not glb_path.exists():

        print("✗ MISSING")

        missing.append({
            "item_id": item_id,
            "path": str(glb_path)
        })

        continue

    # --------------------------------------------------------
    # Check file size
    # --------------------------------------------------------

    size = glb_path.stat().st_size

    if size < 100:

        print("✗ TOO SMALL")

        invalid.append({
            "item_id": item_id,
            "path": str(glb_path),
            "reason": "File is suspiciously small",
            "size": size
        })

        continue

    # --------------------------------------------------------
    # Check GLB magic bytes
    #
    # GLB files begin with:
    #     67 6C 54 46
    #     "glTF"
    # --------------------------------------------------------

    try:

        with open(
            glb_path,
            "rb"
        ) as f:

            magic = f.read(4)

    except Exception as e:

        print("✗ READ ERROR")

        invalid.append({
            "item_id": item_id,
            "path": str(glb_path),
            "reason": str(e)
        })

        continue

    if magic != b"glTF":

        print(
            f"✗ INVALID HEADER "
            f"({magic!r})"
        )

        invalid.append({
            "item_id": item_id,
            "path": str(glb_path),
            "reason": "Invalid GLB header",
            "magic": repr(magic)
        })

        continue

    # --------------------------------------------------------
    # Valid
    # --------------------------------------------------------

    size_mb = size / (1024 * 1024)

    print(
        f"✓ VALID ({size_mb:.2f} MB)"
    )

    valid.append({
        "item_id": item_id,
        "path": str(glb_path),
        "size_bytes": size,
        "size_mb": round(size_mb, 2)
    })


# ------------------------------------------------------------
# SAVE VALIDATION REPORT
# ------------------------------------------------------------

report = {
    "batch_id": BATCH_ID,
    "total": len(records),
    "valid": len(valid),
    "invalid": len(invalid),
    "missing": len(missing),
    "valid_models": valid,
    "invalid_models": invalid,
    "missing_models": missing
}

REPORT_PATH = (
    BATCH_DIR /
    "validation_report.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

total_size = sum(
    item["size_bytes"]
    for item in valid
)

total_size_gb = total_size / (
    1024 ** 3
)

print()
print("=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

print(
    f"Total models:       {len(records)}"
)

print(
    f"Valid GLBs:         {len(valid)}"
)

print(
    f"Invalid GLBs:       {len(invalid)}"
)

print(
    f"Missing GLBs:       {len(missing)}"
)

print(
    f"Total valid size:   {total_size_gb:.2f} GB"
)

print()
print(
    f"Validation report:"
)

print(
    f"  {REPORT_PATH}"
)

print("=" * 60)


# ------------------------------------------------------------
# STOP IF ANYTHING IS WRONG
# ------------------------------------------------------------

if invalid or missing:

    print()
    print("⚠️ BATCH HAS PROBLEMS")
    print("Do NOT upload yet.")

else:

    print()
    print("✓ ALL 25 GLB MODELS PASSED BASIC VALIDATION")
    print("✓ BATCH 001 IS READY FOR PACKAGING")

ABO 3D GLB VALIDATION
Manifest records: 25
GLB directory:    /content/neural_object_reconstruction/abo_batches/batch_001/models

------------------------------------------------------------
CHECKING MODELS
------------------------------------------------------------
[01/25] B075X4QMW7 ✓ VALID (40.49 MB)
[02/25] B084T7GSXM ✓ VALID (32.80 MB)
[03/25] B07S74D9T7 ✓ VALID (18.72 MB)
[04/25] B07F2X8K62 ✓ VALID (53.24 MB)
[05/25] B07Q754Q5X ✓ VALID (63.86 MB)
[06/25] B082VLWBGD ✓ VALID (34.60 MB)
[07/25] B07ML7JSGM ✓ VALID (13.10 MB)
[08/25] B07HSJDMXQ ✓ VALID (36.05 MB)
[09/25] B087CQFYDY ✓ VALID (14.85 MB)
[10/25] B00IIFW2L4 ✓ VALID (20.29 MB)
[11/25] B07JK89F5M ✓ VALID (9.36 MB)
[12/25] B075X4N4W9 ✓ VALID (29.38 MB)
[13/25] B07QHL2D2V ✓ VALID (26.58 MB)
[14/25] B07YBHC881 ✓ VALID (33.84 MB)
[15/25] B071LQHVZB ✓ VALID (32.89 MB)
[16/25] B07TMH6289 ✓ VALID (29.34 MB)
[17/25] B07Y33KVDM ✓ VALID (21.48 MB)
[18/25] B082VMDZKG ✓ VALID (46.43 MB)
[19/25] B07QCMB7LV ✓ VALID (56.93 MB)
[20/25] B07S

In [ ]:
# ============================================================
# HUGGING FACE SETUP
# ============================================================

!pip -q install -U huggingface_hub

import os
from pathlib import Path

from huggingface_hub import HfApi, login


# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

HF_REPO = "ndeda/neural-object-reconstruction-abo"

PROJECT_ROOT = Path(
    "/content/neural_object_reconstruction"
)

QUALITY_POOL = (
    PROJECT_ROOT
    / "abo_candidates"
    / "3d_quality_pool_categorized.jsonl"
)

METADATA_PATH = (
    PROJECT_ROOT
    / "abo_metadata"
    / "3dmodels"
    / "metadata"
    / "3dmodels.csv.gz"
)

print("=" * 60)
print("HUGGING FACE SETUP")
print("=" * 60)

print(f"Repository:")
print(f"  {HF_REPO}")

print()
print("Log in to Hugging Face.")
print("Use a token with write permission.")


# ------------------------------------------------------------
# LOGIN
# ------------------------------------------------------------

login()


# ------------------------------------------------------------
# API
# ------------------------------------------------------------

api = HfApi()

try:

    info = api.repo_info(
        repo_id=HF_REPO,
        repo_type="dataset"
    )

    print()
    print("✓ Dataset repository found")
    print(f"  {info.id}")

except Exception:

    print()
    print("Dataset repository does not appear to exist.")
    print("Creating it...")

    api.create_repo(
        repo_id=HF_REPO,
        repo_type="dataset",
        private=False,
        exist_ok=True
    )

    print("✓ Dataset repository created")


# ------------------------------------------------------------
# VERIFY LOCAL FILES
# ------------------------------------------------------------

print()
print("-" * 60)
print("CHECKING LOCAL DATA")
print("-" * 60)

if not QUALITY_POOL.exists():
    raise FileNotFoundError(
        f"Quality pool not found:\n{QUALITY_POOL}"
    )

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"3D metadata not found:\n{METADATA_PATH}"
    )

print("✓ Quality pool found")
print("✓ 3D metadata found")

print()
print("=" * 60)
print("HUGGING FACE SETUP COMPLETE")
print("=" * 60)

HUGGING FACE SETUP
Repository:
  ndeda/neural-object-reconstruction-abo

Log in to Hugging Face.
Use a token with write permission.

✓ Dataset repository found
  ndeda/neural-object-reconstruction-abo

------------------------------------------------------------
CHECKING LOCAL DATA
------------------------------------------------------------
✓ Quality pool found
✓ 3D metadata found

HUGGING FACE SETUP COMPLETE


In [ ]:
# ============================================================
# ABO 3D PRODUCTION BATCH PIPELINE
#
# BUILD
#   ↓
# DOWNLOAD
#   ↓
# VALIDATE
#   ↓
# UPLOAD TO HUGGING FACE
#   ↓
# VERIFY REMOTE FILES
#   ↓
# DELETE LOCAL BATCH
#
# One batch = 25 models
# ============================================================

import os
import json
import shutil
import time
import requests
import csv

from pathlib import Path
from huggingface_hub import HfApi


# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    "/content/neural_object_reconstruction"
)

QUALITY_POOL_PATH = (
    PROJECT_ROOT
    / "abo_candidates"
    / "3d_quality_pool_categorized.jsonl"
)

METADATA_PATH = (
    PROJECT_ROOT
    / "abo_metadata"
    / "3dmodels"
    / "metadata"
    / "3dmodels.csv.gz"
)

LISTINGS_DIR = (
    PROJECT_ROOT
    / "abo_metadata"
    / "listings"
    / "metadata"
)

BATCHES_DIR = (
    PROJECT_ROOT
    / "abo_batches"
)

HF_REPO = (
    "ndeda/neural-object-reconstruction-abo"
)

HF_BASE_PATH = "batches"

BATCH_SIZE = 25

MODEL_BASE_URL = (
    "https://amazon-berkeley-objects.s3.amazonaws.com"
    "/3dmodels/original"
)


# ============================================================
# HUGGING FACE API
# ============================================================

api = HfApi()


# ============================================================
# HELPER — LOAD QUALITY POOL
# ============================================================

def load_quality_pool():

    records = []

    with open(
        QUALITY_POOL_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if line:
                records.append(
                    json.loads(line)
                )

    return records


# ============================================================
# HELPER — BUILD BATCH MANIFEST
# ============================================================

def build_batch(batch_id):

    print()
    print("=" * 60)
    print(f"BUILDING BATCH {batch_id:03d}")
    print("=" * 60)

    quality_pool = load_quality_pool()

    print(
        f"Quality pool records: "
        f"{len(quality_pool)}"
    )

    start = (
        batch_id - 1
    ) * BATCH_SIZE

    end = start + BATCH_SIZE

    selected = quality_pool[start:end]

    if len(selected) != BATCH_SIZE:

        raise RuntimeError(
            f"Batch {batch_id:03d} contains "
            f"{len(selected)} records instead of "
            f"{BATCH_SIZE}."
        )

    batch_name = (
        f"batch_{batch_id:03d}"
    )

    batch_dir = (
        BATCHES_DIR
        / batch_name
    )

    models_dir = (
        batch_dir
        / "models"
    )

    batch_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    models_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    manifest_path = (
        batch_dir
        / "manifest.jsonl"
    )

    csv_path = (
        batch_dir
        / "manifest.csv"
    )

    manifest_records = []

    for item in selected:

        record = dict(item)

        manifest_records.append(
            record
        )

    # --------------------------------------------------------
    # JSONL
    # --------------------------------------------------------

    with open(
        manifest_path,
        "w",
        encoding="utf-8"
    ) as f:

        for record in manifest_records:

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    if manifest_records:

        keys = sorted(
            set().union(
                *[
                    record.keys()
                    for record in manifest_records
                ]
            )
        )

        with open(
            csv_path,
            "w",
            encoding="utf-8",
            newline=""
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=keys
            )

            writer.writeheader()

            for record in manifest_records:

                row = {}

                for key in keys:

                    value = record.get(
                        key,
                        ""
                    )

                    if isinstance(
                        value,
                        (dict, list)
                    ):

                        value = json.dumps(
                            value,
                            ensure_ascii=False
                        )

                    row[key] = value

                writer.writerow(row)


    print()
    print(
        f"Selected: {len(selected)}"
    )

    print(
        f"Range: {start} → {end - 1}"
    )

    return (
        batch_dir,
        models_dir,
        manifest_records
    )


# ============================================================
# HELPER — DOWNLOAD ONE MODEL
# ============================================================

def download_model(
    model_path,
    output_path,
    timeout=180
):

    url = (
        f"{MODEL_BASE_URL}/"
        f"{model_path}"
    )

    if output_path.exists():

        if output_path.stat().st_size > 0:

            return {
                "status": "exists",
                "url": url
            }


    temp_path = output_path.with_suffix(
        ".glb.part"
    )

    if temp_path.exists():
        temp_path.unlink()


    try:

        with requests.get(
            url,
            stream=True,
            timeout=timeout
        ) as response:

            # ------------------------------------------------
            # Never retry 404
            # ------------------------------------------------

            if response.status_code == 404:

                return {
                    "status": "404",
                    "url": url,
                    "error": "Not found"
                }

            response.raise_for_status()

            with open(
                temp_path,
                "wb"
            ) as f:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:

                        f.write(chunk)


        if not temp_path.exists():

            return {
                "status": "empty",
                "url": url
            }


        size = temp_path.stat().st_size

        if size == 0:

            temp_path.unlink()

            return {
                "status": "empty",
                "url": url
            }


        temp_path.replace(
            output_path
        )


        return {
            "status": "downloaded",
            "url": url,
            "size_bytes": size,
            "size_mb": size / (
                1024 * 1024
            )
        }


    except Exception as e:

        if temp_path.exists():
            temp_path.unlink()

        return {
            "status": "error",
            "url": url,
            "error": str(e)
        }


# ============================================================
# HELPER — DOWNLOAD BATCH
# ============================================================

def download_batch(
    batch_id,
    models_dir,
    manifest_records
):

    print()
    print("-" * 60)
    print(
        f"DOWNLOADING BATCH {batch_id:03d}"
    )
    print("-" * 60)

    results = []

    for index, record in enumerate(
        manifest_records,
        start=1
    ):

        item_id = record["item_id"]

        model_path = record.get(
            "model_path"
        )

        category = record.get(
            "category",
            "unknown"
        )

        print(
            f"[{index:02d}/{len(manifest_records):02d}] "
            f"{item_id} | {category}"
        )

        if not model_path:

            print(
                "       ✗ NO MODEL PATH"
            )

            results.append({
                "item_id": item_id,
                "status": "missing_path"
            })

            continue


        output_path = (
            models_dir
            / f"{item_id}.glb"
        )


        result = download_model(
            model_path,
            output_path
        )

        result.update({
            "item_id": item_id,
            "3dmodel_id": record.get(
                "3dmodel_id",
                item_id
            ),
            "model_path": model_path,
            "category": category,
            "local_path": str(
                output_path
            )
        })

        results.append(result)


        if result["status"] == "downloaded":

            print(
                f"       ✓ "
                f"{result['size_mb']:.2f} MB"
            )

        elif result["status"] == "exists":

            print(
                "       ✓ ALREADY EXISTS"
            )

        else:

            print(
                f"       ✗ "
                f"{result['status']}"
            )


    return results


# ============================================================
# HELPER — VALIDATE GLBS
# ============================================================

def validate_batch(
    models_dir,
    manifest_records
):

    print()
    print("-" * 60)
    print("VALIDATING GLB FILES")
    print("-" * 60)

    valid = []
    invalid = []

    for index, record in enumerate(
        manifest_records,
        start=1
    ):

        item_id = record["item_id"]

        path = (
            models_dir
            / f"{item_id}.glb"
        )

        print(
            f"[{index:02d}/{len(manifest_records):02d}] "
            f"{item_id}",
            end=" "
        )


        if not path.exists():

            print("✗ MISSING")

            invalid.append({
                "item_id": item_id,
                "reason": "missing"
            })

            continue


        size = path.stat().st_size


        if size < 100:

            print("✗ TOO SMALL")

            invalid.append({
                "item_id": item_id,
                "reason": "too_small",
                "size": size
            })

            continue


        try:

            with open(
                path,
                "rb"
            ) as f:

                header = f.read(4)

        except Exception as e:

            print("✗ READ ERROR")

            invalid.append({
                "item_id": item_id,
                "reason": str(e)
            })

            continue


        if header != b"glTF":

            print("✗ INVALID GLB")

            invalid.append({
                "item_id": item_id,
                "reason": "invalid_glb_header"
            })

            continue


        size_mb = (
            size /
            (1024 * 1024)
        )

        print(
            f"✓ {size_mb:.2f} MB"
        )

        valid.append({
            "item_id": item_id,
            "size_bytes": size,
            "size_mb": round(
                size_mb,
                2
            )
        })


    report = {
        "total": len(
            manifest_records
        ),
        "valid": len(valid),
        "invalid": len(invalid),
        "valid_models": valid,
        "invalid_models": invalid
    }

    return report


# ============================================================
# HELPER — WRITE REPORTS
# ============================================================

def write_reports(
    batch_dir,
    manifest_records,
    download_results,
    validation_report
):

    download_manifest = (
        batch_dir
        / "download_manifest.jsonl"
    )

    with open(
        download_manifest,
        "w",
        encoding="utf-8"
    ) as f:

        for result in download_results:

            f.write(
                json.dumps(
                    result,
                    ensure_ascii=False
                )
                + "\n"
            )


    validation_path = (
        batch_dir
        / "validation_report.json"
    )

    with open(
        validation_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            validation_report,
            f,
            indent=2,
            ensure_ascii=False
        )


# ============================================================
# HELPER — UPLOAD BATCH
# ============================================================

def upload_batch(
    batch_id,
    batch_dir
):

    print()
    print("-" * 60)
    print(
        f"UPLOADING BATCH {batch_id:03d}"
    )
    print("-" * 60)

    remote_path = (
        f"{HF_BASE_PATH}/"
        f"batch_{batch_id:03d}"
    )

    print(
        f"Remote path:"
    )

    print(
        f"  {remote_path}"
    )

    print()
    print("Uploading...")


    api.upload_folder(
        repo_id=HF_REPO,
        repo_type="dataset",
        folder_path=str(batch_dir),
        path_in_repo=remote_path,
        commit_message=(
            f"Add ABO 3D batch "
            f"{batch_id:03d}"
        )
    )


    print()
    print("✓ Upload completed")


# ============================================================
# HELPER — VERIFY HUGGING FACE
# ============================================================

def verify_remote_batch(
    batch_id,
    manifest_records
):

    print()
    print("-" * 60)
    print(
        f"VERIFYING HUGGING FACE BATCH "
        f"{batch_id:03d}"
    )
    print("-" * 60)


    files = api.list_repo_files(
        repo_id=HF_REPO,
        repo_type="dataset"
    )


    remote_prefix = (
        f"{HF_BASE_PATH}/"
        f"batch_{batch_id:03d}/"
        f"models/"
    )


    expected = []

    for record in manifest_records:

        expected.append(
            remote_prefix
            + record["item_id"]
            + ".glb"
        )


    missing = [
        path
        for path in expected
        if path not in files
    ]

    remote_count = sum(1 for path in expected if path in files)

    print(
        f"Expected GLBs: {len(expected)}"
    )

    print(
        f"Remote GLBs:   {remote_count}"
    )


    if missing:

        print()
        print(
            "✗ REMOTE VERIFICATION FAILED"
        )

        for path in missing:

            print(
                f"  Missing: {path}"
            )

        return False


    print()
    print(
        "✓ ALL MODELS VERIFIED ON "
        "HUGGING FACE"
    )

    return True


# ============================================================
# HELPER — DELETE LOCAL BATCH
# ============================================================

def delete_local_batch(
    batch_dir
):

    print()
    print("-" * 60)
    print("CLEANING LOCAL COLAB STORAGE")
    print("-" * 60)

    if not batch_dir.exists():

        print(
            "Batch directory already removed."
        )

        return


    shutil.rmtree(
        batch_dir
    )


    if batch_dir.exists():

        raise RuntimeError(
            "Failed to remove local batch."
        )


    print(
        "✓ Local batch deleted"
    )


# ============================================================
# MAIN PROCESS
# ============================================================

def process_batch(
    batch_id
):

    # --------------------------------------------------------
    # BUILD
    # --------------------------------------------------------

    (
        batch_dir,
        models_dir,
        manifest_records
    ) = build_batch(
        batch_id
    )


    # --------------------------------------------------------
    # DOWNLOAD
    # --------------------------------------------------------

    download_results = download_batch(
        batch_id,
        models_dir,
        manifest_records
    )


    # --------------------------------------------------------
    # STOP IF DOWNLOAD FAILED
    # --------------------------------------------------------

    failed_downloads = [
        result
        for result in download_results
        if result["status"]
        not in (
            "downloaded",
            "exists"
        )
    ]


    if failed_downloads:

        write_reports(
            batch_dir,
            manifest_records,
            download_results,
            {
                "status": "download_failed",
                "failed": failed_downloads
            }
        )

        raise RuntimeError(
            f"Batch {batch_id:03d} "
            f"has {len(failed_downloads)} "
            f"download failures. "
            f"NOT uploading."
        )


    # --------------------------------------------------------
    # VALIDATE
    # --------------------------------------------------------

    validation_report = validate_batch(
        models_dir,
        manifest_records
    )


    # --------------------------------------------------------
    # SAVE REPORTS
    # --------------------------------------------------------

    write_reports(
        batch_dir,
        manifest_records,
        download_results,
        validation_report
    )


    # --------------------------------------------------------
    # STOP IF VALIDATION FAILED
    # --------------------------------------------------------

    if validation_report["invalid"] > 0:

        raise RuntimeError(
            f"Batch {batch_id:03d} "
            f"failed GLB validation. "
            f"NOT uploading."
        )


    # --------------------------------------------------------
    # UPLOAD
    # --------------------------------------------------------

    upload_batch(
        batch_id,
        batch_dir
    )


    # --------------------------------------------------------
    # VERIFY
    # --------------------------------------------------------

    verified = verify_remote_batch(
        batch_id,
        manifest_records
    )


    # --------------------------------------------------------
    # NEVER DELETE IF VERIFICATION FAILED
    # --------------------------------------------------------

    if not verified:

        raise RuntimeError(
            "Hugging Face verification failed. "
            "Local files have been preserved."
        )


    # --------------------------------------------------------
    # DELETE LOCAL
    # --------------------------------------------------------

    delete_local_batch(
        batch_dir
    )


    # --------------------------------------------------------
    # FINAL STATUS
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print(
        f"BATCH {batch_id:03d} COMPLETE"
    )
    print("=" * 60)

    print(
        "✓ 25 models selected"
    )

    print(
        "✓ 25 models downloaded"
    )

    print(
        "✓ 25 models validated"
    )

    print(
        "✓ 25 models uploaded"
    )

    print(
        "✓ 25 models verified remotely"
    )

    print(
        "✓ Local batch deleted"
    )

    print()
    print(
        "Ready for next batch."
    )

    print("=" * 60)

In [ ]:
# ============================================================
# RUN ABO BATCH 002
# ============================================================

process_batch(2)


BUILDING BATCH 002
Quality pool records: 250

Selected: 25
Range: 25 → 49

------------------------------------------------------------
DOWNLOADING BATCH 002
------------------------------------------------------------
[01/25] B07QC85X9C | furniture
       ✓ 15.00 MB
[02/25] B00NO73Q84 | fashion_accessories
       ✓ 4.74 MB
[03/25] B07B3XXD3P | furniture
       ✓ 25.29 MB
[04/25] B07VSV4783 | furniture
       ✓ 49.30 MB
[05/25] B07DBHCKHY | furniture
       ✓ 36.07 MB
[06/25] B07TJVTZRF | furniture
       ✓ 9.49 MB
[07/25] B082Q9C9YJ | furniture
       ✓ 69.31 MB
[08/25] B07TZ9BMJG | furniture
       ✓ 20.96 MB
[09/25] B07JGPK68C | furniture
       ✓ 0.81 MB
[10/25] B07SJ75Y1M | furniture
       ✓ 23.78 MB
[11/25] B075NP5K3L | furniture
       ✓ 13.19 MB
[12/25] B082JH6LSF | furniture
       ✓ 38.41 MB
[13/25] B075X1T4ZS | furniture
       ✓ 38.56 MB
[14/25] B07DB92HMS | furniture
       ✓ 25.58 MB
[15/25] B07MM5H3HX | furniture
       ✓ 9.40 MB
[16/25] B07QV18T5Y | mechanical
       

In [ ]:
# ============================================================
# UPLOAD + VERIFY EXISTING BATCH 001
# ============================================================

BATCH_ID = 1

BATCH_DIR = (
    PROJECT_ROOT
    / "abo_batches"
    / "batch_001"
)

MANIFEST_PATH = (
    BATCH_DIR
    / "manifest.jsonl"
)

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    manifest_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print(
    f"Batch 001 records: "
    f"{len(manifest_records)}"
)

validation_report = validate_batch(
    BATCH_DIR / "models",
    manifest_records
)

if validation_report["invalid"] != 0:

    raise RuntimeError(
        "Batch 001 validation failed. "
        "Do not upload."
    )

print(
    "✓ Batch 001 validation passed"
)

upload_batch(
    1,
    BATCH_DIR
)

verified = verify_remote_batch(
    1,
    manifest_records
)

if not verified:

    raise RuntimeError(
        "Batch 001 remote verification failed. "
        "Keeping local files."
    )

print(
    "✓ Batch 001 uploaded and verified"
)

delete_local_batch(
    BATCH_DIR
)

print()
print("=" * 60)
print("BATCH 001 COMPLETE")
print("=" * 60)

Batch 001 records: 25

------------------------------------------------------------
VALIDATING GLB FILES
------------------------------------------------------------
[01/25] B075X4QMW7 ✓ 40.49 MB
[02/25] B084T7GSXM ✓ 32.80 MB
[03/25] B07S74D9T7 ✓ 18.72 MB
[04/25] B07F2X8K62 ✓ 53.24 MB
[05/25] B07Q754Q5X ✓ 63.86 MB
[06/25] B082VLWBGD ✓ 34.60 MB
[07/25] B07ML7JSGM ✓ 13.10 MB
[08/25] B07HSJDMXQ ✓ 36.05 MB
[09/25] B087CQFYDY ✓ 14.85 MB
[10/25] B00IIFW2L4 ✓ 20.29 MB
[11/25] B07JK89F5M ✓ 9.36 MB
[12/25] B075X4N4W9 ✓ 29.38 MB
[13/25] B07QHL2D2V ✓ 26.58 MB
[14/25] B07YBHC881 ✓ 33.84 MB
[15/25] B071LQHVZB ✓ 32.89 MB
[16/25] B07TMH6289 ✓ 29.34 MB
[17/25] B07Y33KVDM ✓ 21.48 MB
[18/25] B082VMDZKG ✓ 46.43 MB
[19/25] B07QCMB7LV ✓ 56.93 MB
[20/25] B07SGZP8VC ✓ 18.50 MB
[21/25] B07VDD538M ✓ 14.86 MB
[22/25] B07P6JV16D ✓ 59.88 MB
[23/25] B075YP4WVQ ✓ 49.70 MB
[24/25] B07K4YXTZH ✓ 7.06 MB
[25/25] B071PDYKYB ✓ 35.42 MB
✓ Batch 001 validation passed

-------------------------------------------------------

In [ ]:
# ============================================================
# RUN ABO BATCH 003
# ============================================================

process_batch(3)


BUILDING BATCH 003
Quality pool records: 250

Selected: 25
Range: 50 → 74

------------------------------------------------------------
DOWNLOADING BATCH 003
------------------------------------------------------------
[01/25] B082BL4G1J | furniture
       ✓ 36.89 MB
[02/25] B07K6S2G6J | furniture
       ✓ 10.70 MB
[03/25] B0853JGLT7 | furniture
       ✓ 36.47 MB
[04/25] B07WL2VHTG | furniture
       ✓ 38.12 MB
[05/25] B085FHCKY4 | furniture
       ✓ 41.77 MB
[06/25] B07RNMNFQ5 | decorative
       ✓ 28.17 MB
[07/25] B01LONQ3TS | electronics
       ✓ 8.25 MB
[08/25] B085FHBF2S | furniture
       ✓ 45.21 MB
[09/25] B07QJXMX23 | furniture
       ✓ 46.27 MB
[10/25] B082VLYP6B | furniture
       ✓ 26.04 MB
[11/25] B08DWMY83P | furniture
       ✓ 31.09 MB
[12/25] B07GP6DMDF | furniture
       ✓ 21.55 MB
[13/25] B07MBFDRYD | furniture
       ✓ 46.82 MB
[14/25] B07RTZ6NBL | decorative
       ✓ 19.20 MB
[15/25] B082VMDZKZ | furniture
       ✓ 45.43 MB
[16/25] B0735SK5LP | decorative
       ✓ 4

In [ ]:
# ============================================================
# RUN ABO BATCH 004
# ============================================================

process_batch(4)


BUILDING BATCH 004
Quality pool records: 250

Selected: 25
Range: 75 → 99

------------------------------------------------------------
DOWNLOADING BATCH 004
------------------------------------------------------------
[01/25] B07P5LN16X | furniture
       ✓ 35.86 MB
[02/25] B07QB8L7ZC | kitchen
       ✓ 16.53 MB
[03/25] B084W2G21K | furniture
       ✓ 65.46 MB
[04/25] B07QFRT4NR | tools
       ✓ 38.89 MB
[05/25] B07BWLC7ZR | furniture
       ✓ 44.23 MB
[06/25] B07FJVXVNW | decorative
       ✓ 7.50 MB
[07/25] B075X33RZL | furniture
       ✓ 61.16 MB
[08/25] B07MBFDJQ3 | furniture
       ✓ 27.05 MB
[09/25] B07V4FNHCD | kitchen
       ✓ 37.17 MB
[10/25] B088HDDTM7 | furniture
       ✓ 16.51 MB
[11/25] B0853KTNN8 | furniture
       ✓ 43.51 MB
[12/25] B07QK35BD1 | tools
       ✓ 15.04 MB
[13/25] B082QDDJXJ | furniture
       ✓ 67.92 MB
[14/25] B07BL5WPHR | decorative
       ✓ 1.80 MB
[15/25] B085GXHL6X | furniture
       ✓ 26.66 MB
[16/25] B07BWJCPSN | furniture
       ✓ 49.77 MB
[17/25] 

In [ ]:
# ============================================================
# RUN ABO BATCH 005
# ============================================================

process_batch(5)


BUILDING BATCH 005
Quality pool records: 250

Selected: 25
Range: 100 → 124

------------------------------------------------------------
DOWNLOADING BATCH 005
------------------------------------------------------------
[01/25] B07RPPBPTP | decorative
       ✓ 17.30 MB
[02/25] B07B7N6JH3 | furniture
       ✓ 4.74 MB
[03/25] B07QJ24FSL | furniture
       ✓ 52.18 MB
[04/25] B075X4F3V5 | furniture
       ✓ 27.03 MB
[05/25] B07PBZ9CLJ | furniture
       ✓ 42.45 MB
[06/25] B07TTY3FYP | furniture
       ✓ 43.43 MB
[07/25] B07K7LZVPT | furniture
       ✓ 27.54 MB
[08/25] B0825CP3NS | furniture
       ✓ 20.32 MB
[09/25] B07M7NRPZR | furniture
       ✓ 35.74 MB
[10/25] B07DBF3TWR | furniture
       ✓ 24.50 MB
[11/25] B07B4ZXKFQ | furniture
       ✓ 24.61 MB
[12/25] B07F2HNX7W | tools
       ✓ 38.22 MB
[13/25] B07QBMQ729 | furniture
       ✓ 40.31 MB
[14/25] B07QD5G1KC | kitchen
       ✓ 24.40 MB
[15/25] B07QBMQQ34 | decorative
       ✓ 40.98 MB
[16/25] B07PXDFW6L | furniture
       ✓ 26.27 MB

In [ ]:
# ============================================================
# RUN ABO BATCH 006
# ============================================================

process_batch(6)


BUILDING BATCH 006
Quality pool records: 250

Selected: 25
Range: 125 → 149

------------------------------------------------------------
DOWNLOADING BATCH 006
------------------------------------------------------------
[01/25] B082VL7BQ1 | furniture
       ✓ 35.09 MB
[02/25] B07YM1FJ1M | fashion_accessories
       ✓ 25.58 MB
[03/25] B00EUL2B16 | furniture
       ✓ 32.51 MB
[04/25] B0824FH9GS | furniture
       ✓ 57.31 MB
[05/25] B07K7NYB8K | decorative
       ✓ 13.67 MB
[06/25] B082JHXMHL | kitchen
       ✓ 23.46 MB
[07/25] B07QFB1L2P | furniture
       ✓ 20.78 MB
[08/25] B07HZ1P1FX | furniture
       ✓ 73.24 MB
[09/25] B082JH9VWG | decorative
       ✓ 32.94 MB
[10/25] B0831Y1G6Y | furniture
       ✓ 75.04 MB
[11/25] B07PBZ74HD | furniture
       ✓ 31.15 MB
[12/25] B07MF1SRM6 | containers
       ✓ 70.89 MB
[13/25] B07L9BVBCJ | furniture
       ✓ 13.20 MB
[14/25] B07RMYWFHY | furniture
       ✓ 20.15 MB
[15/25] B07TXPYLVY | tools
       ✓ 23.55 MB
[16/25] B07B4YC7LD | containers
    

In [ ]:
# ============================================================
# RUN ABO BATCH 007
# ============================================================

process_batch(7)


BUILDING BATCH 007
Quality pool records: 250

Selected: 25
Range: 150 → 174

------------------------------------------------------------
DOWNLOADING BATCH 007
------------------------------------------------------------
[01/25] B0861YKM8S | furniture
       ✓ 20.66 MB
[02/25] B07GZY7JFV | furniture
       ✓ 21.38 MB
[03/25] B07PZCRS6F | furniture
       ✓ 11.61 MB
[04/25] B07SVY2HW1 | electronics
       ✓ 20.19 MB
[05/25] B082QCDT5R | furniture
       ✓ 67.53 MB
[06/25] B017DORX3M | electronics
       ✓ 36.30 MB
[07/25] B07B4YXXPZ | furniture
       ✓ 38.56 MB
[08/25] B07JD47LW3 | furniture
       ✓ 53.05 MB
[09/25] B07M6PKC6B | furniture
       ✓ 20.28 MB
[10/25] B07SFXGYPW | furniture
       ✓ 12.55 MB
[11/25] B07JKJSTVJ | furniture
       ✓ 44.60 MB
[12/25] B07HPJ9V3J | furniture
       ✓ 34.21 MB
[13/25] B07QD6ZDDH | kitchen
       ✓ 26.58 MB
[14/25] B07QX2BYDH | furniture
       ✓ 10.88 MB
[15/25] B07B4ZGB6K | furniture
       ✓ 23.45 MB
[16/25] B086VLRYXS | furniture
       ✓ 1

In [ ]:
# ============================================================
# RUN ABO BATCH 008
# ============================================================

process_batch(8)


BUILDING BATCH 008
Quality pool records: 250

Selected: 25
Range: 175 → 199

------------------------------------------------------------
DOWNLOADING BATCH 008
------------------------------------------------------------
[01/25] B075ZBLWBH | mechanical
       ✓ 39.72 MB
[02/25] B07J1YW3YT | furniture
       ✓ 35.72 MB
[03/25] B07TGC3432 | furniture
       ✓ 27.63 MB
[04/25] B07K7NV52J | furniture
       ✓ 32.57 MB
[05/25] B07374SBFN | furniture
       ✓ 16.13 MB
[06/25] B071ZJ6CD5 | mechanical
       ✓ 50.87 MB
[07/25] B07B4NW6J9 | furniture
       ✓ 72.44 MB
[08/25] B07SPY6R7V | furniture
       ✓ 17.51 MB
[09/25] B01MQJV7ID | furniture
       ✓ 25.28 MB
[10/25] B07H8PWFFS | furniture
       ✓ 2.13 MB
[11/25] B07ML7P93X | decorative
       ✓ 8.38 MB
[12/25] B07MHMSDJL | furniture
       ✓ 8.18 MB
[13/25] B082VLYCPC | furniture
       ✓ 48.75 MB
[14/25] B07Y3254F9 | decorative
       ✓ 17.39 MB
[15/25] B07H8NZHG9 | furniture
       ✓ 13.32 MB
[16/25] B082VKTV1P | furniture
       ✓ 47

In [ ]:
# ============================================================
# RUN ABO BATCH 009
# ============================================================

process_batch(9)


BUILDING BATCH 009
Quality pool records: 250

Selected: 25
Range: 200 → 224

------------------------------------------------------------
DOWNLOADING BATCH 009
------------------------------------------------------------
[01/25] B07WQ4P35G | furniture
       ✓ 15.55 MB
[02/25] B082XCSHKB | fashion_accessories
       ✓ 16.62 MB
[03/25] B085FJW92H | furniture
       ✓ 56.84 MB
[04/25] B07RQMSJRX | decorative
       ✓ 37.99 MB
[05/25] B01LP0V4JY | sports
       ✓ 42.04 MB
[06/25] B07HSG73VJ | decorative
       ✓ 59.22 MB
[07/25] B08G1YWN8L | furniture
       ✓ 56.33 MB
[08/25] B07MJJWYS8 | furniture
       ✓ 26.15 MB
[09/25] B07R4T5PN8 | furniture
       ✓ 84.91 MB
[10/25] B082Q8RSQ7 | furniture
       ✓ 31.85 MB
[11/25] B07M6PKCPF | unknown
       ✓ 46.45 MB
[12/25] B07QGWLBCV | furniture
       ✓ 57.49 MB
[13/25] B07MBFDJT2 | furniture
       ✓ 21.88 MB
[14/25] B07GZW9PKJ | containers
       ✓ 11.71 MB
[15/25] B07B4NW75W | furniture
       ✓ 40.40 MB
[16/25] B07H8PS4FZ | furniture
    

In [ ]:
# ============================================================
# RUN ABO BATCH 010
# ============================================================

process_batch(10)


BUILDING BATCH 010
Quality pool records: 250

Selected: 25
Range: 225 → 249

------------------------------------------------------------
DOWNLOADING BATCH 010
------------------------------------------------------------
[01/25] B0853KZW8B | tools
       ✓ 73.72 MB
[02/25] B07QBMQPH3 | decorative
       ✓ 78.87 MB
[03/25] B07D4FS7GY | tools
       ✓ 38.89 MB
[04/25] B07H8PQFS4 | furniture
       ✓ 34.68 MB
[05/25] B082QCY5RF | furniture
       ✓ 62.09 MB
[06/25] B07TWQTVXL | containers
       ✓ 22.67 MB
[07/25] B07TC2NYX8 | tools
       ✓ 39.33 MB
[08/25] B07RWCQNPL | furniture
       ✓ 11.63 MB
[09/25] B07RF89K3K | furniture
       ✓ 42.33 MB
[10/25] B07HZ75R6L | furniture
       ✓ 46.04 MB
[11/25] B07QGWNLWD | tools
       ✓ 37.19 MB
[12/25] B002CM3J2A | furniture
       ✓ 30.56 MB
[13/25] B072M1WJC7 | furniture
       ✓ 33.75 MB
[14/25] B07MCCRNLV | furniture
       ✓ 23.45 MB
[15/25] B07S6WNZ7Y | furniture
       ✓ 5.03 MB
[16/25] B07B4W5RF7 | containers
       ✓ 37.57 MB
[17/25] 

In [54]:
# ============================================================
# FINAL PROJECT DOCUMENTATION + HUGGING FACE DATASET CARD
# Neural Object Reconstruction & Relighting Lab
#
# This is the final documentation cell.
#
# It:
#   1. Inspects the actual ABO dataset collected in this notebook
#   2. Creates Hugging Face dataset documentation
#   3. Creates citation metadata
#   4. Creates a machine-readable dataset summary
#   5. Documents provenance and limitations
#   6. Uploads the documentation to Hugging Face
#
# IMPORTANT:
# This documentation describes the work actually performed.
# It does NOT claim that the dataset was created from scratch.
#
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import csv
import os
import re

from huggingface_hub import HfApi


# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    "/content/neural_object_reconstruction"
)

HF_REPO = (
    "ndeda/neural-object-reconstruction-abo"
)

HF_REPO_TYPE = "dataset"

DOCUMENTATION_DIR = (
    PROJECT_ROOT
    / "huggingface_documentation"
)

DOCUMENTATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QUALITY_POOL_PATH = (
    PROJECT_ROOT
    / "abo_candidates"
    / "3d_quality_pool_categorized.jsonl"
)

BATCH_ROOT = (
    PROJECT_ROOT
    / "abo_batches"
)

MODEL_METADATA_PATH = (
    PROJECT_ROOT
    / "abo_metadata"
    / "3dmodels"
    / "metadata"
    / "3dmodels.csv.gz"
)

MODEL_BASE_URL = (
    "https://amazon-berkeley-objects.s3.amazonaws.com"
    "/3dmodels/original"
)

CREATED_AT = datetime.now(
    timezone.utc
).strftime(
    "%Y-%m-%d %H:%M:%S UTC"
)


# ============================================================
# 2. BASIC INFORMATION
# ============================================================

PROJECT_TITLE = (
    "Neural Object Reconstruction & Relighting Lab"
)

DATASET_TITLE = (
    "Amazon Berkeley Objects 3D Selection "
    "for Neural Object Reconstruction"
)

SHORT_DESCRIPTION = (
    "A quality-filtered subset of 3D product assets "
    "from the Amazon Berkeley Objects dataset, prepared "
    "for experiments in multi-view reconstruction, "
    "3D Gaussian Splatting, material decomposition, "
    "and relighting."
)


# ============================================================
# 3. INSPECT LOCAL DATA
# ============================================================

print("=" * 72)
print("FINAL DATASET DOCUMENTATION")
print("=" * 72)

print(
    f"\nProject root:\n{PROJECT_ROOT}"
)

print(
    f"\nHugging Face repository:\n"
    f"{HF_REPO}"
)


# ============================================================
# LOAD QUALITY POOL
# ============================================================

quality_records = []

if QUALITY_POOL_PATH.exists():

    with open(
        QUALITY_POOL_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if line:

                try:

                    quality_records.append(
                        json.loads(line)
                    )

                except json.JSONDecodeError:

                    pass


print(
    f"\nQuality pool records: "
    f"{len(quality_records):,}"
)


# ============================================================
# DISCOVER DOWNLOADED MODELS
# ============================================================

downloaded_models = []

batch_statistics = []

if BATCH_ROOT.exists():

    for batch_dir in sorted(
        BATCH_ROOT.glob("batch_*")
    ):

        if not batch_dir.is_dir():
            continue

        models_dir = (
            batch_dir
            / "models"
        )

        if not models_dir.exists():
            continue

        glb_files = sorted(
            models_dir.glob("*.glb")
        )

        batch_statistics.append({
            "batch": batch_dir.name,
            "models": len(glb_files)
        })

        for glb in glb_files:

            downloaded_models.append(
                glb
            )


TOTAL_DOWNLOADED = len(
    downloaded_models
)


print(
    f"Downloaded GLB models found locally: "
    f"{TOTAL_DOWNLOADED:,}"
)


# ============================================================
# DISCOVER MANIFESTS
# ============================================================

manifest_files = sorted(
    BATCH_ROOT.glob(
        "batch_*/manifest.jsonl"
    )
)

download_manifest_files = sorted(
    BATCH_ROOT.glob(
        "batch_*/download_manifest.jsonl"
    )
)


print(
    f"Batch manifests: "
    f"{len(manifest_files):,}"
)

print(
    f"Download manifests: "
    f"{len(download_manifest_files):,}"
)


# ============================================================
# CATEGORY STATISTICS
# ============================================================

categories = Counter()

for record in quality_records:

    category = record.get(
        "category",
        "unknown"
    )

    if category is None:
        category = "unknown"

    category = str(
        category
    ).strip().lower()

    if not category:
        category = "unknown"

    categories[category] += 1


# ============================================================
# BATCH STATISTICS
# ============================================================

TOTAL_BATCHES = len(
    batch_statistics
)

BATCH_SIZE_OBSERVED = (
    max(
        (
            x["models"]
            for x in batch_statistics
        ),
        default=0
    )
)


# ============================================================
# 4. WRITE DATASET SUMMARY JSON
# ============================================================

dataset_summary = {

    "dataset_name":
        DATASET_TITLE,

    "project":
        PROJECT_TITLE,

    "repository":
        HF_REPO,

    "source_dataset":
        "Amazon Berkeley Objects (ABO)",

    "source_host":
        "Amazon S3",

    "source_model_format":
        "GLB / glTF",

    "selection_method":
        "Metadata-based product and 3D model matching "
        "followed by quality ranking and category-aware "
        "selection.",

    "local_quality_pool_records":
        len(quality_records),

    "downloaded_glb_models":
        TOTAL_DOWNLOADED,

    "batches_processed":
        TOTAL_BATCHES,

    "observed_batch_size":
        BATCH_SIZE_OBSERVED,

    "categories":
        dict(
            categories
        ),

    "created_at":
        CREATED_AT,

    "purpose": [
        "3D object reconstruction",
        "multi-view learning",
        "3D Gaussian Splatting",
        "material decomposition",
        "relighting research",
        "computer vision experiments"
    ],

    "note":
        "This repository contains a selected subset of "
        "3D assets and metadata prepared from the "
        "Amazon Berkeley Objects dataset. It is not "
        "the complete ABO dataset."
}


SUMMARY_PATH = (
    DOCUMENTATION_DIR
    / "dataset_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dataset_summary,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    f"\n✓ Created:\n{SUMMARY_PATH}"
)


# ============================================================
# 5. DATASET CARD
# ============================================================

README = f"""---
pretty_name: Amazon Berkeley Objects 3D Selection for Neural Object Reconstruction
task_categories:
  - image-feature-extraction
  - computer-vision
  - depth-estimation
tags:
  - 3d
  - 3d-reconstruction
  - neural-rendering
  - gaussian-splatting
  - inverse-rendering
  - material-decomposition
  - relighting
  - multi-view
  - gltf
  - glb
  - amazon-berkeley-objects
  - computer-vision
  - neural-rendering
dataset_type:
  - 3D
language:
  - en
---

# Amazon Berkeley Objects 3D Selection

This repository contains a selected collection of 3D product assets prepared for the **Neural Object Reconstruction & Relighting Lab**.

The 3D assets used in this project come from the **Amazon Berkeley Objects (ABO)** dataset and are accessed through Amazon S3. The purpose of this repository is not to redistribute the entire ABO collection, but to document and preserve the subset selected for the reconstruction experiments.

## Project

**Neural Object Reconstruction & Relighting Lab**

The larger project investigates whether casual smartphone video can be converted into a 3D representation that is useful for novel-view rendering, material estimation and interactive relighting.

The planned reconstruction pipeline combines:

- object segmentation
- depth estimation
- camera pose estimation
- 3D Gaussian Splatting
- multi-view feature extraction
- material decomposition
- uncertainty estimation
- differentiable rendering
- relighting

The project specification describes the intended material representation in terms of albedo, roughness, specular properties, surface normals and illumination, together with uncertainty estimates.

## Why ABO?

The original project plan considered several sources of 3D assets. For the implementation documented in this repository, I used the **Amazon Berkeley Objects 3D data instead of Objaverse**.

This was a practical choice because the project needs real product-style objects with usable 3D representations that can be selected, filtered and processed systematically.

The notebook therefore focuses on:

1. inspecting ABO metadata;
2. matching product records with available 3D assets;
3. ranking candidate objects;
4. applying category-aware selection;
5. downloading selected GLB assets;
6. validating the downloaded files;
7. organizing the assets into batches;
8. uploading the selected data and metadata to Hugging Face.

## Dataset contents

The repository contains selected 3D assets in GLB format together with metadata and manifests describing the selected objects.

Depending on the completed batches, the repository may contain:

- `.glb` 3D models
- JSONL manifests
- CSV manifests
- selection metadata
- download records
- validation information
- dataset documentation

The dataset is organized into processing batches rather than treating the whole source collection as one undifferentiated download.

## Current dataset statistics

The final statistics detected from the Colab environment are:

- Quality-pool records: **{len(quality_records):,}**
- Downloaded GLB models detected locally: **{TOTAL_DOWNLOADED:,}**
- Processed batches detected: **{TOTAL_BATCHES:,}**
- Largest observed batch size: **{BATCH_SIZE_OBSERVED:,}**

The exact contents of the Hugging Face repository should be treated as the authoritative record of what was actually uploaded.

## Data selection

The selection process was designed to avoid blindly downloading a very large 3D collection.

The workflow first works with metadata and then selects candidate objects before downloading the actual GLB files.

The selection process includes:

- availability of a 3D asset;
- product-to-model matching;
- metadata quality;
- object category;
- suitability for the intended computer-vision experiments;
- basic GLB validation.

This makes the dataset a **research subset**, rather than a mirror of the complete ABO dataset.

## File validation

Downloaded GLB files are checked before they are uploaded.

The validation stage checks that:

- the expected file exists;
- the file is not empty;
- the file has a reasonable minimum size;
- the GLB header is valid.

A file that fails validation should not be uploaded as part of the usable dataset.

## Intended use

The selected assets are intended for research and development involving:

- 3D object reconstruction;
- multi-view computer vision;
- neural rendering;
- 3D Gaussian Splatting;
- geometry-aware learning;
- material prediction;
- inverse rendering;
- relighting;
- synthetic data generation;
- computer graphics experiments.

The dataset can also be used as an object source for generating additional rendered training data.

For example, a selected object can later be rendered from multiple camera positions with controlled lighting and material settings.

## Relationship to the larger project

The 3D assets are only one component of the overall research pipeline.

The larger project separates the problem into several stages:

**Video → segmentation → depth → camera poses → 3D reconstruction → multi-view material inference → uncertainty → rendering → relighting**

The 3D asset collection is therefore mainly useful for the object and synthetic-data side of the research.

The complete project also considers real-world material/lighting benchmarks and a smartphone-video evaluation dataset.

## Training data and ground truth

The selected ABO models are not, by themselves, a complete supervised material dataset.

The intended next step is to use the 3D objects to generate controlled renders containing information such as:

- RGB images
- depth
- surface normals
- segmentation masks
- albedo
- roughness
- specular properties
- illumination
- camera parameters

This provides a way to generate controlled supervision for later material-decomposition experiments.

## Dataset splits

When these objects are used for machine-learning experiments, the split should be made at the **object level**, not by randomly distributing views of the same object between training and testing.

This is important because multiple views of one object are highly correlated.

A suitable experimental split is:

- 70% objects — training
- 15% objects — validation
- 15% objects — testing

The test objects should remain unseen during training.

## Important limitations

This dataset should not be interpreted as ground truth for physically perfect material reconstruction.

3D assets may contain differences in:

- geometry quality;
- topology;
- textures;
- material definitions;
- scale;
- coordinate systems;
- lighting assumptions;
- mesh completeness.

Consequently, additional preprocessing may be required before using the assets for physically based rendering or supervised material learning.

The larger project also recognizes that reflective and transparent objects are substantially harder for material decomposition and relighting.

## Provenance

The 3D assets originate from the **Amazon Berkeley Objects (ABO)** dataset.

This repository contains a selected subset prepared for the Neural Object Reconstruction & Relighting Lab.

The selection and processing code was developed in Google Colab and uses Amazon S3 as the source for the 3D model files.

The original dataset's terms, attribution requirements and licensing conditions remain applicable to the source data.

Users should check the original ABO documentation before redistributing or using individual assets outside the intended research context.

## Source

Amazon Berkeley Objects:

https://amazon-berkeley-objects.s3.amazonaws.com/

The source dataset should be cited according to the official ABO publication and dataset documentation.

## Processing environment

The data preparation workflow was developed using:

- Python
- Google Colab
- pandas
- NumPy
- requests
- Hugging Face Hub
- Amazon S3
- JSON/JSONL metadata
- CSV metadata
- GLB/glTF 3D assets

## Reproducibility

The processing notebook records the main selection and download stages.

The workflow is intentionally batch-based so that a large collection can be processed without requiring the complete 3D source archive to be stored locally at once.

Each batch can be:

1. selected;
2. downloaded;
3. validated;
4. uploaded;
5. remotely verified;
6. removed from local storage after verification.

## What this dataset is not

This repository is **not**:

- the complete Amazon Berkeley Objects dataset;
- a newly created collection of 3D models;
- a claim that every source model is suitable for training;
- a complete ground-truth material dataset;
- a replacement for real-world relighting benchmarks.

It is a curated research subset prepared for a larger 3D reconstruction and relighting project.

## Related research

The larger project investigates a material-aware extension of 3D Gaussian Splatting.

The intended representation includes:

- position;
- covariance;
- opacity;
- learned material features;
- uncertainty.

The material features include estimates related to:

- albedo;
- roughness;
- specular properties;
- surface normals;
- illumination.

The purpose is to move beyond RGB-only appearance reconstruction toward a representation that can be used for controlled relighting.

## Current status

**Dataset preparation status:** completed for the batches uploaded to this repository.

**Research status:** the dataset is one component of the broader Neural Object Reconstruction & Relighting Lab.

The repository should therefore be viewed as a research data resource rather than a finished commercial dataset.

## Acknowledgements

This work builds on the Amazon Berkeley Objects dataset and the broader research community working on 3D reconstruction, neural rendering, inverse rendering and computer graphics.

## Citation

If you use this processed subset, please cite both this repository and the original Amazon Berkeley Objects work.

Created: {CREATED_AT}
"""


README_PATH = (
    DOCUMENTATION_DIR
    / "README.md"
)

with open(
    README_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        README
    )


print(
    f"✓ Created:\n{README_PATH}"
)


# ============================================================
# 6. DATASET INFORMATION DOCUMENT
# ============================================================

DATASET_INFO = f"""# Dataset Information

## Name

{DATASET_TITLE}

## Project

{PROJECT_TITLE}

## Source

Amazon Berkeley Objects (ABO), accessed through Amazon S3.

## Purpose

The dataset is intended to provide a controlled collection of 3D product assets for experiments related to neural object reconstruction and relighting.

## Processing

The notebook performs the following operations:

1. Download and inspect ABO metadata.
2. Identify available 3D models.
3. Match product records to 3D assets.
4. Build a candidate object pool.
5. Rank candidate objects according to available metadata.
6. Apply category-aware selection.
7. Build batches.
8. Download GLB files from Amazon S3.
9. Validate the GLB files.
10. Upload verified batches to Hugging Face.
11. Verify the remote files.

## Statistics

Quality pool records: {len(quality_records):,}

Downloaded GLB models detected locally: {TOTAL_DOWNLOADED:,}

Processed batches detected: {TOTAL_BATCHES:,}

Created: {CREATED_AT}

## Expected downstream use

The selected objects can be used to create rendered multi-view training data.

Potential supervision includes:

- RGB
- depth
- normals
- segmentation masks
- albedo
- roughness
- specular
- illumination
- camera parameters

## Important distinction

This repository contains selected source 3D assets.

It is not itself a complete material-decomposition training dataset.

A later rendering stage can turn these assets into a supervised dataset suitable for material learning.

## Object-level splitting

Machine-learning experiments should split by object identity rather than by individual rendered view.

This prevents different views of the same object from leaking between training and evaluation.

## Limitations

The source assets can vary in:

- geometry;
- topology;
- textures;
- material definitions;
- scale;
- completeness;
- quality.

Additional preprocessing may therefore be necessary.

## Source attribution

The original 3D assets come from the Amazon Berkeley Objects dataset.

Please consult the original ABO documentation for the appropriate attribution and licensing requirements before redistribution.

"""


DATASET_INFO_PATH = (
    DOCUMENTATION_DIR
    / "DATASET_INFO.md"
)

with open(
    DATASET_INFO_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        DATASET_INFO
    )


# ============================================================
# 7. PROVENANCE / LICENSE NOTE
# ============================================================

LICENSE_NOTE = """# Provenance and Licensing

The 3D assets in this repository originate from the Amazon Berkeley Objects (ABO) dataset.

This repository contains a selected and processed subset prepared for research experiments.

The repository owner does not claim ownership of the original 3D assets.

Before using, redistributing, modifying or incorporating individual assets into another dataset or product, consult the original Amazon Berkeley Objects documentation and applicable licenses.

The processing code in the associated notebook is separate from the original source data.

Source:

Amazon Berkeley Objects
Amazon S3:
https://amazon-berkeley-objects.s3.amazonaws.com/

The original dataset and its contributors should receive appropriate attribution according to their published terms.
"""


LICENSE_PATH = (
    DOCUMENTATION_DIR
    / "LICENSE_NOTE.md"
)

with open(
    LICENSE_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        LICENSE_NOTE
    )


# ============================================================
# 8. CITATION.CFF
# ============================================================

CITATION = f"""cff-version: 1.2.0
title: "Amazon Berkeley Objects 3D Selection for Neural Object Reconstruction"
message: "If you use this processed subset, please cite this repository and the original Amazon Berkeley Objects dataset."
type: dataset
authors:
  - family-names: "Ndeda"
    given-names: "Jeremy"
repository-code: "https://huggingface.co/datasets/{HF_REPO}"
repository: "https://huggingface.co/datasets/{HF_REPO}"
abstract: >
  A selected and quality-filtered subset of Amazon Berkeley Objects
  3D product assets prepared for experiments in neural object
  reconstruction, multi-view learning, neural rendering,
  material decomposition and relighting.
keywords:
  - 3D reconstruction
  - neural rendering
  - 3D Gaussian Splatting
  - inverse rendering
  - material decomposition
  - relighting
  - computer vision
"""


CITATION_PATH = (
    DOCUMENTATION_DIR
    / "CITATION.cff"
)

with open(
    CITATION_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        CITATION
    )


# ============================================================
# 9. MACHINE-READABLE METADATA
# ============================================================

HF_METADATA = {

    "dataset_name":
        DATASET_TITLE,

    "project_name":
        PROJECT_TITLE,

    "source":
        "Amazon Berkeley Objects",

    "source_location":
        "Amazon S3",

    "huggingface_repository":
        HF_REPO,

    "format":
        ["GLB", "JSONL", "CSV"],

    "task":
        [
            "3D reconstruction",
            "multi-view learning",
            "neural rendering",
            "material decomposition",
            "relighting"
        ],

    "selection":
        {
            "quality_pool_records":
                len(quality_records),

            "downloaded_models":
                TOTAL_DOWNLOADED,

            "batches":
                TOTAL_BATCHES
        },

    "categories":
        dict(categories),

    "created_at":
        CREATED_AT
}


HF_METADATA_PATH = (
    DOCUMENTATION_DIR
    / "metadata.json"
)

with open(
    HF_METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        HF_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 10. DATASET MANIFEST INDEX
# ============================================================

manifest_index = []

for manifest in manifest_files:

    batch_name = (
        manifest.parent.name
    )

    try:

        with open(
            manifest,
            "r",
            encoding="utf-8"
        ) as f:

            count = sum(
                1
                for line in f
                if line.strip()
            )

    except Exception:

        count = 0

    manifest_index.append({

        "batch":
            batch_name,

        "manifest":
            manifest.name,

        "records":
            count
    })


MANIFEST_INDEX_PATH = (
    DOCUMENTATION_DIR
    / "manifest_index.json"
)

with open(
    MANIFEST_INDEX_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest_index,
        f,
        indent=2
    )


# ============================================================
# 11. CREATE A SIMPLE PROJECT CHANGELOG
# ============================================================

CHANGELOG = f"""# Dataset Preparation Changelog

## Final documented version

Date: {CREATED_AT}

### Data source

Switched the project 3D object source from the originally planned Objaverse workflow to the Amazon Berkeley Objects (ABO) 3D data hosted on Amazon S3.

### Selection

The notebook:

- inspected ABO 3D metadata;
- matched product records to 3D assets;
- created a candidate pool;
- ranked candidate objects;
- applied category-aware selection;
- created fixed-size batches.

### Download

Selected GLB assets were downloaded individually from the ABO S3 3D model endpoint.

### Validation

Downloaded models were checked for:

- existence;
- non-zero size;
- valid GLB header.

### Storage

Verified batches were uploaded to:

`{HF_REPO}`

### Purpose

The selected data will support experiments in:

- multi-view reconstruction;
- 3D Gaussian Splatting;
- neural rendering;
- material decomposition;
- uncertainty-aware reconstruction;
- relighting.

### Note

This repository represents a selected research subset and not the complete ABO dataset.
"""


CHANGELOG_PATH = (
    DOCUMENTATION_DIR
    / "CHANGELOG.md"
)

with open(
    CHANGELOG_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        CHANGELOG
    )


# ============================================================
# 12. COPY DOCUMENTATION TO PROJECT ROOT
# ============================================================

# Keeping a copy in the project root makes the final Colab
# output easier to inspect before uploading.

for source_file in DOCUMENTATION_DIR.iterdir():

    if source_file.is_file():

        destination = (
            PROJECT_ROOT
            / source_file.name
        )

        destination.write_text(
            source_file.read_text(
                encoding="utf-8"
            ),
            encoding="utf-8"
        )


# ============================================================
# 13. DISPLAY FINAL SUMMARY
# ============================================================

print()
print("=" * 72)
print("DOCUMENTATION GENERATED")
print("=" * 72)

print(
    f"\nDocumentation directory:\n"
    f"  {DOCUMENTATION_DIR}"
)

print("\nFiles created:")

for file in sorted(
    DOCUMENTATION_DIR.iterdir()
):

    if file.is_file():

        print(
            f"  ✓ {file.name}"
        )


print()
print("DATASET SUMMARY")
print("-" * 72)

print(
    f"Source dataset:       Amazon Berkeley Objects"
)

print(
    f"3D source:            Amazon S3"
)

print(
    f"Quality pool:         {len(quality_records):,}"
)

print(
    f"Downloaded GLBs:      {TOTAL_DOWNLOADED:,}"
)

print(
    f"Processed batches:    {TOTAL_BATCHES:,}"
)

print(
    f"Hugging Face repo:    {HF_REPO}"
)


# ============================================================
# 14. UPLOAD DOCUMENTATION TO HUGGING FACE
# ============================================================

print()
print("=" * 72)
print("UPLOADING DOCUMENTATION TO HUGGING FACE")
print("=" * 72)

api = HfApi()

documentation_files = [

    README_PATH,

    DATASET_INFO_PATH,

    LICENSE_PATH,

    CITATION_PATH,

    HF_METADATA_PATH,

    SUMMARY_PATH,

    MANIFEST_INDEX_PATH,

    CHANGELOG_PATH
]


for file_path in documentation_files:

    if not file_path.exists():

        print(
            f"✗ Missing: {file_path.name}"
        )

        continue

    print(
        f"Uploading: {file_path.name}"
    )

    api.upload_file(
        path_or_fileobj=str(
            file_path
        ),
        path_in_repo=file_path.name,
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE
    )

    print(
        f"  ✓ Uploaded"
    )


# ============================================================
# 15. UPLOAD THE NOTEBOOK ITSELF
# ============================================================

NOTEBOOK_CANDIDATES = [

    PROJECT_ROOT / "01_aws_selection.ipynb",

    Path(
        "/content/01_aws_selection.ipynb"
    )
]


NOTEBOOK_PATH = None

for candidate in NOTEBOOK_CANDIDATES:

    if candidate.exists():

        NOTEBOOK_PATH = candidate
        break


if NOTEBOOK_PATH is not None:

    print()
    print(
        f"Uploading processing notebook:"
    )

    print(
        f"  {NOTEBOOK_PATH}"
    )

    api.upload_file(
        path_or_fileobj=str(
            NOTEBOOK_PATH
        ),
        path_in_repo="notebooks/01_aws_selection.ipynb",
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE
    )

    print(
        "  ✓ Notebook uploaded"
    )

else:

    print(
        "\n⚠ Processing notebook was not found "
        "at the expected paths."
    )


# ============================================================
# 16. VERIFY HUGGING FACE REPOSITORY
# ============================================================

print()
print("=" * 72)
print("VERIFYING HUGGING FACE REPOSITORY")
print("=" * 72)

try:

    repo_files = api.list_repo_files(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE
    )

    required_files = [

        "README.md",
        "DATASET_INFO.md",
        "LICENSE_NOTE.md",
        "CITATION.cff",
        "metadata.json",
        "dataset_summary.json",
        "manifest_index.json",
        "CHANGELOG.md"
    ]

    print(
        f"\nRemote files: "
        f"{len(repo_files):,}"
    )

    print(
        "\nRequired documentation:"
    )

    missing = []

    for required in required_files:

        if required in repo_files:

            print(
                f"  ✓ {required}"
            )

        else:

            print(
                f"  ✗ {required}"
            )

            missing.append(
                required
            )

    if missing:

        print()
        print(
            "WARNING: Some documentation files "
            "were not found remotely:"
        )

        for item in missing:

            print(
                f"  - {item}"
            )

    else:

        print()
        print(
            "✓ All required documentation files "
            "are present."
        )


except Exception as e:

    print(
        "\nCould not verify remote repository."
    )

    print(
        f"Reason: {e}"
    )


# ============================================================
# 17. FINAL MESSAGE
# ============================================================

print()
print("=" * 72)
print("FINAL DATASET DOCUMENTATION COMPLETE")
print("=" * 72)

print()
print(
    "The Hugging Face repository now contains:"
)

print(
    "  • Dataset card"
)

print(
    "  • Dataset information"
)

print(
    "  • Provenance/licensing note"
)

print(
    "  • Citation metadata"
)

print(
    "  • Machine-readable metadata"
)

print(
    "  • Dataset summary"
)

print(
    "  • Batch manifest index"
)

print(
    "  • Changelog"
)

print(
    "  • Processing notebook"
)

print()
print(
    f"Hugging Face dataset:"
)

print(
    f"https://huggingface.co/datasets/{HF_REPO}"
)

print()
print(
    "This repository documents the actual ABO-based "
    "dataset preparation workflow rather than the "
    "original Objaverse-based project proposal."
)

FINAL DATASET DOCUMENTATION

Project root:
/content/neural_object_reconstruction

Hugging Face repository:
ndeda/neural-object-reconstruction-abo

Quality pool records: 250
Downloaded GLB models found locally: 0
Batch manifests: 0
Download manifests: 0

✓ Created:
/content/neural_object_reconstruction/huggingface_documentation/dataset_summary.json
✓ Created:
/content/neural_object_reconstruction/huggingface_documentation/README.md

DOCUMENTATION GENERATED

Documentation directory:
  /content/neural_object_reconstruction/huggingface_documentation

Files created:
  ✓ CHANGELOG.md
  ✓ CITATION.cff
  ✓ DATASET_INFO.md
  ✓ LICENSE_NOTE.md
  ✓ README.md
  ✓ dataset_summary.json
  ✓ manifest_index.json
  ✓ metadata.json

DATASET SUMMARY
------------------------------------------------------------------------
Source dataset:       Amazon Berkeley Objects
3D source:            Amazon S3
Quality pool:         250
Downloaded GLBs:      0
Processed batches:    0
Hugging Face repo:    ndeda/neural-

/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:11610: UserWarning: Warnings while validating metadata in README.md:
- The task_categories "computer-vision" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, text-to-video, image-text-to-text, image-text-to-image, image-t

  ✓ Uploaded
Uploading: DATASET_INFO.md
  ✓ Uploaded
Uploading: LICENSE_NOTE.md
  ✓ Uploaded
Uploading: CITATION.cff
  ✓ Uploaded
Uploading: metadata.json
  ✓ Uploaded
Uploading: dataset_summary.json
  ✓ Uploaded
Uploading: manifest_index.json
  ✓ Uploaded
Uploading: CHANGELOG.md
  ✓ Uploaded

⚠ Processing notebook was not found at the expected paths.

VERIFYING HUGGING FACE REPOSITORY

Remote files: 301

Required documentation:
  ✓ README.md
  ✓ DATASET_INFO.md
  ✓ LICENSE_NOTE.md
  ✓ CITATION.cff
  ✓ metadata.json
  ✓ dataset_summary.json
  ✓ manifest_index.json
  ✓ CHANGELOG.md

✓ All required documentation files are present.

FINAL DATASET DOCUMENTATION COMPLETE

The Hugging Face repository now contains:
  • Dataset card
  • Dataset information
  • Provenance/licensing note
  • Citation metadata
  • Machine-readable metadata
  • Dataset summary
  • Batch manifest index
  • Changelog
  • Processing notebook

Hugging Face dataset:
https://huggingface.co/datasets/ndeda/neural-object-r

In [55]:
# ============================================================
# RESEARCH PROJECT DOCUMENTATION
# Neural Object Reconstruction & Relighting Lab
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import os

PROJECT_NAME = (
    "Neural Object Reconstruction & Relighting Lab"
)

PROJECT_SUBTITLE = (
    "Uncertainty-Aware Multi-View Material Decomposition "
    "for Relightable 3D Objects"
)

PROJECT_ROOT = Path(
    "/content/neural_object_reconstruction"
)

DOCS_DIR = PROJECT_ROOT / "research_documentation"

DOCS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HF_REPO = "ndeda/neural-object-reconstruction-abo"
HF_REPO_TYPE = "dataset"

CREATED_AT = datetime.now(
    timezone.utc
).strftime("%Y-%m-%d %H:%M:%S UTC")

print("=" * 72)
print(PROJECT_NAME)
print("=" * 72)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Hugging Face repository: {HF_REPO}")
print(f"Documentation directory: {DOCS_DIR}")
print(f"Generated: {CREATED_AT}")

Neural Object Reconstruction & Relighting Lab

Project root: /content/neural_object_reconstruction
Hugging Face repository: ndeda/neural-object-reconstruction-abo
Documentation directory: /content/neural_object_reconstruction/research_documentation
Generated: 2026-08-14 07:20:42 UTC


In [56]:
# ============================================================
# RESEARCH PROJECT OVERVIEW
# ============================================================

PROJECT_OVERVIEW = r"""
# Neural Object Reconstruction & Relighting Lab

## Uncertainty-Aware Multi-View Material Decomposition for Relightable 3D Objects

### Overview

The Neural Object Reconstruction & Relighting Lab is a research project
investigating how casually captured multi-view imagery can be converted
into detailed, material-aware and uncertainty-aware 3D object
representations.

The project is motivated by a limitation in conventional 3D
reconstruction systems.

A model may reproduce the appearance of an object from the viewpoints
seen during capture without actually learning a representation that can
be reliably manipulated under new lighting.

This project therefore investigates a different formulation.

Instead of treating appearance as a single RGB signal, the system aims
to estimate a representation containing information related to:

- geometry
- surface appearance
- material properties
- illumination
- camera configuration
- uncertainty

The long-term objective is to reconstruct an object from a short
smartphone video and produce a representation that can be rendered from
new viewpoints and under previously unseen lighting conditions.

---

## Research Problem

The appearance observed in an image is influenced by multiple factors.

These include:

- object geometry
- surface reflectance
- illumination
- camera position
- occlusion
- sensor noise
- image quality

These factors are entangled in ordinary RGB observations.

Consequently, a system trained only to reproduce RGB images may produce
excellent novel-view rendering while still failing to recover
physically meaningful material properties.

The research problem is therefore:

> Can multi-view observations, geometric information, explicit material
> prediction and uncertainty estimation be combined within a neural 3D
> representation to produce more controllable and reliable object
> relighting?

---

## Research Objective

The primary objective is to develop and evaluate a pipeline capable of
transforming casual multi-view object imagery into a material-aware
3D representation.

The intended pipeline is:

Smartphone Video
→ Object Segmentation
→ Depth Estimation
→ Camera Pose Estimation
→ 3D Reconstruction
→ Multi-View Feature Learning
→ Material Decomposition
→ Uncertainty Estimation
→ Material-Aware 3D Representation
→ Differentiable Rendering
→ Relighting

The project is research-oriented. Individual components will be
implemented and evaluated independently before being combined into the
complete system.

---

## Research Hypothesis

The central hypothesis is:

> Explicitly modeling material properties and their uncertainty within
> a multi-view 3D representation can provide more reliable and
> controllable relighting than conventional RGB-only 3D reconstruction.

The project will test this hypothesis through controlled ablation
experiments and comparisons against progressively stronger baselines.

---

## Research Scope

The research covers:

- multi-view object reconstruction
- neural rendering
- 3D Gaussian Splatting
- depth estimation
- object segmentation
- camera pose estimation
- material decomposition
- uncertainty estimation
- differentiable rendering
- relighting
- synthetic data generation
- real-world smartphone capture

The project does not assume that perfect physical material recovery is
possible from casual video.

The objective is to estimate a useful and interpretable approximation
that improves controllable rendering.
"""

overview_path = DOCS_DIR / "PROJECT_OVERVIEW.md"

overview_path.write_text(
    PROJECT_OVERVIEW,
    encoding="utf-8"
)

print(f"✓ Created {overview_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/PROJECT_OVERVIEW.md


In [57]:
# ============================================================
# RESEARCH QUESTIONS AND HYPOTHESES
# ============================================================

RESEARCH_QUESTIONS = r"""
# Research Questions

## RQ1 — Multi-View Information

Does combining multiple observations of an object improve material
estimation compared with single-view prediction?

### Hypothesis

Multi-view observations should provide stronger constraints on
geometry, surface appearance and material properties than isolated
images.

---

## RQ2 — Geometric Conditioning

Does incorporating depth and geometric information improve material
decomposition?

### Hypothesis

Providing depth and surface geometry should reduce ambiguities between
object structure, appearance and illumination.

---

## RQ3 — Material-Aware 3D Representation

Can material information stored within a 3D Gaussian representation
produce better relighting than RGB-only Gaussian representations?

### Hypothesis

Separating material properties from illumination should allow the
renderer to modify lighting while preserving the object's underlying
appearance properties.

---

## RQ4 — Uncertainty

Can predicted uncertainty identify regions where material estimates
are unreliable?

### Hypothesis

High uncertainty should correlate with reconstruction errors and
difficult regions such as reflective, transparent, occluded or poorly
observed surfaces.

---

## RQ5 — Segmentation Quality

Does improved object segmentation improve downstream reconstruction
and material estimation?

### Hypothesis

Cleaner object boundaries should reduce background contamination and
improve the quality of geometry and appearance estimation.

---

## RQ6 — Synthetic-to-Real Transfer

Can controlled synthetic training data generated from 3D objects improve
performance on real smartphone video?

### Hypothesis

Synthetic data can provide useful supervision for properties such as
depth, normals, albedo and roughness, while real-world adaptation is
required to handle the domain gap.

---

# Primary Research Hypothesis

The overall hypothesis is that a material-aware and uncertainty-aware
multi-view representation can produce more controllable relighting
than an RGB-only representation while maintaining competitive
novel-view rendering quality.
"""

rq_path = DOCS_DIR / "RESEARCH_QUESTIONS.md"

rq_path.write_text(
    RESEARCH_QUESTIONS,
    encoding="utf-8"
)

print(f"✓ Created {rq_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/RESEARCH_QUESTIONS.md


In [58]:
# ============================================================
# RESEARCH METHODOLOGY
# ============================================================

METHODOLOGY = r"""
# Methodology

## 1. 3D Data Preparation

The project uses the Amazon Berkeley Objects (ABO) 3D data as the
primary source of object assets.

The implementation accesses the 3D assets through Amazon S3.

The project does not attempt to reproduce the entire source dataset.

Instead, a research subset is constructed through:

1. metadata inspection;
2. product-to-3D model matching;
3. candidate generation;
4. quality filtering;
5. category-aware selection;
6. batch construction;
7. GLB downloading;
8. file validation.

The selected objects are intended primarily for controlled synthetic
data generation and reconstruction experiments.

---

## 2. Synthetic Data Generation

Selected 3D objects can be rendered from multiple camera viewpoints.

Rendering parameters can be varied to generate controlled observations.

Potential variables include:

- camera position;
- camera orientation;
- focal length;
- illumination;
- material parameters;
- background;
- object orientation.

The rendering process can produce synchronized ground-truth channels:

- RGB;
- depth;
- normals;
- segmentation;
- albedo;
- roughness;
- specular properties;
- illumination;
- camera parameters.

This provides supervision that is difficult to obtain from ordinary
real-world video.

---

## 3. Real-World Input

The target real-world input is a short smartphone video of an object.

The video is converted into a sequence of frames.

The frames are then processed through the perception pipeline.

---

## 4. Object Segmentation

The segmentation stage isolates the target object from its background.

The project investigates combining general-purpose segmentation with
high-resolution foreground refinement.

The output includes:

- object mask;
- boundary information;
- confidence.

Segmentation quality is evaluated independently and through its effect
on downstream reconstruction.

---

## 5. Depth Estimation

Depth is estimated from RGB observations.

Depth provides geometric constraints for subsequent reconstruction.

The project initially uses pretrained depth estimation models and may
later investigate adaptation using synthetic data.

---

## 6. Camera Pose Estimation

The relative camera motion is estimated from the captured sequence.

The initial baseline uses structure-from-motion techniques.

The estimated camera parameters provide the coordinate system required
for multi-view reconstruction.

---

## 7. 3D Reconstruction

The baseline 3D representation is 3D Gaussian Splatting.

The reconstructed object is represented using Gaussian primitives
instead of requiring an explicit textured mesh as the primary
representation.

The baseline contains parameters related to:

- position;
- covariance;
- rotation;
- opacity;
- appearance.

---

## 8. Material Decomposition

The proposed extension adds material-related information.

The model estimates quantities such as:

- albedo;
- roughness;
- specular response;
- normals;
- illumination.

The objective is to separate properties associated with the object
from properties associated with the environment.

---

## 9. Uncertainty Estimation

The system estimates confidence associated with material predictions.

This is particularly important for:

- reflective surfaces;
- transparent objects;
- dark surfaces;
- occluded regions;
- unseen regions;
- motion-blurred frames.

The uncertainty representation is evaluated against reconstruction
errors.

---

## 10. Differentiable Rendering

The predicted representation is rendered using a differentiable
renderer.

The renderer provides the connection between the learned material
representation and image-space observations.

The rendering loss can be used to optimize the representation against
observed images.

---

## 11. Relighting

The final rendering stage modifies illumination while keeping the
estimated object representation fixed.

The objective is to evaluate whether the reconstructed material
representation supports plausible appearance under lighting conditions
not present during capture.
"""

method_path = DOCS_DIR / "METHODOLOGY.md"

method_path.write_text(
    METHODOLOGY,
    encoding="utf-8"
)

print(f"✓ Created {method_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/METHODOLOGY.md


In [59]:
# ============================================================
# EXPERIMENTAL DESIGN
# ============================================================

EXPERIMENTS = r"""
# Experimental Design

The project uses controlled experiments to determine which components
actually contribute to reconstruction and relighting performance.

The experiments are organized as baselines, ablations and the proposed
system.

---

## Experiment A — RGB Gaussian Baseline

A conventional RGB-based 3D Gaussian Splatting system is used as the
primary reconstruction baseline.

This establishes the performance of appearance-only reconstruction.

---

## Experiment B — RGB + Depth

Depth information is incorporated into the reconstruction pipeline.

The purpose is to determine whether explicit geometric information
improves reconstruction.

---

## Experiment C — Single-View Material Prediction

Material properties are predicted using individual observations.

This provides a baseline for evaluating the value of multi-view
information.

---

## Experiment D — Multi-View Material Prediction

Multiple observations are processed jointly.

The purpose is to determine whether multi-view consistency improves
material estimation.

---

## Experiment E — Material-Aware Gaussian Representation

Material features are incorporated into the Gaussian representation.

The representation can include:

- albedo;
- roughness;
- specular properties;
- normals;
- illumination.

---

## Experiment F — Uncertainty-Aware Representation

The material-aware representation is extended with uncertainty.

The objective is to determine whether uncertainty improves robustness
and provides meaningful information about difficult regions.

---

## Experiment G — Full Proposed System

The complete system combines:

- segmentation;
- depth;
- camera estimation;
- multi-view reconstruction;
- material prediction;
- uncertainty;
- material-aware Gaussian representation;
- differentiable rendering;
- relighting.

---

# Ablation Strategy

Components are removed individually from the complete system.

Examples include:

- without depth;
- without multi-view features;
- without uncertainty;
- without material features;
- without segmentation refinement.

The purpose is to measure the individual contribution of each
component.

---

# Data Split

Splits are performed at the object level.

A proposed split is:

- 70% training;
- 15% validation;
- 15% testing.

Different views of the same object must remain in the same split.

This prevents view-level leakage and produces a more meaningful
generalization measurement.

---

# Synthetic-to-Real Evaluation

Synthetic data is used for controlled supervision.

Real smartphone captures are reserved for evaluating generalization.

This provides two complementary evaluation settings:

### Controlled evaluation

Known geometry and material parameters.

### Real-world evaluation

Unknown geometry, materials and illumination.
"""

experiment_path = DOCS_DIR / "EXPERIMENTAL_DESIGN.md"

experiment_path.write_text(
    EXPERIMENTS,
    encoding="utf-8"
)

print(f"✓ Created {experiment_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/EXPERIMENTAL_DESIGN.md


In [60]:
# ============================================================
# EVALUATION PROTOCOL
# ============================================================

EVALUATION = r"""
# Evaluation Protocol

The project does not rely on a single image-quality metric.

Evaluation is divided into several research dimensions.

---

## 1. Reconstruction Quality

Evaluation includes:

- novel-view rendering;
- multi-view consistency;
- geometric consistency;
- image reconstruction quality.

Possible image metrics include:

- PSNR;
- SSIM;
- LPIPS.

---

## 2. Segmentation Quality

Segmentation is evaluated using:

- IoU;
- Dice;
- boundary F-score.

The downstream effect of segmentation quality on reconstruction is
also measured.

---

## 3. Depth Quality

Where ground truth is available, depth is evaluated using:

- absolute relative error;
- RMSE;
- scale-invariant error;
- threshold accuracy.

---

## 4. Material Quality

Synthetic data provides direct ground truth for material experiments.

Potential metrics include:

### Albedo

- MAE;
- RMSE;
- perceptual similarity.

### Roughness

- MAE;
- RMSE.

### Normals

- angular error.

### Specular properties

- parameter error;
- rendered appearance error.

---

## 5. Relighting Quality

Relighting is evaluated by comparing rendered results under novel
lighting against ground-truth renders where available.

Evaluation focuses on:

- appearance consistency;
- material preservation;
- illumination response;
- perceptual similarity.

---

## 6. Uncertainty Evaluation

Uncertainty is evaluated by examining whether confidence correlates
with actual prediction quality.

Useful measurements include:

- calibration;
- error versus uncertainty;
- confidence intervals;
- uncertainty ranking;
- high-error/high-uncertainty overlap.

---

## 7. Real-World Evaluation

Real-world experiments evaluate:

- reconstruction stability;
- camera-motion robustness;
- incomplete observation;
- reflective surfaces;
- transparent surfaces;
- difficult lighting;
- background variation.

The objective is to determine how well synthetic supervision transfers
to real smartphone video.
"""

evaluation_path = DOCS_DIR / "EVALUATION_PROTOCOL.md"

evaluation_path.write_text(
    EVALUATION,
    encoding="utf-8"
)

print(f"✓ Created {evaluation_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/EVALUATION_PROTOCOL.md


In [61]:
# ============================================================
# LIMITATIONS AND RESEARCH RISKS
# ============================================================

LIMITATIONS = r"""
# Limitations and Research Risks

## Material Ambiguity

Material and illumination are inherently difficult to separate from
ordinary RGB observations.

A visually plausible prediction does not necessarily represent the
true physical material.

---

## Reflective Objects

Highly reflective surfaces may reproduce the environment rather than
their own intrinsic appearance.

These objects are expected to produce high uncertainty.

---

## Transparent Objects

Transparent and translucent materials are particularly difficult
because their appearance depends on the environment and geometry
behind the surface.

---

## Incomplete Observation

A smartphone video may not observe every surface of an object.

Unseen regions cannot be reliably reconstructed from observations that
do not contain information about them.

---

## Depth Errors

Pretrained monocular depth models can contain:

- scale errors;
- geometric distortions;
- boundary artifacts.

These errors can propagate into reconstruction.

---

## Synthetic-to-Real Domain Gap

Synthetic renders do not perfectly reproduce real photographs.

Differences can occur in:

- sensor characteristics;
- noise;
- exposure;
- motion blur;
- material appearance;
- illumination;
- camera response.

Synthetic training therefore cannot be assumed to transfer directly to
real-world captures.

---

## Dataset Bias

ABO is primarily a product-oriented 3D object collection.

The resulting research dataset may therefore be biased toward certain
types of objects and appearances.

---

## Evaluation Leakage

Randomly splitting views rather than objects can result in severe
evaluation leakage.

Object-level splitting is therefore required.

---

## Research Interpretation

Results should be interpreted as evidence for or against specific
research hypotheses.

The project does not claim universal physical material recovery.
"""
limitations_path = DOCS_DIR / "LIMITATIONS.md"

limitations_path.write_text(
    LIMITATIONS,
    encoding="utf-8"
)

print(f"✓ Created {limitations_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/LIMITATIONS.md


In [62]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

REPRODUCIBILITY = r"""
# Reproducibility

The project is designed so that data preparation and experiments can
be reproduced from documented inputs and processing configurations.

## Dataset Provenance

The 3D object source is:

Amazon Berkeley Objects (ABO)

The 3D assets are accessed through Amazon S3.

---

## Data Processing

The data preparation pipeline records:

- source metadata;
- candidate objects;
- selected objects;
- batch identifiers;
- downloaded files;
- validation results;
- manifests.

---

## Synthetic Data

Each generated training sample should record:

- object identifier;
- camera parameters;
- lighting parameters;
- material parameters;
- render configuration;
- random seed;
- output channels.

---

## Model Experiments

Each experiment should record:

- model configuration;
- dataset split;
- random seed;
- optimizer;
- learning rate;
- batch size;
- number of epochs/iterations;
- checkpoint;
- evaluation metrics.

---

## Environment

The current research workflow uses:

- Python;
- Google Colab;
- TensorFlow-compatible deep-learning components where applicable;
- OpenCV;
- Hugging Face Hub;
- Amazon S3;
- GLB/glTF;
- computer-vision and 3D reconstruction tools.

---

## Reproducibility Principle

A result should not be considered fully reproducible unless another
researcher can determine:

1. which data was used;
2. which preprocessing was performed;
3. which model configuration was used;
4. which training configuration was used;
5. which evaluation split was used;
6. which checkpoint produced the reported result.
"""

repro_path = DOCS_DIR / "REPRODUCIBILITY.md"

repro_path.write_text(
    REPRODUCIBILITY,
    encoding="utf-8"
)

print(f"✓ Created {repro_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/REPRODUCIBILITY.md


In [63]:
# ============================================================
# RESEARCH ROADMAP
# ============================================================

ROADMAP = r"""
# Research Roadmap

## Phase 1 — 3D Dataset Preparation

Status: Active / Completed for selected batches

- obtain ABO metadata;
- identify available 3D models;
- match product records;
- rank candidate objects;
- perform category-aware selection;
- download GLB assets;
- validate assets;
- publish selected data.

---

## Phase 2 — Synthetic Rendering

Status: Planned

- establish rendering pipeline;
- generate multi-view images;
- generate depth;
- generate normals;
- generate segmentation;
- generate albedo;
- generate roughness;
- generate specular information;
- generate lighting metadata;
- create object-level splits.

---

## Phase 3 — Baseline Reconstruction

Status: Planned

- establish 3D Gaussian Splatting baseline;
- integrate depth;
- evaluate novel-view synthesis;
- establish reproducible benchmarks.

---

## Phase 4 — Material Prediction

Status: Planned

- implement multi-view feature extraction;
- predict albedo;
- predict roughness;
- predict specular properties;
- estimate normals;
- estimate illumination.

---

## Phase 5 — Uncertainty

Status: Planned

- implement uncertainty prediction;
- evaluate calibration;
- analyze difficult object regions;
- investigate uncertainty-guided optimization.

---

## Phase 6 — Relighting

Status: Planned

- integrate material-aware rendering;
- implement controlled lighting;
- evaluate novel illumination;
- compare against RGB-only reconstruction.

---

## Phase 7 — Real-World Evaluation

Status: Planned

- collect smartphone videos;
- evaluate segmentation;
- estimate depth;
- estimate camera poses;
- reconstruct real objects;
- evaluate material predictions;
- evaluate relighting.

---

## Phase 8 — Final Research Evaluation

Status: Planned

- complete ablations;
- compare baselines;
- evaluate synthetic-to-real transfer;
- analyze failure cases;
- document results;
- release reproducible artifacts.
"""

roadmap_path = DOCS_DIR / "ROADMAP.md"

roadmap_path.write_text(
    ROADMAP,
    encoding="utf-8"
)

print(f"✓ Created {roadmap_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/ROADMAP.md


In [64]:
# ============================================================
# DATA GOVERNANCE AND RESEARCH RESPONSIBILITY
# ============================================================

GOVERNANCE = r"""
# Data Governance and Research Responsibility

## Source Data

The project uses 3D assets originating from the Amazon Berkeley Objects
dataset.

The project does not claim ownership of the original source assets.

Users of the processed dataset should consult the original source
documentation and licensing terms.

---

## Dataset Redistribution

This repository contains a selected research subset.

It should not be interpreted as authorization to redistribute the
complete source dataset.

Users are responsible for complying with the terms associated with the
original data.

---

## Research Transparency

The project documents:

- source datasets;
- preprocessing;
- model assumptions;
- evaluation methodology;
- known limitations;
- failure cases.

Results should be interpreted within these limitations.

---

## Responsible Evaluation

Performance should not be reported using only favorable examples.

Difficult cases should be included in evaluation, particularly:

- reflective objects;
- transparent objects;
- dark materials;
- partially observed objects;
- complex backgrounds;
- poor lighting.

---

## Reproducibility

Research claims should be supported by reproducible experiments,
documented configurations and clearly defined evaluation splits.
"""

governance_path = DOCS_DIR / "DATA_GOVERNANCE.md"

governance_path.write_text(
    GOVERNANCE,
    encoding="utf-8"
)

print(f"✓ Created {governance_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/DATA_GOVERNANCE.md


In [65]:
# ============================================================
# HUGGING FACE RESEARCH README
# ============================================================

HF_README = f"""---
license: other
task_categories:
  - computer-vision
  - image-feature-extraction
tags:
  - 3d
  - 3d-reconstruction
  - neural-rendering
  - gaussian-splatting
  - inverse-rendering
  - material-decomposition
  - relighting
  - multi-view
  - amazon-berkeley-objects
  - glb
  - gltf
pretty_name: Neural Object Reconstruction & Relighting Lab
---

# Neural Object Reconstruction & Relighting Lab

## Uncertainty-Aware Multi-View Material Decomposition for Relightable 3D Objects

The **Neural Object Reconstruction & Relighting Lab** is a research
project investigating how casually captured multi-view imagery can be
converted into material-aware and uncertainty-aware 3D representations
that can be rendered under new viewpoints and lighting conditions.

The project combines computer vision, neural rendering, multi-view
learning, 3D Gaussian Splatting, material decomposition, uncertainty
estimation and differentiable rendering.

---

## Research Objective

The central objective is to investigate whether explicit modeling of
material properties and uncertainty within a multi-view 3D
representation can provide more controllable relighting than
RGB-only reconstruction.

The intended system is:

**Smartphone Video**

→ Segmentation

→ Depth

→ Camera Pose

→ 3D Reconstruction

→ Multi-View Features

→ Material Decomposition

→ Uncertainty

→ Material-Aware 3D Representation

→ Differentiable Rendering

→ Relighting

---

## 3D Data

The implemented 3D data pipeline uses the **Amazon Berkeley Objects
(ABO)** dataset.

The assets are accessed through Amazon S3.

The project uses a selected subset rather than attempting to reproduce
the complete ABO dataset.

The selection pipeline performs:

- metadata inspection;
- product/model matching;
- candidate selection;
- quality filtering;
- category-aware selection;
- batch processing;
- GLB validation.

---

## Why 3D Assets Are Used

The selected 3D objects provide a controlled source for synthetic
training and evaluation data.

Objects can be rendered from multiple camera positions and lighting
conditions.

The resulting data can contain ground truth for:

- RGB;
- depth;
- normals;
- segmentation;
- albedo;
- roughness;
- specular properties;
- illumination;
- camera parameters.

This makes it possible to investigate material decomposition under
controlled conditions before evaluating the system on real-world
smartphone captures.

---

## Proposed Representation

The research investigates a material-aware extension of 3D Gaussian
Splatting.

In addition to conventional Gaussian parameters, the representation
can contain learned material features and uncertainty.

Potential material components include:

- albedo;
- roughness;
- specular properties;
- normals;
- illumination.

The uncertainty component is intended to indicate regions where the
system has lower confidence.

---

## Experiments

The project evaluates the system progressively.

### Baseline

RGB-only 3D Gaussian Splatting.

### Geometry-Aware Baseline

RGB + depth.

### Material Baseline

Single-view material prediction.

### Multi-View Model

Multi-view material prediction.

### Proposed Representation

Material-aware Gaussian representation.

### Full System

Material-aware representation + uncertainty + differentiable
relighting.

---

## Evaluation

Evaluation includes:

### Reconstruction

- PSNR;
- SSIM;
- LPIPS;
- multi-view consistency.

### Segmentation

- IoU;
- Dice;
- boundary F-score.

### Material

- albedo error;
- roughness error;
- normal angular error;
- specular estimation error.

### Relighting

- novel-lighting reconstruction quality;
- perceptual similarity;
- material consistency.

### Uncertainty

- calibration;
- uncertainty/error correlation;
- difficult-region detection.

---

## Dataset Splitting

Experiments use object-level splits.

Views of the same object must remain within the same split.

A proposed configuration is:

- 70% training;
- 15% validation;
- 15% testing.

This reduces view-level data leakage.

---

## Limitations

The project does not claim perfect physical material recovery.

The problem is inherently ambiguous, particularly for:

- reflective surfaces;
- transparent objects;
- dark materials;
- unseen regions;
- incomplete camera trajectories;
- poor depth estimates.

Synthetic data also introduces a domain gap relative to real smartphone
captures.

---

## Research Status

The project is being developed incrementally.

The current data engineering stage establishes a selected 3D object
collection from Amazon Berkeley Objects.

Subsequent stages extend the collection into a controlled synthetic
rendering pipeline, neural reconstruction experiments, material
prediction, uncertainty estimation and real-world smartphone
evaluation.

---

## Documentation

Detailed research documentation is included in this repository:

- `PROJECT_OVERVIEW.md`
- `RESEARCH_QUESTIONS.md`
- `METHODOLOGY.md`
- `EXPERIMENTAL_DESIGN.md`
- `EVALUATION_PROTOCOL.md`
- `LIMITATIONS.md`
- `REPRODUCIBILITY.md`
- `ROADMAP.md`
- `DATA_GOVERNANCE.md`

---

## Source Data

The 3D assets originate from the Amazon Berkeley Objects dataset.

Users should consult the original dataset documentation and licensing
terms before using or redistributing source assets.

---

## Citation

If this research or processed data is used, please cite this
repository together with the original Amazon Berkeley Objects work.

Created: {CREATED_AT}
"""

HF_README_PATH = DOCS_DIR / "README.md"

HF_README_PATH.write_text(
    HF_README,
    encoding="utf-8"
)

print(f"✓ Created {HF_README_PATH}")

✓ Created /content/neural_object_reconstruction/research_documentation/README.md


In [66]:
# ============================================================
# CITATION
# ============================================================

CITATION = """cff-version: 1.2.0
title: "Neural Object Reconstruction & Relighting Lab"
message: "If you use this research project or processed dataset, please cite this repository and the original Amazon Berkeley Objects dataset."
type: dataset

authors:
  - family-names: "Ndeda"
    given-names: "Jeremy"

repository-code: "https://huggingface.co/datasets/ndeda/neural-object-reconstruction-abo"
repository: "https://huggingface.co/datasets/ndeda/neural-object-reconstruction-abo"

keywords:
  - 3D reconstruction
  - neural rendering
  - 3D Gaussian Splatting
  - material decomposition
  - inverse rendering
  - relighting
  - uncertainty estimation
  - multi-view learning
  - computer vision

abstract: >
  Research project investigating uncertainty-aware multi-view material
  decomposition and material-aware 3D representations for relightable
  object reconstruction.
"""

citation_path = DOCS_DIR / "CITATION.cff"

citation_path.write_text(
    CITATION,
    encoding="utf-8"
)

print(f"✓ Created {citation_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/CITATION.cff


In [67]:
# ============================================================
# RESEARCH METADATA
# ============================================================

research_metadata = {
    "project": PROJECT_NAME,
    "subtitle": PROJECT_SUBTITLE,

    "research_area": [
        "3D computer vision",
        "neural rendering",
        "inverse rendering",
        "multi-view learning",
        "material decomposition",
        "3D Gaussian Splatting",
        "uncertainty estimation",
        "relighting"
    ],

    "primary_3d_source": {
        "name": "Amazon Berkeley Objects",
        "abbreviation": "ABO",
        "access": "Amazon S3"
    },

    "research_input": [
        "multi-view images",
        "smartphone video"
    ],

    "major_components": [
        "object segmentation",
        "depth estimation",
        "camera pose estimation",
        "3D reconstruction",
        "multi-view feature learning",
        "material decomposition",
        "uncertainty estimation",
        "differentiable rendering",
        "relighting"
    ],

    "material_targets": [
        "albedo",
        "roughness",
        "specular properties",
        "surface normals",
        "illumination"
    ],

    "research_status": {
        "dataset_preparation": "active",
        "synthetic_data": "planned",
        "baseline_reconstruction": "planned",
        "material_model": "planned",
        "uncertainty_model": "planned",
        "relighting": "planned",
        "real_world_evaluation": "planned"
    },

    "created_at": CREATED_AT
}

metadata_path = DOCS_DIR / "research_metadata.json"

metadata_path.write_text(
    json.dumps(
        research_metadata,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(f"✓ Created {metadata_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/research_metadata.json


In [68]:
# ============================================================
# RESEARCH DOCUMENTATION INDEX
# ============================================================

INDEX = """# Research Documentation

## Neural Object Reconstruction & Relighting Lab

This directory contains the research documentation for the project.

| Document | Purpose |
|---|---|
| PROJECT_OVERVIEW.md | Research problem, objective and scope |
| RESEARCH_QUESTIONS.md | Research questions and hypotheses |
| METHODOLOGY.md | End-to-end methodology |
| EXPERIMENTAL_DESIGN.md | Baselines, ablations and experiments |
| EVALUATION_PROTOCOL.md | Evaluation metrics and protocol |
| LIMITATIONS.md | Known limitations and research risks |
| REPRODUCIBILITY.md | Reproducibility requirements |
| ROADMAP.md | Development and research roadmap |
| DATA_GOVERNANCE.md | Data provenance and responsible use |
| CITATION.cff | Citation metadata |
| research_metadata.json | Machine-readable project metadata |

## Project Repository

Hugging Face:

https://huggingface.co/datasets/ndeda/neural-object-reconstruction-abo

## Research Structure

The project separates:

1. data preparation;
2. synthetic data generation;
3. perception;
4. reconstruction;
5. material estimation;
6. uncertainty;
7. rendering;
8. relighting;
9. evaluation.

This separation is intentional so that each research component can be
evaluated independently.
"""

index_path = DOCS_DIR / "INDEX.md"

index_path.write_text(
    INDEX,
    encoding="utf-8"
)

print(f"✓ Created {index_path}")

✓ Created /content/neural_object_reconstruction/research_documentation/INDEX.md


In [69]:
# ============================================================
# UPLOAD RESEARCH DOCUMENTATION TO HUGGING FACE
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

print("=" * 72)
print("UPLOADING RESEARCH DOCUMENTATION")
print("=" * 72)

files_to_upload = sorted(
    DOCS_DIR.iterdir()
)

uploaded = 0

for file_path in files_to_upload:

    if not file_path.is_file():
        continue

    print(
        f"\nUploading: {file_path.name}"
    )

    try:

        api.upload_file(
            path_or_fileobj=str(
                file_path
            ),
            path_in_repo=(
                f"research/{file_path.name}"
            ),
            repo_id=HF_REPO,
            repo_type=HF_REPO_TYPE
        )

        uploaded += 1

        print("✓ Uploaded")

    except Exception as e:

        print(
            f"✗ Failed: {e}"
        )

print()
print("=" * 72)
print(f"Uploaded {uploaded} documentation files")
print("=" * 72)

UPLOADING RESEARCH DOCUMENTATION

Uploading: CITATION.cff
✓ Uploaded

Uploading: DATA_GOVERNANCE.md
✓ Uploaded

Uploading: EVALUATION_PROTOCOL.md
✓ Uploaded

Uploading: EXPERIMENTAL_DESIGN.md
✓ Uploaded

Uploading: INDEX.md
✓ Uploaded

Uploading: LIMITATIONS.md
✓ Uploaded

Uploading: METHODOLOGY.md
✓ Uploaded

Uploading: PROJECT_OVERVIEW.md
✓ Uploaded

Uploading: README.md
✓ Uploaded

Uploading: REPRODUCIBILITY.md
✓ Uploaded

Uploading: RESEARCH_QUESTIONS.md
✓ Uploaded

Uploading: ROADMAP.md
✓ Uploaded

Uploading: research_metadata.json
✓ Uploaded

Uploaded 13 documentation files


In [70]:
# ============================================================
# FINAL HUGGING FACE VERIFICATION
# ============================================================

print("=" * 72)
print("VERIFYING RESEARCH REPOSITORY")
print("=" * 72)

try:

    remote_files = api.list_repo_files(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE
    )

    required = [
        "README.md",

        "research/INDEX.md",
        "research/PROJECT_OVERVIEW.md",
        "research/RESEARCH_QUESTIONS.md",
        "research/METHODOLOGY.md",
        "research/EXPERIMENTAL_DESIGN.md",
        "research/EVALUATION_PROTOCOL.md",
        "research/LIMITATIONS.md",
        "research/REPRODUCIBILITY.md",
        "research/ROADMAP.md",
        "research/DATA_GOVERNANCE.md",
        "research/CITATION.cff",
        "research/research_metadata.json"
    ]

    print(
        f"\nRemote repository contains "
        f"{len(remote_files):,} files."
    )

    missing = []

    for filename in required:

        if filename in remote_files:

            print(
                f"✓ {filename}"
            )

        else:

            print(
                f"✗ {filename}"
            )

            missing.append(
                filename
            )

    print()

    if missing:

        print(
            "WARNING: Missing files:"
        )

        for item in missing:

            print(
                f"  - {item}"
            )

    else:

        print(
            "✓ Research documentation is complete."
        )

except Exception as e:

    print(
        f"Verification failed: {e}"
    )


print()
print("=" * 72)
print("RESEARCH DOCUMENTATION COMPLETE")
print("=" * 72)

print(
    "\nHugging Face:"
)

print(
    "https://huggingface.co/datasets/"
    "ndeda/neural-object-reconstruction-abo"
)

VERIFYING RESEARCH REPOSITORY

Remote repository contains 314 files.
✓ README.md
✓ research/INDEX.md
✓ research/PROJECT_OVERVIEW.md
✓ research/RESEARCH_QUESTIONS.md
✓ research/METHODOLOGY.md
✓ research/EXPERIMENTAL_DESIGN.md
✓ research/EVALUATION_PROTOCOL.md
✓ research/LIMITATIONS.md
✓ research/REPRODUCIBILITY.md
✓ research/ROADMAP.md
✓ research/DATA_GOVERNANCE.md
✓ research/CITATION.cff
✓ research/research_metadata.json

✓ Research documentation is complete.

RESEARCH DOCUMENTATION COMPLETE

Hugging Face:
https://huggingface.co/datasets/ndeda/neural-object-reconstruction-abo


In [71]:
!pip -q install -U huggingface_hub

from huggingface_hub import HfApi
from pathlib import Path
import os
import textwrap

api = HfApi()

# CHANGE THIS ONLY IF YOUR REPOSITORY NAME IS DIFFERENT
HF_REPO = "ndeda/neural-object-reconstruction-abo"
REPO_TYPE = "dataset"

print("Hugging Face repository:", HF_REPO)

Hugging Face repository: ndeda/neural-object-reconstruction-abo


In [72]:
from huggingface_hub import login

login()

print("✓ Hugging Face authentication completed.")

✓ Hugging Face authentication completed.


In [74]:
readme_sections = []

readme_sections.append(r"""---
license: other
task_categories:
  - computer-vision
  - image-feature-extraction
tags:
  - 3d
  - 3d-reconstruction
  - neural-rendering
  - inverse-rendering
  - gaussian-splatting
  - 3d-gaussian-splatting
  - material-decomposition
  - relighting
  - multi-view
  - depth-estimation
  - object-segmentation
  - uncertainty-estimation
  - amazon-berkeley-objects
  - abo
  - glb
  - gltf
  - computer-vision
  - computer-graphics
  - synthetic-data
pretty_name: Neural Object Reconstruction and Relighting Research Dataset
---

# Neural Object Reconstruction and Relighting Research Dataset

## Uncertainty-Aware Multi-View Material Decomposition for Relightable 3D Objects
""")

print("✓ Header added.")

✓ Header added.


In [75]:
readme_sections.append(r"""
## Project Description

This repository contains the data infrastructure and research materials
for a study on 3D object reconstruction, material decomposition and
relighting from multi-view RGB observations.

The project investigates whether an object captured using a short
smartphone video can be converted into a structured 3D representation
that preserves useful information about geometry, appearance, material
properties and prediction uncertainty.

The work combines several areas of computer vision and computer
graphics, including:

- multi-view reconstruction;
- object segmentation;
- monocular depth estimation;
- camera pose estimation;
- 3D Gaussian Splatting;
- neural rendering;
- inverse rendering;
- material decomposition;
- uncertainty estimation;
- differentiable rendering;
- novel-view synthesis;
- relighting.

The research is designed as a modular pipeline. Individual components
are evaluated independently before being integrated into the complete
system.

The central objective is to move beyond appearance-only reconstruction
toward a representation that can be used for controlled rendering under
different lighting conditions.
""")

print("✓ Project description added.")

✓ Project description added.


In [76]:
readme_sections.append(r"""
## Research Question

The primary research question is:

> Can uncertainty-aware multi-view material decomposition improve the
> reliability and controllability of 3D Gaussian representations
> reconstructed from casual smartphone video?

### Research Hypothesis

The working hypothesis is that explicitly estimating intrinsic material
properties together with their uncertainty can produce a more useful
representation for novel-view rendering and relighting than a
conventional RGB-only 3D representation.

This hypothesis will be evaluated through controlled experiments,
baseline comparisons and ablation studies.

The hypothesis should not be considered experimentally validated until
the corresponding experiments have been completed.
""")

print("✓ Research question added.")

✓ Research question added.


In [77]:
readme_sections.append(r"""
## Research Objectives

The project has the following objectives:

1. Develop an object segmentation pipeline for casual video.
2. Investigate temporal segmentation and high-resolution foreground
   refinement.
3. Estimate depth from RGB observations.
4. Estimate camera intrinsics and poses.
5. Establish a 3D Gaussian Splatting reconstruction baseline.
6. Build a controlled synthetic multi-view dataset.
7. Develop multi-view material decomposition.
8. Estimate albedo, roughness, specular properties and surface normals.
9. Estimate illumination separately from intrinsic material properties.
10. Estimate uncertainty associated with material predictions.
11. Integrate material information into a 3D Gaussian representation.
12. Develop material-aware differentiable rendering.
13. Evaluate novel-view synthesis.
14. Evaluate relighting under unseen illumination.
15. Investigate synthetic-to-real transfer.
16. Evaluate the complete system using real smartphone captures.
17. Analyze failure cases and uncertainty.
""")

print("✓ Research objectives added.")

✓ Research objectives added.


In [78]:
readme_sections.append(r"""
# Dataset Source

## Amazon Berkeley Objects

The primary source of 3D object assets for this project is the
Amazon Berkeley Objects (ABO) dataset.

The project uses selected 3D assets rather than attempting to reproduce
the complete original dataset.

The 3D assets are accessed through Amazon S3 as part of the data
preparation pipeline.

The resulting collection is a research-derived subset prepared for
controlled experiments and synthetic data generation.

The project does not claim ownership of the original ABO assets.

Users of ABO-derived material should consult the original dataset
documentation, licensing terms and citation requirements.
""")

print("✓ Dataset source section added.")

✓ Dataset source section added.


In [79]:
readme_sections.append(r"""
## Why Amazon Berkeley Objects Is Used

The research requires actual 3D object assets because material
decomposition and relighting experiments require controlled knowledge
of the underlying object.

A 3D object allows the project to generate multiple observations while
controlling variables such as:

- camera position;
- camera orientation;
- object rotation;
- illumination;
- material properties;
- background;
- rendering configuration.

This provides a controlled environment for generating supervision that
is difficult to obtain from ordinary photographs.

Amazon S3 is used as the storage and retrieval layer for the relevant
3D assets.

The current data workflow therefore consists of:

1. metadata discovery;
2. object selection;
3. candidate filtering;
4. S3 asset retrieval;
5. GLB validation;
6. dataset organization;
7. manifest generation.
""")

print("✓ ABO/S3 section added.")

✓ ABO/S3 section added.


In [80]:
readme_sections.append(r"""
# Synthetic Dataset Generation

Selected ABO 3D assets are used as source objects for controlled
synthetic rendering.

The synthetic data generation process is intended to produce multiple
views of the same object while varying camera, material and lighting
conditions.

The intended process is:

ABO 3D Asset
    |
    v
Object Selection
    |
    v
Quality Filtering
    |
    v
Controlled Rendering
    |
    +---- Camera Variation
    +---- Object Rotation
    +---- Lighting Variation
    +---- Material Variation
    +---- Background Variation
    |
    v
Multi-View Training / Evaluation Data

Depending on the rendering configuration, generated samples can contain
synchronized channels such as:

- RGB;
- depth;
- segmentation;
- surface normals;
- albedo;
- roughness;
- specular properties;
- illumination;
- camera parameters.

The synthetic dataset is intended to provide controlled supervision
for material decomposition, uncertainty estimation and relighting.
""")

print("✓ Synthetic dataset section added.")

✓ Synthetic dataset section added.


In [81]:
readme_sections.append(r"""
# Data Roles

| Data Source | Role |
|---|---|
| Amazon Berkeley Objects | Primary 3D object source |
| Amazon S3 | 3D asset retrieval and storage |
| Project synthetic dataset | Controlled training and evaluation |
| OpenIllumination | Material and illumination evaluation |
| OpenRooms | Material and lighting research |
| Tanks and Temples | Multi-view reconstruction evaluation |
| Project smartphone captures | Real-world evaluation |

The datasets listed above serve different purposes. The project does not
assume that a single dataset is sufficient for every research question.
""")

print("✓ Data roles added.")

✓ Data roles added.


In [84]:
pipeline = "\n".join([
    "# Research Pipeline",
    "",
    "The planned end-to-end system consists of the following stages:",
    "",
    "```text",
    "Smartphone Video",
    "       |",
    "       v",
    "Frame Extraction",
    "       |",
    "       v",
    "Object Segmentation",
    "       |",
    "       v",
    "Mask Refinement",
    "       |",
    "       v",
    "Depth Estimation",
    "       |",
    "       v",
    "Camera Pose Estimation",
    "       |",
    "       v",
    "3D Gaussian Splatting",
    "       |",
    "       v",
    "Multi-View Feature Learning",
    "       |",
    "       v",
    "Material Decomposition",
    "       |",
    "       v",
    "Uncertainty Estimation",
    "       |",
    "       v",
    "Material-Aware Gaussian Representation",
    "       |",
    "       v",
    "Differentiable Rendering",
    "       |",
    "       v",
    "Novel Lighting",
    "       |",
    "       v",
    "Relightable 3D Object",
    "```",
    "",
    "The individual stages are developed and evaluated independently before",
    "being combined into the complete research system.",
    "",
    "The segmentation stage investigates SAM 2 and BiRefNet.",
    "",
    "Camera estimation uses COLMAP as the initial structure-from-motion",
    "baseline.",
    "",
    "3D Gaussian Splatting provides the initial reconstruction",
    "representation.",
    "",
    "The material stage investigates albedo, roughness, specular properties,",
    "surface normals and illumination.",
    "",
    "Uncertainty estimation is used to study the reliability of material",
    "predictions.",
    "",
    "The final rendering stage investigates whether the resulting",
    "representation can support novel-view synthesis and relighting."
])

readme_sections.append(pipeline)

print("✓ Research Pipeline added")

✓ Research Pipeline added


In [85]:
segmentation = "\n".join([
    "# Object Segmentation",
    "",
    "Object segmentation separates the target object from the surrounding",
    "scene before reconstruction.",
    "",
    "The project investigates temporal segmentation together with",
    "high-resolution foreground refinement.",
    "",
    "## SAM 2",
    "",
    "SAM 2 is investigated for:",
    "",
    "- object segmentation",
    "- temporal propagation",
    "- video tracking",
    "- segmentation confidence",
    "",
    "## BiRefNet",
    "",
    "BiRefNet is investigated for:",
    "",
    "- foreground refinement",
    "- boundary refinement",
    "- high-resolution masks",
    "",
    "The research also considers a fusion stage combining information from",
    "both approaches.",
    "",
    "Segmentation quality is evaluated independently because errors at this",
    "stage can propagate into depth estimation, reconstruction and material",
    "decomposition."
])

readme_sections.append(segmentation)

print("✓ Segmentation added")

✓ Segmentation added


In [86]:
depth_camera = "\n".join([
    "# Depth Estimation",
    "",
    "Depth estimation provides geometric information for downstream",
    "reconstruction.",
    "",
    "The initial system uses pretrained depth estimation models.",
    "",
    "The general process is:",
    "",
    "```text",
    "RGB Frame",
    "    |",
    "    v",
    "Depth Model",
    "    |",
    "    v",
    "Depth Map",
    "```",
    "",
    "Synthetic data may later be used to investigate domain-specific",
    "adaptation.",
    "",
    "# Camera Pose Estimation",
    "",
    "Camera estimation provides the poses required for multi-view",
    "reconstruction.",
    "",
    "The initial reconstruction workflow uses COLMAP for",
    "structure-from-motion.",
    "",
    "The resulting information includes:",
    "",
    "- camera intrinsics",
    "- camera extrinsics",
    "- feature correspondences",
    "- estimated camera poses",
    "- sparse reconstruction information",
    "",
    "Accurate camera estimation is important because pose errors can affect",
    "3D reconstruction and multi-view material consistency."
])

readme_sections.append(depth_camera)

print("✓ Depth and Camera sections added")

✓ Depth and Camera sections added


In [87]:
gaussian = "\n".join([
    "# 3D Gaussian Splatting",
    "",
    "3D Gaussian Splatting is used as the primary reconstruction baseline.",
    "",
    "A conventional Gaussian representation contains parameters associated",
    "with:",
    "",
    "- position",
    "- scale or covariance",
    "- rotation",
    "- opacity",
    "- appearance",
    "",
    "The research investigates extending this representation with material",
    "information.",
    "",
    "A material-aware Gaussian may contain:",
    "",
    "- position",
    "- scale",
    "- rotation",
    "- opacity",
    "- albedo",
    "- roughness",
    "- specular properties",
    "- normal information",
    "- uncertainty",
    "",
    "The purpose is to investigate whether explicit material information",
    "can improve rendering and relighting compared with an RGB-only",
    "representation."
])

readme_sections.append(gaussian)

print("✓ 3D Gaussian Splatting added")

✓ 3D Gaussian Splatting added


In [88]:
materials = "\n".join([
    "# Multi-View Material Decomposition",
    "",
    "Material decomposition is the central research component.",
    "",
    "Rather than predicting material properties independently from each",
    "image, the project investigates using multiple observations of the",
    "same object.",
    "",
    "The target properties include:",
    "",
    "## Albedo",
    "",
    "An estimate of intrinsic surface appearance.",
    "",
    "## Roughness",
    "",
    "An estimate of the surface roughness response.",
    "",
    "## Specular Properties",
    "",
    "Information describing the specular component of surface appearance.",
    "",
    "## Surface Normals",
    "",
    "Surface orientation information useful for geometry and rendering.",
    "",
    "## Illumination",
    "",
    "An estimate of lighting contributing to the observed appearance.",
    "",
    "The objective is not to claim perfect physical recovery of material",
    "properties.",
    "",
    "The research evaluates whether the learned representation is useful",
    "for reconstruction and relighting."
])

readme_sections.append(materials)

print("✓ Material Decomposition added")

✓ Material Decomposition added


In [89]:
uncertainty = "\n".join([
    "# Uncertainty Estimation",
    "",
    "The project treats uncertainty as an important part of the material",
    "prediction problem.",
    "",
    "Different surface regions have different levels of observability.",
    "",
    "Potentially difficult regions include:",
    "",
    "- reflective surfaces",
    "- transparent objects",
    "- dark surfaces",
    "- occluded regions",
    "- unseen surfaces",
    "- motion-blurred observations",
    "- poorly reconstructed regions",
    "",
    "The research investigates whether uncertainty estimates can identify",
    "areas where material predictions are less reliable.",
    "",
    "Planned analysis includes:",
    "",
    "- uncertainty/error correlation",
    "- calibration",
    "- confidence maps",
    "- high-error/high-uncertainty overlap",
    "",
    "The relationship between uncertainty and prediction error will be",
    "measured experimentally rather than assumed."
])

readme_sections.append(uncertainty)

print("✓ Uncertainty added")

✓ Uncertainty added


In [90]:
relighting = "\n".join([
    "# Relighting",
    "",
    "Relighting is the primary downstream application of the proposed",
    "representation.",
    "",
    "The intended experiment is:",
    "",
    "1. capture an object",
    "2. reconstruct its geometry",
    "3. estimate its material properties",
    "4. construct the material-aware representation",
    "5. change the lighting conditions",
    "6. render the reconstructed object under the new lighting",
    "",
    "The evaluation investigates whether the rendered appearance remains",
    "consistent with the estimated material properties.",
    "",
    "This distinguishes the research from systems that primarily reproduce",
    "the observed RGB appearance."
])

readme_sections.append(relighting)

print("✓ Relighting added")

✓ Relighting added


In [91]:
experiments = "\n".join([
    "# Experimental Design",
    "",
    "The research uses progressively stronger experimental configurations.",
    "",
    "## Experiment 1 — RGB Gaussian Baseline",
    "",
    "A conventional RGB-based 3D Gaussian Splatting system.",
    "",
    "Purpose: establish the baseline quality of novel-view reconstruction.",
    "",
    "## Experiment 2 — RGB + Depth",
    "",
    "Depth information is incorporated into reconstruction.",
    "",
    "Purpose: measure the contribution of explicit geometric information.",
    "",
    "## Experiment 3 — Single-View Material Prediction",
    "",
    "Material properties are predicted from individual observations.",
    "",
    "Purpose: establish a single-view material baseline.",
    "",
    "## Experiment 4 — Multi-View Material Prediction",
    "",
    "Multiple observations of the same object are processed jointly.",
    "",
    "Purpose: measure the contribution of multi-view information.",
    "",
    "## Experiment 5 — Material-Aware Gaussian Representation",
    "",
    "Material properties are stored within the Gaussian representation.",
    "",
    "Purpose: evaluate the effect of explicit material representation on",
    "rendering and relighting.",
    "",
    "## Experiment 6 — Uncertainty-Aware Representation",
    "",
    "Uncertainty estimates are incorporated into material prediction and",
    "the 3D representation.",
    "",
    "Purpose: determine whether uncertainty provides useful information",
    "about prediction reliability.",
    "",
    "## Experiment 7 — Complete Pipeline",
    "",
    "The complete proposed system combines:",
    "",
    "- segmentation",
    "- depth",
    "- camera estimation",
    "- 3D reconstruction",
    "- material decomposition",
    "- uncertainty",
    "- material-aware Gaussians",
    "- differentiable rendering",
    "- relighting"
])

readme_sections.append(experiments)

print("✓ Experimental Design added")

✓ Experimental Design added


In [92]:
ablations = "\n".join([
    "# Ablation Studies",
    "",
    "Component-level ablations are used to determine which parts of the",
    "system contribute to final performance.",
    "",
    "Planned ablations include:",
    "",
    "- without depth",
    "- without multi-view information",
    "- without material features",
    "- without uncertainty",
    "- without segmentation refinement",
    "",
    "The segmentation pipeline may also be compared using:",
    "",
    "```text",
    "SAM 2",
    "   vs.",
    "BiRefNet",
    "   vs.",
    "SAM 2 + BiRefNet",
    "   vs.",
    "SAM 2 + BiRefNet + Learned Fusion",
    "```",
    "",
    "The purpose is to separate the contribution of individual components",
    "from the performance of the complete system."
])

readme_sections.append(ablations)

print("✓ Ablation Studies added")

✓ Ablation Studies added


In [93]:
evaluation = "\n".join([
    "# Evaluation Protocol",
    "",
    "The project evaluates different components using task-specific",
    "metrics.",
    "",
    "## Segmentation",
    "",
    "- Intersection over Union (IoU)",
    "- Dice / F1",
    "- Boundary F-score",
    "",
    "## Depth",
    "",
    "- RMSE",
    "- Absolute Relative Error",
    "- threshold accuracy",
    "",
    "## Novel-View Reconstruction",
    "",
    "- PSNR",
    "- SSIM",
    "- LPIPS",
    "- geometric error",
    "- multi-view consistency",
    "",
    "## Material",
    "",
    "Albedo:",
    "- MAE",
    "- RMSE",
    "- perceptual similarity",
    "",
    "Roughness:",
    "- MAE",
    "- RMSE",
    "",
    "Specular properties:",
    "- MAE",
    "- RMSE",
    "",
    "Surface normals:",
    "- angular error",
    "",
    "## Relighting",
    "",
    "Relighting evaluation includes:",
    "",
    "- image reconstruction quality",
    "- perceptual similarity",
    "- material consistency",
    "- appearance consistency under unseen illumination",
    "",
    "## Uncertainty",
    "",
    "Uncertainty evaluation includes:",
    "",
    "- uncertainty/error correlation",
    "- calibration",
    "- confidence maps",
    "- high-error/high-uncertainty overlap"
])

readme_sections.append(evaluation)

print("✓ Evaluation Protocol added")

✓ Evaluation Protocol added


In [94]:
real_world = "\n".join([
    "# Real-World Evaluation",
    "",
    "Synthetic data provides controlled ground truth but cannot fully",
    "represent real smartphone imagery.",
    "",
    "The project therefore includes a real-world evaluation stage.",
    "",
    "The target input is a short smartphone video of a physical object.",
    "",
    "A typical capture is expected to last approximately 20–60 seconds.",
    "",
    "The intended workflow is:",
    "",
    "```text",
    "Physical Object",
    "      |",
    "      v",
    "Smartphone Video",
    "      |",
    "      v",
    "Frame Extraction",
    "      |",
    "      v",
    "Object Segmentation",
    "      |",
    "      v",
    "Depth Estimation",
    "      |",
    "      v",
    "Camera Pose Estimation",
    "      |",
    "      v",
    "3D Gaussian Reconstruction",
    "      |",
    "      v",
    "Material Decomposition",
    "      |",
    "      v",
    "Uncertainty Estimation",
    "      |",
    "      v",
    "Relighting",
    "```",
    "",
    "Important capture conditions include:",
    "",
    "- sufficient object visibility",
    "- sufficient viewpoint coverage",
    "- reasonable camera motion",
    "- limited motion blur",
    "- reasonable illumination",
    "- minimal severe occlusion"
])

readme_sections.append(real_world)

print("✓ Real-World Evaluation added")

✓ Real-World Evaluation added


In [95]:
synthetic_real = "\n".join([
    "# Synthetic-to-Real Transfer",
    "",
    "Synthetic and real data serve different roles.",
    "",
    "Synthetic data provides:",
    "",
    "- known geometry",
    "- known material properties",
    "- known lighting",
    "- known camera parameters",
    "- pixel-aligned supervision",
    "",
    "Real data introduces:",
    "",
    "- sensor noise",
    "- exposure variation",
    "- white-balance variation",
    "- motion blur",
    "- compression",
    "- lens characteristics",
    "- uncontrolled illumination",
    "- imperfect segmentation",
    "- imperfect depth",
    "",
    "The project therefore treats the synthetic-to-real gap as an explicit",
    "research problem.",
    "",
    "Performance on synthetic data will not automatically be interpreted",
    "as equivalent real-world performance."
])

readme_sections.append(synthetic_real)

print("✓ Synthetic-to-Real section added")

✓ Synthetic-to-Real section added


In [96]:
limitations = "\n".join([
    "# Limitations",
    "",
    "## Material Ambiguity",
    "",
    "Material and illumination are strongly coupled in RGB observations.",
    "",
    "## Reflective Surfaces",
    "",
    "Highly reflective surfaces can reproduce environmental appearance,",
    "making intrinsic material estimation difficult.",
    "",
    "## Transparent Objects",
    "",
    "Transparent and translucent objects remain challenging because their",
    "appearance depends on geometry and surrounding illumination.",
    "",
    "## Unseen Surfaces",
    "",
    "Surfaces that are never observed cannot be reliably reconstructed.",
    "",
    "## Depth Errors",
    "",
    "Depth estimation errors can propagate into reconstruction and material",
    "prediction.",
    "",
    "## Camera Pose Errors",
    "",
    "Incorrect camera poses can reduce multi-view consistency.",
    "",
    "## Synthetic-to-Real Gap",
    "",
    "Synthetic rendering cannot perfectly reproduce real smartphone capture",
    "conditions.",
    "",
    "## Dataset Bias",
    "",
    "ABO is a product-oriented 3D object dataset. A selected subset may",
    "therefore contain biases related to the source collection and object",
    "categories."
])

readme_sections.append(limitations)

print("✓ Limitations added")

✓ Limitations added


In [97]:
governance = "\n".join([
    "# Data Governance and Provenance",
    "",
    "The primary 3D object source is the Amazon Berkeley Objects dataset.",
    "",
    "Selected 3D assets are accessed through Amazon S3 for research data",
    "preparation.",
    "",
    "This repository does not claim ownership of the original ABO assets.",
    "",
    "Any use or redistribution of ABO-derived data must comply with the",
    "original dataset's licensing and usage requirements.",
    "",
    "External datasets, pretrained models and published methods used by",
    "the research remain subject to their respective licenses and citation",
    "requirements.",
    "",
    "Users should consult the original source documentation before",
    "redistributing derived assets."
])

readme_sections.append(governance)

print("✓ Data Governance added")

✓ Data Governance added


In [98]:
reproducibility = "\n".join([
    "# Reproducibility",
    "",
    "The project aims to preserve sufficient information to reproduce data",
    "preparation and model experiments.",
    "",
    "The data pipeline should maintain:",
    "",
    "- source metadata",
    "- object identifiers",
    "- processing batches",
    "- downloaded asset records",
    "- validation results",
    "- dataset manifests",
    "",
    "Synthetic samples should record, where applicable:",
    "",
    "- object identifier",
    "- camera parameters",
    "- lighting parameters",
    "- material parameters",
    "- renderer configuration",
    "- random seed",
    "- generated channels",
    "",
    "Model experiments should record:",
    "",
    "- model configuration",
    "- dataset split",
    "- random seed",
    "- optimizer",
    "- learning rate",
    "- batch size",
    "- training duration",
    "- checkpoint",
    "- evaluation metrics"
])

readme_sections.append(reproducibility)

print("✓ Reproducibility added")

✓ Reproducibility added


In [99]:
status = "\n".join([
    "# Current Status",
    "",
    "The current stage of the project focuses on establishing the 3D data",
    "infrastructure using Amazon Berkeley Objects and Amazon S3.",
    "",
    "Current work includes:",
    "",
    "- metadata discovery",
    "- object selection",
    "- candidate filtering",
    "- S3 retrieval",
    "- GLB validation",
    "- asset organization",
    "- manifest generation",
    "",
    "The research system is being developed incrementally.",
    "",
    "Planned development stages include:",
    "",
    "1. synthetic multi-view rendering",
    "2. segmentation",
    "3. depth estimation",
    "4. camera estimation",
    "5. 3D Gaussian reconstruction",
    "6. material decomposition",
    "7. uncertainty estimation",
    "8. material-aware Gaussian representations",
    "9. differentiable rendering",
    "10. relighting",
    "11. real-world smartphone evaluation",
    "12. quantitative and qualitative evaluation"
])

readme_sections.append(status)

print("✓ Current Status added")

✓ Current Status added


In [100]:
README = "\n\n".join(readme_sections)

print("=" * 60)
print("README READY")
print("=" * 60)
print("Sections:", len(readme_sections))
print("Characters:", f"{len(README):,}")
print("Words:", f"{len(README.split()):,}")

README READY
Sections: 24
Characters: 19,662
Words: 2,578


In [101]:
README_PATH = "/content/README.md"

with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(README)

print("✓ Saved:", README_PATH)
print("✓ Size:", f"{len(README):,}", "characters")

✓ Saved: /content/README.md
✓ Size: 19,662 characters


In [102]:
with open("/content/README.md", "r", encoding="utf-8") as f:
    check = f.read()

print("First 500 characters:")
print(check[:500])

print("\nLast 1000 characters:")
print(check[-1000:])

First 500 characters:
---
license: other
task_categories:
  - computer-vision
  - image-feature-extraction
tags:
  - 3d
  - 3d-reconstruction
  - neural-rendering
  - inverse-rendering
  - gaussian-splatting
  - 3d-gaussian-splatting
  - material-decomposition
  - relighting
  - multi-view
  - depth-estimation
  - object-segmentation
  - uncertainty-estimation
  - amazon-berkeley-objects
  - abo
  - glb
  - gltf
  - computer-vision
  - computer-graphics
  - synthetic-data
pretty_name: Neural Object Reconstruction and

Last 1000 characters:
rial parameters
- renderer configuration
- random seed
- generated channels

Model experiments should record:

- model configuration
- dataset split
- random seed
- optimizer
- learning rate
- batch size
- training duration
- checkpoint
- evaluation metrics

# Current Status

The current stage of the project focuses on establishing the 3D data
infrastructure using Amazon Berkeley Objects and Amazon S3.

Current work includes:

- metadata discovery
- 

In [103]:
upload_result = api.upload_file(
    path_or_fileobj="/content/README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO,
    repo_type="dataset",
    commit_message="Update complete research dataset documentation"
)

print("✓ Hugging Face dataset card uploaded.")
print(upload_result)

✓ Hugging Face dataset card uploaded.
https://huggingface.co/datasets/ndeda/neural-object-reconstruction-abo/commit/12eea924a09245359f9c3ca8563ba22d2086dda7
